# Base integrada mensual por rama: incertidumbre comercial y manufactura de México

**Etapas 2 y 3 del proyecto:** inspección, limpieza, validación e integración de fuentes. **No** se estima ningún modelo, **no** se imputa, **no** se interpola, **no** se balancea el panel y **no** se recorta la muestra a un periodo común.

Este notebook es la versión ejecutable en Python de `code/R/01_preparacion_base_integrada.Rmd`: misma lógica, mismas decisiones y mismos checkpoints. Lee los archivos de `data/raw/` (y el derivado de `data/interim/aranceles/`, solo para verificarlo) y escribe **un solo Excel**: `data/processed/base_incertidumbre_comercial_mexico.xlsx`.

**Cómo leer los checkpoints.** Cada sección termina con una tabla de comprobaciones:

| Estado | Significado | ¿Qué pasa? |
|---|---|---|
| **OK** | La comprobación se ejecutó y se cumplió. | Continúa. |
| **REVISAR** | Advertencia o decisión que requiere atención; no impide construir la base. | Continúa y queda en la hoja `validaciones`. |
| **ERROR** | Problema que impide continuar de forma segura. | El notebook se detiene y no se exporta el Excel. |

Ninguna comprobación se marca `OK` si no se ejecutó: si no se pudo evaluar, aparece como `ERROR` o `REVISAR` con la leyenda *[No se pudo evaluar]*.

**Prefijos de las variables:** `emim_` (EMIM, México) · `expo_` (exportaciones de México) · `eeuu_imp_mx_` (importaciones de EE.UU. desde México y aranceles, Census) · `inpp_` (INPP del subsector) · `eeuu_ip_` (producción industrial de **EE.UU.**, G.17) · `tpu`, `wui_`, `wtui_`, `wpui_`, `epu_mex` (incertidumbre, comunes por mes) · `tc_fix`, `vix_` (controles comunes) · `obs_` (indicadores de existencia de la fila en cada fuente). El sufijo `_real` indica **miles de pesos de julio de 2019**.

## 1. Configuración del proyecto y rutas
### 1.1 Paquetes y raíz del repositorio

In [1]:
import hashlib, html, json, platform, re, sys, unicodedata, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import openpyxl
from IPython.display import HTML, Markdown, display

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.width", 200)

def _buscar_raiz():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "docs" / "inventario_fuentes.csv").exists() and (q / "data" / "raw").is_dir():
            return q
    raise RuntimeError("No se encontró la raíz del repositorio (carpeta con docs/inventario_fuentes.csv y data/raw/).")

RAIZ = _buscar_raiz()
INICIO = datetime.now()
display(HTML("""<style>
.tabla {font-size:82%; border-collapse:collapse; margin-bottom:8px}
.tabla th {background:#dce6f1; border:1px solid #bbb; padding:3px 6px; text-align:left}
.tabla td {border:1px solid #ddd; padding:2px 6px; vertical-align:top}
</style>"""))
display(Markdown(f"**Raíz del repositorio:** `{RAIZ}`  \n**Inicio:** {INICIO:%Y-%m-%d %H:%M:%S}  \n"
                 f"**Python:** {platform.python_version()} · **pandas:** {pd.__version__} · **openpyxl:** {openpyxl.__version__}"))

**Raíz del repositorio:** `/home/user/Incertidumbre-Comercial`  
**Inicio:** 2026-10-01 21:00:56  
**Python:** 3.11.15 · **pandas:** 3.0.6 · **openpyxl:** 3.1.5

### 1.2 Funciones auxiliares: registro de comprobaciones, tablas y conversiones

Todas las comprobaciones van a un registro único que al final se exporta a la hoja `validaciones`. Las funciones de conversión cuentan las celdas que no se pudieron convertir, para que ninguna transformación sea silenciosa.

In [2]:
REG = {"checks": [], "conversiones": [], "exclusiones": [], "limpieza": [], "uniones": []}

class ProcesoDetenido(Exception):
    pass

def _es_na(v):
    try:
        return bool(pd.isna(v))
    except (TypeError, ValueError):
        return False

def registrar(cp, fuente, prueba, ok, resultado="", si_falla="ERROR"):
    assert si_falla in ("ERROR", "REVISAR")
    resultado = str(resultado)
    if ok is None or _es_na(ok):
        estado, resultado = si_falla, "[No se pudo evaluar] " + resultado
    elif bool(ok):
        estado = "OK"
    else:
        estado = si_falla
    REG["checks"].append(dict(checkpoint=cp, fuente=fuente, prueba=prueba, resultado=resultado, estado=estado))
    return estado

def revisar(cp, fuente, prueba, resultado):
    return registrar(cp, fuente, prueba, False, resultado, si_falla="REVISAR")

def excluir(fuente, n, motivo):
    REG["exclusiones"].append(dict(fuente=fuente, filas_o_celdas=int(n), motivo=motivo))

def detener(*partes):
    raise ProcesoDetenido("\n\n*** PROCESO DETENIDO ***\n" + "".join(str(p) for p in partes))

def exigir(condicion, *partes):
    if not bool(condicion):
        detener(*partes)

# ---- Presentación -------------------------------------------------------------
COLORES = {"OK": "#1b7f3b", "REVISAR": "#b26a00", "ERROR": "#b00020"}

def badge(e):
    return (f'<span style="background:{COLORES.get(e, "#555")};color:#fff;padding:1px 6px;'
            f'border-radius:4px;font-weight:bold">{e}</span>')

def fmt_valor(v, ancho=90):
    if _es_na(v):
        return "NA"
    if isinstance(v, (pd.Timestamp, datetime)):
        return v.strftime("%Y-%m-%d")
    if isinstance(v, (bool, np.bool_)):
        return "TRUE" if v else "FALSE"
    if isinstance(v, (int, np.integer)):
        return f"{v:,}" if abs(v) >= 10000 else str(v)
    if isinstance(v, (float, np.floating)):
        return f"{v:,.2f}" if abs(v) >= 1e4 else f"{v:.6g}"
    s = str(v)
    return s if len(s) <= ancho else s[: ancho - 3] + "..."

def tabla(df, titulo=None, max_filas=60, ancho=90):
    if titulo:
        display(Markdown(f"**{titulo}**"))
    df = pd.DataFrame(df)
    n = len(df)
    if n == 0:
        display(Markdown("_(sin filas)_"))
        return
    d = df.head(max_filas).copy()
    d.columns = [str(c) for c in d.columns]
    for c in d.columns:
        d[c] = [html.escape(fmt_valor(v, ancho)) for v in d[c]]
    if "estado" in d.columns:
        d["estado"] = [badge(e) for e in d["estado"]]
    display(HTML(d.to_html(index=False, escape=False, border=0, classes="tabla")))
    if n > max_filas:
        display(Markdown(f"_Se muestran {max_filas} de {n} filas._"))

def aviso(texto, tipo="info"):
    estilos = {"info": ("#e8f1fb", "#2c5aa0"), "revisar": ("#fff4e0", "#b26a00"),
               "error": ("#fde8ea", "#b00020"), "ok": ("#e7f5ec", "#1b7f3b")}
    fondo, borde = estilos[tipo]
    display(HTML(f'<div style="background:{fondo};border-left:6px solid {borde};padding:8px 12px;margin:8px 0">{texto}</div>'))

def titulo_checkpoint(texto):
    display(HTML(f'<div style="border-left:6px solid #2c5aa0;background:#eef3fb;padding:8px 12px;margin:12px 0;'
                 f'font-weight:bold;font-size:115%">&#9873; {html.escape(texto)}</div>'))

RUTA_EXCEL_FINAL = None

def cerrar_checkpoint(cp, nombre):
    x = pd.DataFrame([c for c in REG["checks"] if c["checkpoint"] == cp],
                     columns=["checkpoint", "fuente", "prueba", "resultado", "estado"])
    n_ok, n_rev, n_err = [(x.estado == e).sum() for e in ("OK", "REVISAR", "ERROR")]
    display(Markdown(f"**Resumen del {cp} ({nombre}):** {n_ok} OK · {n_rev} REVISAR · {n_err} ERROR"))
    tabla(x.drop(columns="checkpoint"), max_filas=1000, ancho=400)
    if n_err > 0:
        err = x[x.estado == "ERROR"]
        msg = (f"<b>{cp} ({html.escape(nombre)}): {n_err} comprobación(es) en ERROR. El proceso se detuvo aquí "
               f"y no se exportó ningún archivo.</b><br>"
               + "<br>".join(f"&bull; {html.escape(r.fuente)}: {html.escape(r.prueba)} &rarr; {html.escape(r.resultado)}"
                             for r in err.itertuples()))
        if RUTA_EXCEL_FINAL is not None and RUTA_EXCEL_FINAL.exists():
            msg += (f"<br><b>Atención:</b> <code>{RUTA_EXCEL_FINAL.name}</code> existe de una ejecución anterior "
                    f"y <b>no</b> corresponde a esta ejecución.")
        aviso(msg, "error")
        raise ProcesoDetenido(f"{cp}: " + "; ".join(f"{r.fuente}: {r.prueba} -> {r.resultado}" for r in err.itertuples()))
    elif n_rev > 0:
        aviso(f"<b>{cp}:</b> sin errores críticos. Hay {n_rev} punto(s) marcados como REVISAR: puede continuar, "
              f"pero léalos antes de usar la base.", "revisar")
    else:
        aviso(f"<b>{cp}:</b> todas las comprobaciones se ejecutaron y se cumplieron.", "ok")

# ---- Texto, números y fechas -----------------------------------------------------
def norm_txt(x):
    if _es_na(x):
        return None
    s = unicodedata.normalize("NFD", str(x).strip().lower())
    return "".join(ch for ch in s if unicodedata.category(ch) != "Mn")

MESES_ES = dict(enero=1, febrero=2, marzo=3, abril=4, mayo=5, junio=6, julio=7, agosto=8,
                septiembre=9, octubre=10, noviembre=11, diciembre=12)
MESES_ABR = dict(ene=1, feb=2, mar=3, abr=4, may=5, jun=6, jul=7, ago=8, sep=9, oct=10, nov=11, dic=12)

def fmt_mes(d):
    return "NA" if _es_na(d) else pd.Timestamp(d).strftime("%Y-%m")

def rango_txt(fechas):
    f = pd.Series(fechas).dropna()
    return "sin datos" if f.empty else f"{fmt_mes(f.min())} a {fmt_mes(f.max())}"

def a_mes(s):
    """Normaliza a datetime64[ns] (para que todas las llaves 'mes' sean del mismo tipo)."""
    return pd.to_datetime(pd.Series(s)).astype("datetime64[ns]")

def convertir_num(valores, fuente, variable, codigos=()):
    s = pd.Series(valores).astype(object)
    txt = s.map(lambda v: None if _es_na(v) else str(v).strip())
    vacio = txt.isna() | (txt == "")
    es_cod = ~vacio & txt.isin(list(codigos))
    num = pd.to_numeric(txt.where(~vacio & ~es_cod), errors="coerce").astype(float)
    fallo = ~vacio & ~es_cod & num.isna()
    REG["conversiones"].append(dict(
        fuente=fuente, variable=variable, celdas_con_contenido=int((~vacio).sum()),
        convertidas=int(num.notna().sum()), codigos_de_faltante=int(es_cod.sum()),
        codigos=", ".join(sorted(set(txt[es_cod]))), no_convertibles=int(fallo.sum()),
        ejemplos_no_convertibles=" | ".join(list(dict.fromkeys(txt[fallo]))[:5])))
    return num

def a_fecha(valores):
    """Fecha de Excel (número de serie o fecha ya interpretada como texto) o texto AAAA-MM-DD."""
    s = pd.Series(valores).astype(object)
    txt = s.map(lambda v: None if _es_na(v) else str(v).strip())
    num = pd.to_numeric(txt, errors="coerce")
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    m = num.notna()
    if m.any():
        out[m] = (pd.Timestamp("1899-12-30") + pd.to_timedelta(num[m].round(), unit="D")).astype("datetime64[ns]")
    r = txt.notna() & ~m
    if r.any():
        out[r] = pd.to_datetime(txt[r], errors="coerce", format="ISO8601").astype("datetime64[ns]")
    return out.dt.normalize()

def validar_fechas_mensuales(fechas, fuente, cp="CP3", texto_original=None):
    fechas = pd.Series(fechas).reset_index(drop=True)
    n_na = int(fechas.isna().sum())
    ej = ""
    if n_na and texto_original is not None:
        t = pd.Series(texto_original).reset_index(drop=True)
        ej = " (ej.: " + " | ".join(map(str, t[fechas.isna()].unique()[:3])) + ")"
    registrar(cp, fuente, "Fechas convertidas sin NA", n_na == 0, f"{n_na} fechas no convertidas{ej}")
    f = fechas.dropna()
    registrar(cp, fuente, "Fechas en día 1 (frecuencia mensual)", bool((f.dt.day == 1).all()),
              f"{int((f.dt.day != 1).sum())} fechas con día distinto de 1")
    u = pd.Series(sorted(f.unique()))
    if len(u) and (u.dt.day == 1).all():
        esperado = pd.date_range(u.min(), u.max(), freq="MS")
        huecos = esperado.difference(pd.DatetimeIndex(u))
        registrar(cp, fuente, "Meses consecutivos, sin huecos", len(huecos) == 0,
                  f"{rango_txt(u)} ({len(u)} meses)" if len(huecos) == 0
                  else f"{len(huecos)} meses sin registro: " + ", ".join(fmt_mes(h) for h in huecos[:12]),
                  si_falla="REVISAR")

def n_duplicados(df, llave):
    return int((df.groupby(llave, dropna=False).size() > 1).sum())

def validar_llave(df, llave, fuente, cp):
    d = n_duplicados(df, llave)
    registrar(cp, fuente, f"Llave única ({' + '.join(llave)})", d == 0, f"{d} combinaciones duplicadas de {len(df)} filas")

def resumir_limpieza(fuente, objeto, filas_iniciales, df, llave, columnas_originales, motivo):
    fechas = df["mes"] if "mes" in df else (df["fecha"] if "fecha" in df else pd.Series(dtype="datetime64[ns]"))
    n_cat = next((df[c].nunique() for c in ("rama", "serie", "naics", "id_serie", "pais") if c in df), None)
    REG["limpieza"].append(dict(
        fuente=fuente, objeto_limpio=objeto, filas_iniciales=int(filas_iniciales), filas_finales=len(df),
        explicacion_cambio_filas=motivo, columnas_originales=columnas_originales,
        columnas_homologadas=", ".join(map(str, df.columns)), rango_fechas=rango_txt(fechas),
        n_ramas_o_series=n_cat, llave=" + ".join(llave), duplicados_en_llave=n_duplicados(df, llave)))

def faltantes_por_variable(df, variables):
    return pd.DataFrame({"variable": variables, "n": len(df),
                         "n_faltantes": [int(df[v].isna().sum()) for v in variables]}).assign(
        pct_faltantes=lambda x: (100 * x.n_faltantes / max(len(df), 1)).round(2))

# ---- Lectura -------------------------------------------------------------------------
def leer_texto(ruta_archivo, hoja, usecols=None):
    x = pd.read_excel(ruta_archivo, sheet_name=hoja, header=None, dtype=str, usecols=usecols)
    x.columns = [f"c{i + 1}" for i in range(x.shape[1])]
    return x

def con_encabezado(x, fila=0):
    nm = [f"sin_nombre_{i + 1}" if _es_na(v) or str(v).strip() == "" else str(v).strip()
          for i, v in enumerate(x.iloc[fila])]
    vistos, unicos = {}, []
    for n in nm:
        vistos[n] = vistos.get(n, 0) + 1
        unicos.append(n if vistos[n] == 1 else f"{n}_{vistos[n] - 1}")
    y = x.iloc[fila + 1:].copy()
    y.columns = unicos
    return y.reset_index(drop=True)

def leer_csv_texto(ruta_archivo):
    return pd.read_csv(ruta_archivo, dtype=str, keep_default_na=False, na_values=[""])

def perfil_columnas(df, max_cols=15):
    filas = []
    for c in list(df.columns)[:max_cols]:
        v = df[c].dropna().astype(str)
        v = v[v.str.strip() != ""]
        num = pd.to_numeric(v, errors="coerce")
        filas.append(dict(columna=c, celdas_con_contenido=len(v), celdas_vacias=len(df) - len(v),
                          pct_convertible_a_numero=round(100 * num.notna().mean(), 1) if len(v) else None,
                          ejemplo=v.iloc[0] if len(v) else None))
    return pd.DataFrame(filas)

def inspeccionar(df, nombre, n_filas=5, n_cols=8, max_cols_perfil=15):
    display(Markdown(f"#### {nombre}"))
    nms = [str(c) for c in df.columns]
    display(Markdown(f"- **Dimensiones leídas:** {df.shape[0]:,} filas × {df.shape[1]} columnas\n"
                     f"- **Nombres de columna ({len(nms)}):** {html.escape(', '.join(nms[:20]))}{', ...' if len(nms) > 20 else ''}"))
    tabla(perfil_columnas(df, max_cols_perfil), f"Perfil de columnas (primeras {min(max_cols_perfil, df.shape[1])}; todo se leyó como texto)")
    sel = df.iloc[:, :n_cols]
    tabla(sel.head(n_filas), f"Primeras {n_filas} filas (primeras {sel.shape[1]} columnas)")
    tabla(sel.tail(n_filas), f"Últimas {n_filas} filas (primeras {sel.shape[1]} columnas)")

def iguales(a, b):
    """Igualdad elemento a elemento, con NA = NA."""
    a, b = pd.Series(a).reset_index(drop=True), pd.Series(b).reset_index(drop=True)
    if len(a) != len(b):
        return False
    ambos_na = a.isna() & b.isna()
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b) and not pd.api.types.is_bool_dtype(a):
        eq = np.isclose(a.astype(float), b.astype(float), rtol=1e-12, atol=0, equal_nan=True)
        return bool(eq.all())
    return bool(((a == b).fillna(False) | ambos_na).all())

### 1.3 Decisiones de integración

Todas las decisiones que no pueden resolverse solo con los archivos están aquí. Los valores actuales son los **acordados con el responsable del proyecto el 2026-09-24**; si se cambia alguno, el reporte lo marca como `REVISAR` (difiere de lo acordado).

In [3]:
decisiones = dict(
    # D1. Fuentes cuyas combinaciones (mes, rama) definen las filas del panel: el panel empieza en 2018-01 (EMIM).
    universo_llaves=["MX_PROD_01", "MX_EXP_01"],
    # D2. Census (NAICS 4) a la rama SCIAN con el mismo código, solo en ramas con correspondencia verificada.
    aranceles_por_codigo_identico=True,
    # D3. G.17: no trae total nacional (el control agregado de EE.UU. se incorporará aparte).
    ip_eeuu_subsector_codigo_identico=True,
    ip_eeuu_rama_codigo_identico=True,
    # D4. Variables monetarias también en miles de pesos reales de julio de 2019 (INPP del subsector; dólares con FIX).
    valores_en_miles_de_pesos_reales=True,
    # D5. Ceros del FIX de ene-oct 1991 = sin dato.
    tc_ceros_como_faltantes=True,
    # D6. VIX diario -> mensual: "promedio_mensual", "ultimo_dato_del_mes" o None (no se integra).
    vix_regla_mensual="promedio_mensual",
    # D7. Países de WUI/WTUI/WPUI y promedio mundial (acordado: las tres WTUI: MEX, USA y mundo).
    wui_paises=["MEX", "USA"],
    wui_incluir_promedio_mundial=True,
    # D8. Excluir los meses EMIM sin publicar ("-" en todas las celdas).
    emim_excluir_meses_no_publicados=True,
    # D9. Medida de exposición por rama: coeficiente de exportación de la MIP 2018 (exportaciones / producción bruta).
    #     Acordado el 2026-10-01 (opción 3): sin distinguir destino; no se usa el cociente con la EMIM ni la proporción a EE.UU.
    exposicion="coef_exportacion_mip_2018",
)

exigir(set(decisiones["universo_llaves"]) <= {"MX_PROD_01", "MX_EXP_01", "MX_ARAN_01"} and "MX_PROD_01" in decisiones["universo_llaves"],
       "universo_llaves solo admite MX_PROD_01, MX_EXP_01 y MX_ARAN_01, y debe incluir MX_PROD_01.")
exigir("MX_ARAN_01" not in decisiones["universo_llaves"] or decisiones["aranceles_por_codigo_identico"],
       "MX_ARAN_01 está en universo_llaves pero aranceles_por_codigo_identico = False.")
exigir(decisiones["vix_regla_mensual"] in (None, "promedio_mensual", "ultimo_dato_del_mes"),
       "vix_regla_mensual debe ser None, 'promedio_mensual' o 'ultimo_dato_del_mes'.")

si_no = lambda b: "Sí" if b else "No"
tabla_decisiones = pd.DataFrame([
    ("D1", "Fuentes que definen las filas mes-rama del panel", ", ".join(decisiones["universo_llaves"]), "MX_PROD_01, MX_EXP_01",
     "Panel desde 2018-01 (inicio de la EMIM). Los datos del Census de 2013-2017 quedan fuera del panel y se documentan en 'cobertura'."),
    ("D2", "Aranceles del Census por código NAICS 4 = SCIAN 4", si_no(decisiones["aranceles_por_codigo_identico"]), "Sí",
     "Solo en ramas con correspondencia verificada con los clasificadores oficiales de INEGI y del Census (checkpoint 4)."),
    ("D3a", "Producción EE.UU. (G.17) del subsector NAICS 3 idéntico", si_no(decisiones["ip_eeuu_subsector_codigo_identico"]), "Sí",
     "El G.17 no trae total nacional; el control agregado de EE.UU. se incorporará aparte."),
    ("D3b", "Producción EE.UU. (G.17) de la rama NAICS 4 idéntica", si_no(decisiones["ip_eeuu_rama_codigo_identico"]), "Sí",
     "Solo existe para parte de las ramas; en el resto queda NA. No se agregan series para completar."),
    ("D4", "Variables monetarias en miles de pesos reales (jul-2019)", si_no(decisiones["valores_en_miles_de_pesos_reales"]), "Sí",
     "Deflactor: INPP del subsector. Dólares a pesos con el FIX promedio del mes. Se conservan las nominales."),
    ("D5", "Ceros del tipo de cambio FIX como faltantes", si_no(decisiones["tc_ceros_como_faltantes"]), "Sí",
     "Los ceros de ene-oct 1991 no son valores reales; quedan fuera del panel en cualquier caso."),
    ("D6", "Regla para convertir el VIX diario a mensual", str(decisiones["vix_regla_mensual"]), "promedio_mensual",
     "Promedio de los días hábiles con dato en el mes."),
    ("D7", "Países de la WUI/WTUI/WPUI y promedio mundial",
     ", ".join(decisiones["wui_paises"]) + "; promedio mundial: " + si_no(decisiones["wui_incluir_promedio_mundial"]),
     "MEX, USA; promedio mundial: Sí", "Se usan las tres WTUI (wtui_mex, wtui_usa, wtui_mundo_pib); también WUI y WPUI de los mismos países."),
    ("D8", "Excluir meses EMIM no publicados ('-')", si_no(decisiones["emim_excluir_meses_no_publicados"]), "Sí",
     "Son celdas sin información (ago-dic 2026); conservarlas crearía filas vacías."),
    ("D9", "Medida de exposición por rama", decisiones["exposicion"], "coef_exportacion_mip_2018",
     "Acordada el 2026-10-01 (las demás, el 2026-09-24). El cociente exportaciones/producción con la EMIM supera 1 en 35 ramas (maquila) y la proporción "
     "Census/Banxico a EE.UU. está distorsionada por diferencias de clasificación; el coeficiente de la MIP usa producción bruta."),
], columns=["id", "decision", "valor_actual", "valor_acordado", "nota"])
for d in tabla_decisiones.itertuples():
    ok = d.valor_actual == d.valor_acordado
    registrar("D", f"Decisión {d.id}", d.decision, ok,
              f"Acordada: {d.valor_actual}. {d.nota}" if ok
              else f"El valor actual ({d.valor_actual}) difiere del acordado ({d.valor_acordado}). {d.nota}", si_falla="REVISAR")
tabla(tabla_decisiones, "Decisiones vigentes en esta ejecución", ancho=300)

**Decisiones vigentes en esta ejecución**

id,decision,valor_actual,valor_acordado,nota
D1,Fuentes que definen las filas mes-rama del panel,"MX_PROD_01, MX_EXP_01","MX_PROD_01, MX_EXP_01",Panel desde 2018-01 (inicio de la EMIM). Los datos del Census de 2013-2017 quedan fuera del panel y se documentan en 'cobertura'.
D2,Aranceles del Census por código NAICS 4 = SCIAN 4,Sí,Sí,Solo en ramas con correspondencia verificada con los clasificadores oficiales de INEGI y del Census (checkpoint 4).
D3a,Producción EE.UU. (G.17) del subsector NAICS 3 idéntico,Sí,Sí,El G.17 no trae total nacional; el control agregado de EE.UU. se incorporará aparte.
D3b,Producción EE.UU. (G.17) de la rama NAICS 4 idéntica,Sí,Sí,Solo existe para parte de las ramas; en el resto queda NA. No se agregan series para completar.
D4,Variables monetarias en miles de pesos reales (jul-2019),Sí,Sí,Deflactor: INPP del subsector. Dólares a pesos con el FIX promedio del mes. Se conservan las nominales.
D5,Ceros del tipo de cambio FIX como faltantes,Sí,Sí,Los ceros de ene-oct 1991 no son valores reales; quedan fuera del panel en cualquier caso.
D6,Regla para convertir el VIX diario a mensual,promedio_mensual,promedio_mensual,Promedio de los días hábiles con dato en el mes.
D7,Países de la WUI/WTUI/WPUI y promedio mundial,"MEX, USA; promedio mundial: Sí","MEX, USA; promedio mundial: Sí","Se usan las tres WTUI (wtui_mex, wtui_usa, wtui_mundo_pib); también WUI y WPUI de los mismos países."
D8,Excluir meses EMIM no publicados ('-'),Sí,Sí,Son celdas sin información (ago-dic 2026); conservarlas crearía filas vacías.
D9,Medida de exposición por rama,coef_exportacion_mip_2018,coef_exportacion_mip_2018,"Acordada el 2026-10-01 (las demás, el 2026-09-24). El cociente exportaciones/producción con la EMIM supera 1 en 35 ramas (maquila) y la proporción Census/Banxico a EE.UU. está distorsionada por diferencias de clasificación; el coeficiente de la MIP usa producción bruta."


### 1.4 Archivos de entrada y salida

Lista explícita de archivos (no se depende del orden en que el sistema los devuelva). Los originales **solo se leen**.

In [4]:
fuentes = pd.DataFrame([
    ("MX_PROD_01", "data/raw/produccion_mexico/inegi_emim_valores_rama_mensual.xlsx", "EMIM (INEGI): personal, horas, valor de producción y ventas"),
    ("MX_EXP_01", "data/raw/exportaciones/exportaciones_mex_scian_mensual.xlsx", "Exportaciones de México por clase SCIAN (Banxico, Comext)"),
    ("MX_ARAN_01", "data/raw/aranceles/aranceles_mex_naics4.csv", "Importaciones de EE.UU. desde México y aranceles (Census)"),
    ("MX_ARAN_02", "data/interim/aranceles/tasa_efectiva_mex_naics4.csv", "Tasa efectiva en formato ancho (derivado de MX_ARAN_01)"),
    ("US_PROD_01", "data/raw/produccion_estados_unidos/ip_naics_sa.csv", "Producción industrial de EE.UU. (Fed G.17), SA"),
    ("US_PROD_02", "data/raw/produccion_estados_unidos/ip_naics_series.csv", "Catálogo de series G.17"),
    ("INC_01", "data/raw/incertidumbre/tpu_caldara_iacoviello.xlsx", "TPU (Caldara, Iacoviello et al.)"),
    ("INC_02", "data/raw/incertidumbre/wui_ahir_bloom_furceri_2026_08.xlsx", "WUI / WTUI / WPUI (Ahir, Bloom y Furceri)"),
    ("INC_03", "data/raw/incertidumbre/epu_mexico_baker_bloom_davis.xlsx", "EPU México (Baker, Bloom y Davis)"),
    ("MACRO_01", "data/raw/controles_macroeconomicos/banxico_tipo_cambio_fix_mensual.xlsx", "Tipo de cambio FIX, promedio mensual (Banxico)"),
    ("MACRO_02", "data/raw/controles_macroeconomicos/fred_vix_diario.csv", "VIX diario (CBOE vía FRED)"),
    ("PRECIOS_01", "data/raw/precios/inegi_inpp_origen_scian_mensual.xlsx", "INPP por origen SCIAN (INEGI)"),
    ("CLAS_01", "data/raw/clasificadores/inegi_scian_2018_categorias.xlsx", "Catálogo SCIAN 2018 (INEGI); marca T = categoría trilateral"),
    ("CLAS_02", "data/raw/clasificadores/census_naics_2017_codigos.xlsx", "Códigos NAICS 2017 (Census)"),
    ("CLAS_03", "data/raw/clasificadores/census_naics_2022_codigos.xlsx", "Códigos NAICS 2022 (Census)"),
    ("CLAS_04", "data/raw/clasificadores/census_concordancia_naics_2022_2017.xlsx", "Concordancia NAICS 2022 -> 2017 (Census)"),
    ("CLAS_05", "data/raw/clasificadores/census_concordancia_naics_2017_2012.xlsx", "Concordancia NAICS 2017 -> 2012 (Census)"),
    ("MIP_01", "data/raw/mip/inegi_mip2018_coef_exportacion_rama.xlsx", "Coeficiente de exportación por rama, MIP 2018 (INEGI)"),
    ("DOC_CORR", "docs/correspondencia_rama_inpp.csv", "Correspondencia rama EMIM -> serie INPP del subsector"),
    ("DOC_INV", "docs/inventario_fuentes.csv", "Inventario de fuentes (huellas SHA-256)"),
], columns=["id", "ruta", "contenido"])

def ruta(id_):
    r = fuentes.loc[fuentes.id == id_, "ruta"]
    exigir(len(r) == 1, f"No existe la fuente '{id_}' en la tabla 'fuentes'.")
    return RAIZ / r.iloc[0]

DIR_PROCESADOS = RAIZ / "data" / "processed"
RUTA_EXCEL_FINAL = DIR_PROCESADOS / "base_incertidumbre_comercial_mexico.xlsx"
display(Markdown(f"**Archivo de salida:** `{RUTA_EXCEL_FINAL.relative_to(RAIZ)}`"))

**Archivo de salida:** `data/processed/base_incertidumbre_comercial_mexico.xlsx`

## 2. Inspección de los archivos originales
### Checkpoint 1: archivos de entrada

Existencia, tamaño, formato, hojas y prueba de lectura de cada archivo. Se calcula su huella MD5 (para comprobar al final que no cambiaron durante la ejecución) y se compara su SHA-256 con la del inventario.

In [5]:
titulo_checkpoint("CHECKPOINT 1 · Archivos de entrada")
inventario = leer_csv_texto(ruta("DOC_INV"))
filas = []
for f in fuentes.itertuples():
    p = RAIZ / f.ruta
    existe = p.exists()
    formato = p.suffix.lstrip(".").lower()
    hojas, n_hojas, lectura, md5, sha_ok = None, None, "no evaluada", None, None
    if existe:
        try:
            if formato == "xlsx":
                hojas_l = pd.ExcelFile(p, engine="openpyxl").sheet_names
                hojas, n_hojas = ", ".join(hojas_l), len(hojas_l)
                pd.read_excel(p, sheet_name=hojas_l[-1], header=None, nrows=3)
            else:
                p.open(encoding="utf-8").readline()
            lectura = "correcta"
        except Exception as ex:
            lectura = f"ERROR: {ex}"
        contenido = p.read_bytes()
        md5 = hashlib.md5(contenido).hexdigest()
        sha_inv = inventario.loc[inventario.nombre_organizado == f.ruta, "sha256"]
        if len(sha_inv) == 1:
            sha_ok = hashlib.sha256(contenido).hexdigest() == sha_inv.iloc[0]
    filas.append(dict(id=f.id, archivo=p.name, ruta=f.ruta, existe=existe,
                      tamano_kb=round(p.stat().st_size / 1024, 1) if existe else None, formato=formato,
                      n_hojas=n_hojas, hojas=hojas, lectura=lectura, md5=md5, sha256_igual_inventario=sha_ok))
cp1 = pd.DataFrame(filas)
for f in cp1.itertuples():
    registrar("CP1", f.id, "El archivo existe", f.existe, f.ruta)
    if f.existe:
        registrar("CP1", f.id, "El archivo se puede leer", f.lectura == "correcta", f.lectura)
        if not f.id.startswith("DOC_"):
            if f.sha256_igual_inventario is None:
                revisar("CP1", f.id, "Huella SHA-256 igual a la del inventario", "No evaluada: el archivo no tiene huella en el inventario.")
            else:
                registrar("CP1", f.id, "Huella SHA-256 igual a la del inventario", f.sha256_igual_inventario,
                          "El archivo es el mismo que se inventarió." if f.sha256_igual_inventario
                          else "El archivo cambió desde que se inventarió (¿nueva descarga?).", si_falla="REVISAR")
md5_inicio = dict(zip(cp1.id, cp1.md5))
tabla(cp1.drop(columns="md5"), "Archivos de entrada", ancho=150)
cerrar_checkpoint("CP1", "archivos de entrada")

**Archivos de entrada**

id,archivo,ruta,existe,tamano_kb,formato,n_hojas,hojas,lectura,sha256_igual_inventario
MX_PROD_01,inegi_emim_valores_rama_mensual.xlsx,data/raw/produccion_mexico/inegi_emim_valores_rama_mensual.xlsx,TRUE,915.5,xlsx,1,Tabulado,correcta,TRUE
MX_EXP_01,exportaciones_mex_scian_mensual.xlsx,data/raw/exportaciones/exportaciones_mex_scian_mensual.xlsx,TRUE,1637.7,xlsx,3,"Datos, TD, Autopartes",correcta,TRUE
MX_ARAN_01,aranceles_mex_naics4.csv,data/raw/aranceles/aranceles_mex_naics4.csv,TRUE,1642.3,csv,NA,NA,correcta,TRUE
MX_ARAN_02,tasa_efectiva_mex_naics4.csv,data/interim/aranceles/tasa_efectiva_mex_naics4.csv,TRUE,151.3,csv,NA,NA,correcta,TRUE
US_PROD_01,ip_naics_sa.csv,data/raw/produccion_estados_unidos/ip_naics_sa.csv,TRUE,1403.8,csv,NA,NA,correcta,TRUE
US_PROD_02,ip_naics_series.csv,data/raw/produccion_estados_unidos/ip_naics_series.csv,TRUE,9.5,csv,NA,NA,correcta,TRUE
INC_01,tpu_caldara_iacoviello.xlsx,data/raw/incertidumbre/tpu_caldara_iacoviello.xlsx,TRUE,332.7,xlsx,5,"ReadMe, Series_description, TPU_MONTHLY, TPU_QUARTERLY, TPU_DAILY",correcta,TRUE
INC_02,wui_ahir_bloom_furceri_2026_08.xlsx,data/raw/incertidumbre/wui_ahir_bloom_furceri_2026_08.xlsx,TRUE,631,xlsx,11,"T0, F1, T1, T2, T3, T4, T5, F2, T6, F3, T7",correcta,TRUE
INC_03,epu_mexico_baker_bloom_davis.xlsx,data/raw/incertidumbre/epu_mexico_baker_bloom_davis.xlsx,TRUE,19.3,xlsx,1,Mexican News-Based Index,correcta,TRUE
MACRO_01,banxico_tipo_cambio_fix_mensual.xlsx,data/raw/controles_macroeconomicos/banxico_tipo_cambio_fix_mensual.xlsx,TRUE,18.8,xlsx,1,Hoja1,correcta,TRUE


**Resumen del CP1 (archivos de entrada):** 58 OK · 0 REVISAR · 0 ERROR

fuente,prueba,resultado,estado
MX_PROD_01,El archivo existe,data/raw/produccion_mexico/inegi_emim_valores_rama_mensual.xlsx,OK
MX_PROD_01,El archivo se puede leer,correcta,OK
MX_PROD_01,Huella SHA-256 igual a la del inventario,El archivo es el mismo que se inventarió.,OK
MX_EXP_01,El archivo existe,data/raw/exportaciones/exportaciones_mex_scian_mensual.xlsx,OK
MX_EXP_01,El archivo se puede leer,correcta,OK
MX_EXP_01,Huella SHA-256 igual a la del inventario,El archivo es el mismo que se inventarió.,OK
MX_ARAN_01,El archivo existe,data/raw/aranceles/aranceles_mex_naics4.csv,OK
MX_ARAN_01,El archivo se puede leer,correcta,OK
MX_ARAN_01,Huella SHA-256 igual a la del inventario,El archivo es el mismo que se inventarió.,OK
MX_ARAN_02,El archivo existe,data/interim/aranceles/tasa_efectiva_mex_naics4.csv,OK


### Lectura cruda de todas las fuentes

Todas las hojas se leen **como texto**, sin que el programa adivine tipos. Cada columna se convierte después de forma explícita y se cuentan las celdas que no se pudieron convertir.

In [6]:
crudos = {}
crudos["emim"] = leer_texto(ruta("MX_PROD_01"), "Tabulado")
crudos["expo_datos"] = leer_texto(ruta("MX_EXP_01"), "Datos")
crudos["expo_td"] = leer_texto(ruta("MX_EXP_01"), "TD")
crudos["expo_auto"] = leer_texto(ruta("MX_EXP_01"), "Autopartes")
crudos["census"] = leer_csv_texto(ruta("MX_ARAN_01"))
crudos["aran2"] = leer_csv_texto(ruta("MX_ARAN_02"))
crudos["ip_sa"] = leer_csv_texto(ruta("US_PROD_01"))
crudos["ip_cat"] = leer_csv_texto(ruta("US_PROD_02"))
hojas_tpu = pd.ExcelFile(ruta("INC_01")).sheet_names
exigir({"ReadMe", "Series_description", "TPU_MONTHLY", "TPU_QUARTERLY", "TPU_DAILY"} <= set(hojas_tpu),
       "INC_01 (TPU): faltan hojas esperadas. Encontradas: ", ", ".join(hojas_tpu))
for h, k in [("ReadMe", "tpu_readme"), ("Series_description", "tpu_desc"), ("TPU_MONTHLY", "tpu_m"),
             ("TPU_QUARTERLY", "tpu_q"), ("TPU_DAILY", "tpu_d")]:
    crudos[k] = leer_texto(ruta("INC_01"), h)
hojas_wui = pd.ExcelFile(ruta("INC_02")).sheet_names
exigir({"T0", "T1", "T5", "T6", "T7", "F1", "F2", "F3"} <= set(hojas_wui), "INC_02 (WUI): faltan hojas esperadas.")
crudos["wui"] = {h: leer_texto(ruta("INC_02"), h) for h in hojas_wui}
hojas_epu = pd.ExcelFile(ruta("INC_03")).sheet_names
exigir(len(hojas_epu) == 1, "INC_03 (EPU): se esperaba una sola hoja; hay ", len(hojas_epu), ". Indique cuál contiene el índice.")
crudos["epu"] = leer_texto(ruta("INC_03"), hojas_epu[0])
crudos["tc"] = leer_texto(ruta("MACRO_01"), 0)
crudos["vix"] = leer_csv_texto(ruta("MACRO_02"))
crudos["inpp"] = leer_texto(ruta("PRECIOS_01"), "INPP")
crudos["corr"] = leer_csv_texto(ruta("DOC_CORR"))
crudos["mip"] = leer_texto(ruta("MIP_01"), 0)
tabla(pd.DataFrame([(k, *(v.shape if isinstance(v, pd.DataFrame) else (None, None)),
                     "hoja/archivo" if isinstance(v, pd.DataFrame) else f"{len(v)} hojas") for k, v in crudos.items()],
                   columns=["objeto", "filas", "columnas", "detalle"]), "Objetos leídos (lectura cruda, sin transformar)")

**Objetos leídos (lectura cruda, sin transformar)**

objeto,filas,columnas,detalle
emim,1286,111,hoja/archivo
expo_datos,"25,211.00",7,hoja/archivo
expo_td,95,88,hoja/archivo
expo_auto,92,9,hoja/archivo
census,"17,024.00",8,hoja/archivo
aran2,163,109,hoja/archivo
ip_sa,1292,232,hoja/archivo
ip_cat,231,3,hoja/archivo
tpu_readme,13,1,hoja/archivo
tpu_desc,5,1,hoja/archivo


### Checkpoint 2: estructura y contenido de las fuentes
#### MX_PROD_01 · EMIM (INEGI)
Cada fila es una combinación *variable × unidad SCIAN* (sector 31-33, subsector, rama o clase); cada columna, a partir de la cuarta, es un mes. Encabezado doble (año / mes en texto), cuatro variables apiladas en vertical y notas al pie. Formato **ancho**.

In [7]:
titulo_checkpoint("CHECKPOINT 2 · Estructura de las fuentes")
e = crudos["emim"]
inspeccionar(e, "EMIM · hoja Tabulado", n_filas=10, n_cols=6)
pos_hdr = np.flatnonzero(((e.c1 == "Variable") & (e.c2 == "Tipo de dato")).to_numpy())
registrar("CP2", "MX_PROD_01", "Se encuentran dos filas de encabezado consecutivas (año / mes)",
          len(pos_hdr) == 2 and pos_hdr[1] - pos_hdr[0] == 1, f"Filas detectadas: {list(pos_hdr)}")
datos_e = e[(e.c2.notna()) & (e.c2 != "Tipo de dato")]
tabla(datos_e.groupby(["c1", "c2"]).size().reset_index(name="filas").rename(columns={"c1": "variable", "c2": "tipo_dato"}),
      "Filas de datos por variable y tipo de dato")
cod = datos_e.c3.str.extract(r"^(\S+)")[0]
nivel_cp2 = np.select([cod == "31-33", cod.str.len() == 3, cod.str.len() == 4, cod.str.len() == 6],
                      ["sector 31-33", "subsector (3 dígitos)", "rama (4 dígitos)", "clase (6 dígitos)"], "no reconocido")
tabla(pd.Series(nivel_cp2).value_counts().rename_axis("nivel").reset_index(name="filas (4 variables)"),
      "Filas por nivel de agregación SCIAN (agregados y detalle conviven en la misma tabla)")
valores_e = datos_e.iloc[:, 3:].stack()
no_num = valores_e[pd.to_numeric(valores_e, errors="coerce").isna()]
tabla(no_num.value_counts().rename_axis("valor_no_numerico").reset_index(name="celdas"), "Celdas de datos no numéricas (códigos de la fuente)")
notas_cp2 = e.loc[(e.index > pos_hdr.max()) & e.c2.isna() & e.c1.notna(), "c1"]
tabla(pd.DataFrame({"nota_al_pie": notas_cp2}), "Notas al pie (no son datos)", ancho=250)

#### EMIM · hoja Tabulado

- **Dimensiones leídas:** 1,286 filas × 111 columnas
- **Nombres de columna (111):** c1, c2, c3, c4, c5, c6, c7, c8, c9, c10, c11, c12, c13, c14, c15, c16, c17, c18, c19, c20, ...

**Perfil de columnas (primeras 15; todo se leyó como texto)**

columna,celdas_con_contenido,celdas_vacias,pct_convertible_a_numero,ejemplo
c1,1273,13,0,Instituto Nacional de Estadística y Geografía (INEGI)
c2,1258,28,0,Tipo de dato
c3,1258,28,0,Sector SCIAN
c4,1258,28,99.9,2018
c5,1258,28,99.9,2018
c6,1258,28,99.9,2018
c7,1258,28,99.9,2018
c8,1258,28,99.9,2018
c9,1258,28,99.9,2018
c10,1258,28,99.9,2018


**Primeras 10 filas (primeras 6 columnas)**

...

nivel,filas (4 variables)
clase (6 dígitos),824
rama (4 dígitos),344
subsector (3 dígitos),84
sector 31-33,4


**Celdas de datos no numéricas (códigos de la fuente)**

valor_no_numerico,celdas
-,6280
ND,184


**Notas al pie (no son datos)**

nota_al_pie
Notas:
No Calculable: NC Dato no calculable
No Disponible: ND No disponible por motivos de confidencialidad.
Cifras preliminares: Cifras preliminares a partir de Ago 2025
Cifras revisadas: Cifras revisadas a partir de Ene 2025
"Nota general: Los datos de la EMIM serie 2018 fueron revisados y actualizados por la incorporación de cuestionarios recibidos de forma extemporánea, así como por los datos aclarados por los informantes. Conforme a lo anterior, el estatus de las se..."
"El 14 de agosto del 2023 la EMIM actualiza la denominación de dos capítulos, 12 variables y 15 índices, para dar mayor claridad a su descripción. Además, se incorporan para su difusión, dos nuevos índices: Índice de remuneraciones medias reales de..."
"Se modifican las cifras de dos índices del periodo 2007 a 2017 por el proceso de retropolación, el primero: índice de remuneraciones pagadas al personal ocupado, antes se utilizaba el índice de remuneraciones totales base 2013=100 que incluía las ..."
Nota aclaratoria: El personal ocupado reportado en este Programa Estadístico es conceptualmente diferente (ver metadatos) a la definición de puestos de trabajo afiliados al Instituto Mexicano del Seguro Social.
"Las clases de actividad 313113 Fabricación de hilos para coser y bordar y 314912 Confección de productos de textiles recubiertos y de materiales sucedáneos, no son consideradas como dominios de estudio, aunque sí formarán parte de la oferta estadí..."


#### MX_EXP_01 · Exportaciones por clase SCIAN (Banxico, Comext)
**Hoja `Datos`:** cada fila es *año × mes × clase SCIAN de 6 dígitos*, formato **largo**, con valor en pesos y en dólares; la columna `Clase` contiene en realidad la **rama** (4 dígitos). **`TD`:** tabla dinámica del usuario (solo para verificar la agregación). **`Autopartes`:** hoja de trabajo del usuario de origen no documentado; **no se integra**.

In [8]:
x = con_encabezado(crudos["expo_datos"], 0)
inspeccionar(x, "Exportaciones · hoja Datos", n_cols=7)
esperadas = ["Tipo de operación", "Año", "Mes", "SCIAN", "Clase", "Valor en pesos", "Valor en dólares"]
registrar("CP2", "MX_EXP_01", "Hoja Datos: columnas esperadas", [norm_txt(c) for c in x.columns] == [norm_txt(c) for c in esperadas],
          "Columnas leídas: " + ", ".join(x.columns))
tabla(x.iloc[:, 1].value_counts().sort_index().rename_axis("anio").reset_index(name="filas"), "Filas por año")
por_mes = x.groupby([x.iloc[:, 1], x.iloc[:, 2]]).size()
tabla(pd.DataFrame([dict(meses=len(por_mes), min_clases_por_mes=por_mes.min(), max_clases_por_mes=por_mes.max())]),
      "Clases con registro por mes (no todas las clases exportan todos los meses)")
inspeccionar(crudos["expo_td"], "Exportaciones · hoja TD (tabla dinámica del usuario)", n_cols=6, max_cols_perfil=6)
inspeccionar(crudos["expo_auto"], "Exportaciones · hoja Autopartes (hoja de trabajo del usuario)", n_cols=9, max_cols_perfil=9)
revisar("CP2", "MX_EXP_01", "Hoja Autopartes",
        "Serie derivada de origen no documentado (xparts no coincide con la rama 3363 de Datos; xparts_r_ae parece desestacionalizada). No se integra.")

#### Exportaciones · hoja Datos

- **Dimensiones leídas:** 25,210 filas × 7 columnas
- **Nombres de columna (7):** Tipo de operación, Año, Mes, SCIAN, Clase, Valor en pesos, Valor en dólares

**Perfil de columnas (primeras 7; todo se leyó como texto)**

columna,celdas_con_contenido,celdas_vacias,pct_convertible_a_numero,ejemplo
Tipo de operación,"25,210",0,100,2
Año,"25,210",0,100,2019
Mes,"25,210",0,100,1
SCIAN,"25,210",0,100,311110
Clase,"25,210",0,100,3111
Valor en pesos,"25,210",0,100,466100000
Valor en dólares,"25,210",0,100,24316759


**Primeras 5 filas (primeras 7 columnas)**

...

**Primeras 5 filas (primeras 9 columnas)**

c1,c2,c3,c4,c5,c6,c7,c8,c9
fecha,xparts,inpp336,xparts_r,NA,fecha,xparts_r,xparts_r_ae,xparts
2019-01-01 00:00:00,4978000000,99.171466028507,5019589000.095114,NA,2019-01-01 00:00:00,5019589000.095114,6555408182.19671,NA
2019-02-01 00:00:00,5372000000,99.024922850082,5424896930.374686,NA,2019-02-01 00:00:00,5424896930.374686,6205956665.68436,-5.330736192162618
2019-03-01 00:00:00,4664000000,99.127384899837,4705057038.186498,NA,2019-03-01 00:00:00,4705057038.186498,4399392836.32411,-29.11015862146745
2019-04-01 00:00:00,5876000000,99.124183456997,5927917683.729705,NA,2019-04-01 00:00:00,5927917683.729705,6448543998.18445,46.57804470065233


**Últimas 5 filas (primeras 9 columnas)**

c1,c2,c3,c4,c5,c6,c7,c8,c9
2026-03-01 00:00:00,12394816820,116.964041638808,10597117410.046364,NA,2026-03-01 00:00:00,10597117410.046364,9604290685.57477,7.654420168174241
2026-04-01 00:00:00,11226733095,116.579361522975,9630120587.671497,NA,2026-04-01 00:00:00,9630120587.671497,9584331919.2021,-0.20781093602931833
2026-05-01 00:00:00,9971364237,116.352247746194,8569979893.083908,NA,2026-05-01 00:00:00,8569979893.083908,8760351861.27755,-8.597156952314194
2026-06-01 00:00:00,12193187404,116.413843639372,10474001220.827465,NA,2026-06-01 00:00:00,10474001220.827465,10353129117.0569,18.181658465337833
2026-07-01 00:00:00,11145753912,116.991063350503,9527013083.561377,NA,2026-07-01 00:00:00,9527013083.561377,9197844119.02068,-11.158800252310925


'REVISAR'

#### MX_ARAN_01 y MX_ARAN_02 · Importaciones de EE.UU. desde México y aranceles (Census)
**MX_ARAN_01:** cada fila es *mes × código NAICS de 4 dígitos*, formato **largo**; dólares corrientes y tasas en proporción. Incluye códigos no manufactureros. **MX_ARAN_02:** la misma tasa efectiva en formato ancho; solo se usa para verificar.

In [9]:
inspeccionar(crudos["census"], "Census · aranceles_mex_naics4.csv", n_cols=8)
registrar("CP2", "MX_ARAN_01", "Columnas esperadas",
          list(crudos["census"].columns) == ["fecha", "naics", "descripcion", "valor_importado", "valor_gravable", "aranceles", "tasa_efectiva", "tasa_gravable"],
          ", ".join(crudos["census"].columns))
tipo_c = np.select([crudos["census"].naics.str.match(r"^3\d{3}$"), crudos["census"].naics.str.match(r"^1\d{3}$"),
                    crudos["census"].naics.str.match(r"^2\d{3}$"), crudos["census"].naics.str.match(r"^9\d{3}$")],
                   ["manufacturero (3xxx)", "agropecuario (11xx)", "minero (21xx)", "clasificación especial (9xxx)"], "otro")
g = crudos["census"].assign(tipo=tipo_c).groupby("tipo")
tabla(pd.DataFrame({"codigos": g.naics.nunique(), "filas": g.size(),
                    "meses_min": g.naics.apply(lambda s: s.value_counts().min()),
                    "meses_max": g.naics.apply(lambda s: s.value_counts().max())}).reset_index(), "Códigos NAICS por tipo")
inspeccionar(crudos["aran2"], "Census derivado · tasa_efectiva_mex_naics4.csv (formato ancho)", n_cols=8, max_cols_perfil=8)

#### Census · aranceles_mex_naics4.csv

- **Dimensiones leídas:** 17,024 filas × 8 columnas
- **Nombres de columna (8):** fecha, naics, descripcion, valor_importado, valor_gravable, aranceles, tasa_efectiva, tasa_gravable

**Perfil de columnas (primeras 8; todo se leyó como texto)**

columna,celdas_con_contenido,celdas_vacias,pct_convertible_a_numero,ejemplo
fecha,"17,024",0,0,2013-01-01
naics,"17,024",0,100,1111
descripcion,"17,024",0,0,OILSEEDS & GRAINS
valor_importado,"17,024",0,100,4365826
valor_gravable,"17,024",0,100,0
aranceles,"17,024",0,100,0
tasa_efectiva,"16,871",153,100,0.0
tasa_gravable,"13,417",3607,100,0.0009748794457114008


**Primeras 5 filas (primeras 8 columnas)**

...

columna,celdas_con_contenido,celdas_vacias,pct_convertible_a_numero,ejemplo
fecha,163,0,0,2013-01-01
1111,163,0,100,0.000000
1112,163,0,100,0.000000
1113,163,0,100,0.000001
1114,163,0,100,0.000000
1119,163,0,100,0.000000
1121,148,15,100,0.000000
1123,21,142,100,0.000000


**Primeras 5 filas (primeras 8 columnas)**

fecha,1111,1112,1113,1114,1119,1121,1123
2013-01-01,0.000000,0.000000,0.000001,0.000000,0.000000,0.000000,NA
2013-02-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000003,NA
2013-03-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NA
2013-04-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000001,NA
2013-05-01,0.000000,0.000002,0.000000,0.000043,0.000000,0.000000,NA


**Últimas 5 filas (primeras 8 columnas)**

fecha,1111,1112,1113,1114,1119,1121,1123
2026-03-01,0.000520,0.000197,0.000002,0.000027,0.000036,NA,NA
2026-04-01,0.000000,0.000024,0.000000,0.000000,0.000000,NA,NA
2026-05-01,0.000000,0.000015,0.000024,0.000020,0.000000,NA,NA
2026-06-01,0.000000,0.000022,0.000005,0.000010,0.003860,NA,NA
2026-07-01,0.000000,0.000069,0.000016,0.000051,0.000743,NA,NA


#### US_PROD_01 y US_PROD_02 · Producción industrial de EE.UU. (Fed G.17)
Cada fila es un mes y cada columna una serie (formato **ancho**), índice con ajuste estacional. **Hallazgo clave:** el archivo **no contiene una serie nacional total**; solo series por industria con códigos NAICS heterogéneos.

In [10]:
inspeccionar(crudos["ip_sa"], "G.17 · ip_naics_sa.csv", n_cols=8, max_cols_perfil=8)
inspeccionar(crudos["ip_cat"], "G.17 · ip_naics_series.csv (catálogo)", n_cols=3)

def tipo_naics(n):
    n = pd.Series(n).astype(str)
    return pd.Series(np.select(
        [n.str.match(r"^\d{3}$"), n.str.match(r"^\d{4}$"), n.str.match(r"^\d{2}$"), n.str.match(r"^\d{5,6}$"),
         n.str.contains(r"pt\.", regex=True), n.str.contains(r"[-,]", regex=True)],
        ["NAICS 3 dígitos", "NAICS 4 dígitos", "NAICS 2 dígitos", "NAICS 5-6 dígitos", "parte de industria (pt.)", "rango o lista de códigos"],
        "otro"), index=n.index)

tabla(crudos["ip_cat"].assign(tipo_codigo=tipo_naics(crudos["ip_cat"].naics),
                              manufactura=crudos["ip_cat"].naics.str.match(r"^3[1-3]"))
      .groupby(["tipo_codigo", "manufactura"]).size().reset_index(name="series"), "Series G.17 por tipo de código NAICS")
total = crudos["ip_cat"][crudos["ip_cat"].descripcion.map(norm_txt).str.contains(r"^total\b|^manufacturing\b|industrial production", regex=True)]
registrar("CP2", "US_PROD_02", "¿Existe una serie nacional total (industria o manufactura)?", len(total) > 0,
          ", ".join(total.serie) if len(total) else "No: solo series por industria. La producción de EE.UU. no puede entrar como variable nacional común.",
          si_falla="REVISAR")

#### G.17 · ip_naics_sa.csv

- **Dimensiones leídas:** 1,292 filas × 232 columnas
- **Nombres de columna (232):** fecha, G321, G327, G331, G332, G333, G334, G335, G3361T3, G3364T9, G337, G339, G311A2, G313A4, G315A6, G322, G323, G324, G325, G326, ...

**Perfil de columnas (primeras 8; todo se leyó como texto)**

columna,celdas_con_contenido,celdas_vacias,pct_convertible_a_numero,ejemplo
fecha,1292,0,0,1919-01-01
G321,656,636,100,84.2003
G327,656,636,100,84.4233
G331,656,636,100,113.6578
G332,656,636,100,67.4751
G333,656,636,100,57.8087
G334,656,636,100,0.4069
G335,656,636,100,79.0296


**Primeras 5 filas (primeras 8 columnas)**

...

**Últimas 5 filas (primeras 3 columnas)**

serie,descripcion,naics
G5131P,"Newspaper, periodical, book, and directory publishers",5131pt.
G51311P,Newspaper publishers,51311pt.
G51312T9P,"Periodical, book, and other publishers",51312-9pt.
G33611B,"Automobile, business",336110pt.
G33611U,"Light trucks, business",336110pt.


**Series G.17 por tipo de código NAICS**

tipo_codigo,manufactura,series
NAICS 2 dígitos,FALSE,1
NAICS 3 dígitos,FALSE,3
NAICS 3 dígitos,TRUE,21
NAICS 4 dígitos,FALSE,6
NAICS 4 dígitos,TRUE,67
NAICS 5-6 dígitos,FALSE,11
NAICS 5-6 dígitos,TRUE,50
parte de industria (pt.),FALSE,12
parte de industria (pt.),TRUE,28
rango o lista de códigos,FALSE,3


'REVISAR'

#### INC_01 · TPU (Caldara, Iacoviello et al.)
Hojas **mensual** (se integra), **trimestral** y **diaria** (no se integran). La hoja `ReadMe` dice que la cobertura es 1985-2020, pero los datos van más allá.

In [11]:
tabla(crudos["tpu_readme"][["c1"]].dropna().rename(columns={"c1": "texto"}), "Hoja ReadMe", ancho=200)
tabla(crudos["tpu_desc"][["c1"]].dropna().rename(columns={"c1": "texto"}), "Hoja Series_description", ancho=400)
inspeccionar(con_encabezado(crudos["tpu_m"]), "TPU · hoja TPU_MONTHLY", n_cols=6)
inspeccionar(con_encabezado(crudos["tpu_q"]), "TPU · hoja TPU_QUARTERLY", n_cols=3)
inspeccionar(con_encabezado(crudos["tpu_d"]), "TPU · hoja TPU_DAILY", n_cols=6)

**Hoja ReadMe**

texto
(1) TPU_MONTHLY
Contains:
DATE - monthly from 1985-2020
N7 - total number of articles across 7 major newspapers
TPU_RAW - raw count of TPU-related newspaper articles
TPU_SHARE - percent share of TPU newspaper articles
TPU - normalized index of newspaper articles share; equals 100 for an article share of 1 %
(2) TPU_QUARTERLY
Contains:
DATEQ - quarterly from 1985-2020


**Hoja Series_description**

texto
Description of the series
"TPU : The trade policy uncertainty index (TPU) is constructed by staff in the International Finance Division of the Federal Reserve Board and measures media attention to news related to trade policy uncertainty. The index reflects automated text-search results of the electronic archives of 7 leading newspapers discussing trade policy uncertainty: Boston Globe, Chicago Tribune, Guardian, Los Ang..."
"Cite as: Caldara, Dario, Matteo Iacoviello, Patrick Molligo, Andrea Prestipino, and Andrea Raffo, ""The Economic Effects of Trade Policy Uncertainty,"" manuscript presented at the 91st meeting of the Carnegie-Rochester-NYU Conference on Public Policy, April 2019."
"The search terms are: ((""tariff*"" OR ""import duty"" OR ""import barrier"" OR ""trade treaty"" OR ""trade policy"" OR ""trade act"" OR dumping OR ""import fee"" OR (tax n/10 (""foreign goods"" OR ""foreign oil"" OR ""import*"")) OR ((imports OR import OR imported) n/10 (surtax OR surcharge*))) AND (uncertainty OR uncertain OR risk* OR dubious OR unclear OR potential OR probabl* OR predict* OR danger*))."


#### TPU · hoja TPU_MONTHLY

...

columna,celdas_con_contenido,celdas_vacias,pct_convertible_a_numero,ejemplo
DATE,4282,0,0,2015-01-01 00:00:00
NUMBER_ARTICLES,4282,0,100,690
TPUD_ARTICLES,4282,0,100,0
TPUD_index,4282,0,100,0
TPUD_index_MA7,4276,6,100,27.0948752645567
TPUD_index_MA30,4253,29,100,23.6130232402306


**Primeras 5 filas (primeras 6 columnas)**

DATE,NUMBER_ARTICLES,TPUD_ARTICLES,TPUD_index,TPUD_index_MA7,TPUD_index_MA30
2015-01-01 00:00:00,690,0,0,NA,NA
2015-01-02 00:00:00,726,1,13.7741046831955,NA,NA
2015-01-03 00:00:00,746,1,13.4048257372654,NA,NA
2015-01-04 00:00:00,765,2,26.1437908496732,NA,NA
2015-01-05 00:00:00,593,5,84.3170320404721,NA,NA


**Últimas 5 filas (primeras 6 columnas)**

DATE,NUMBER_ARTICLES,TPUD_ARTICLES,TPUD_index,TPUD_index_MA7,TPUD_index_MA30
2026-09-17 00:00:00,466,16,343.3476394849785,252.39105748808,281.076342847685
2026-09-18 00:00:00,499,13,260.5210420841684,246.0353644649344,282.8014353430445
2026-09-19 00:00:00,416,14,336.5384615384615,259.4382237415801,281.4670828057632
2026-09-20 00:00:00,608,14,230.2631578947368,263.9036784230599,281.1191807749751
2026-09-21 00:00:00,309,8,258.8996763754045,272.8781419948965,279.7741076433489


#### INC_02 · World Uncertainty Index (Ahir, Bloom y Furceri)
`T1` (WUI), `T6` (WTUI, comercial) y `T7` (WPUI, política): mes × país (ISO3), formato **ancho**. `F1`-`F3`: promedio mundial ponderado por PIB (escala distinta) y celdas de gráficas. Nota de la fuente: *"For countries with missing reports, the previous month's value is used"* (arrastre hecho por la fuente).

In [12]:
t0 = crudos["wui"]["T0"]
tabla(t0[t0.c1.notna() | t0.c2.notna()][["c1", "c2", "c4"]].rename(columns={"c1": "hoja", "c2": "descripcion", "c4": "actualizacion"}),
      "Hoja T0 (descripción de hojas)", ancho=350)
for h in ("T1", "T6", "T7"):
    inspeccionar(con_encabezado(crudos["wui"][h]), f"WUI · hoja {h}", n_cols=6, max_cols_perfil=6)
inspeccionar(crudos["wui"]["F2"], "WUI · hoja F2 (promedio mundial WTUI y celdas de gráfica)", n_cols=6, max_cols_perfil=6)
t5 = con_encabezado(crudos["wui"]["T5"])
tabla(t5[t5.iso_code.isin(decisiones["wui_paises"])], "Países seleccionados en el catálogo T5")

**Hoja T0 (descripción de hojas)**

hoja,descripcion,actualizacion
Tab,Description,Update
F1,This tab contains the figure for the World Uncertainty Index (WUI). This index measures overall uncertainty across the globe. The index is GDP weighted average for 71 countries.,Aug-2026 data added.
T1,"This tab contains the time series of the World Uncertainty Index (WUI) for 71 countries from January 2008 onwards. All indices have been computed by counting the frequency of the word ""uncertainty"" (or its variant) in EIU country reports. The indices are normalized by total number of words and rescaled by multiplying by 1,000. A higher number m...",Aug-2026 data added.
T2,"This tab contains the time series of the total count of the word ""uncertainty"" (or its variant) in the EIU country reports for 71 countries from January 2008 onwards. A higher number means higher uncertainty and vice versa.",Aug-2026 data added.
T3,This tab contains the time series of the total number of words in the EIU reports for 71 countries from January 2008 onwards.,Aug-2026 data added.
T4,This tab contains the time series of the total number of pages in the EIU countries reports for 71 countries from January 2008 onwards.,Aug-2026 data added.
T5,"This tab contains the list of country names and the respective ISO codes, WEO codes, and country classifications. Under the regions column, AFR = Africa, APD = Asia and the Pacific, EUR = Europe, MCD = Middle East and Central Asia, and WHD = Western Hemisphere. Under the IMF_income column, 1 = advanced economies, 2 = emerging economies, and 3 = ...",No update.
F2,This tab contains the figure for the World Trade Uncertainty Index (WTUI). This index measures trade uncertainty across the globe. The index is GDP weighted average for 71 countries.,Aug-2026 data added.
T6,"This tab contains the time series of the World Trade Uncertainty Index (WTUI) for 71 countries from January 2008 onwards. All indices have been computed by counting the frequency of the word uncertainty (or its variant) near to a word related to trade (tariff, trade, export, import, World Trade Organization (WTO), North American Free Trade Agree...",Aug-2026 data added.
F3,This tab contains the figure for the World Policy Uncertainty Index (WPUI). This index measures policy uncertainty across the globe. The index is GDP weighted average for 71 countries.,Aug-2026 data added.


#### WUI · hoja T1

- **Dimensiones leídas:** 224 filas × 72 columnas
- **Nombres de columna (72):** date, AGO, ARE, ARG, AUS, AUT, AZE, BEL, BGD, BOL, BRA, CAN, CHE, CHL, CHN, COL, CRI, CZE, DEU, DOM, ...

**Perfil de columnas (primeras 6; todo se leyó como texto)**

...

c1,c2,c3,c4,c5,c6
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA
date,"WTUI, GDP weighted average",NA,NA,NA,NA
2008-01-01 00:00:00,2.321501,NA,NA,NA,NA
2008-02-01 00:00:00,262.6446,NA,NA,NA,"World Trade Uncertainty Index, GDP weighted average"


**Últimas 5 filas (primeras 6 columnas)**

c1,c2,c3,c4,c5,c6
2026-04-01 00:00:00,35611.99,NA,NA,NA,NA
2026-05-01 00:00:00,29840.7,NA,NA,NA,NA
2026-06-01 00:00:00,27182.7,NA,NA,NA,NA
2026-07-01 00:00:00,25970.26,NA,NA,NA,NA
2026-08-01 00:00:00,25761.29,NA,NA,NA,NA


**Países seleccionados en el catálogo T5**

iso_code,weo_code,country_name,region,IMF_income
MEX,273,Mexico,WHD,2
USA,111,United States,WHD,1


#### INC_03 · EPU México · MACRO_01 · Tipo de cambio FIX · MACRO_02 · VIX · PRECIOS_01 · INPP · DOC_CORR
- **EPU:** `Year` (texto) y `Month`, formato **largo**; la última fila es una nota de la fuente.
- **FIX:** metadatos de Banxico y luego `Fecha` / `SF17908`; ene-oct 1991 vienen en 0 (sin dato).
- **VIX:** un **día hábil** por fila; feriados vacíos. Se convierte a mensual con la regla D6 (promedio mensual).
- **INPP:** metadatos (título, periodo, unidad) y una fila `Fecha` con las claves de 43 series; `N/E` = la serie aún no existe.
- **Correspondencia rama → INPP:** tabla del proyecto que asigna a cada rama la serie de su subsector.
- **Coeficiente de exportación MIP 2018:** una fila por rama (código, «código - nombre», proporción exportaciones / producción bruta); un dato por rama, sin dimensión temporal.

In [13]:
inspeccionar(crudos["epu"], "EPU México", n_cols=3)
inspeccionar(crudos["tc"], "Tipo de cambio FIX", n_filas=20, n_cols=2)
inspeccionar(crudos["vix"], "VIX diario", n_cols=2)
inspeccionar(crudos["inpp"], "INPP", n_filas=14, n_cols=5, max_cols_perfil=8)
inspeccionar(crudos["corr"], "correspondencia_rama_inpp.csv", n_cols=6)
inspeccionar(crudos["mip"], "Coeficiente de exportación MIP 2018 (una fila por rama: código, código - nombre, proporción)", n_cols=3)
cerrar_checkpoint("CP2", "estructura de las fuentes")

#### EPU México

- **Dimensiones leídas:** 370 filas × 3 columnas
- **Nombres de columna (3):** c1, c2, c3

**Perfil de columnas (primeras 3; todo se leyó como texto)**

columna,celdas_con_contenido,celdas_vacias,pct_convertible_a_numero,ejemplo
c1,370,0,99.5,Year
c2,369,1,99.7,Month
c3,369,1,99.7,Mexican Policy Uncertainty Index


**Primeras 5 filas (primeras 3 columnas)**

...

c1,c2,c3
NA,Rama,Export
3111,3111 - Elaboración de alimentos para animales,0.028869465781660077
3112,3112 - Molienda de granos y de semillas y obtención de aceites y grasas,0.06098070806760998
3113,"3113 - Elaboración de azúcares, chocolates, dulces y similares",0.20164235682187914
3114,"3114 - Conservación de frutas, verduras, guisos y otros alimentos preparados",0.31565131496087295


**Últimas 5 filas (primeras 3 columnas)**

c1,c2,c3
3371,"3371 - Fabricación de muebles, excepto de oficina y estantería",0.36422781886433614
3372,3372 - Fabricación de muebles de oficina y estantería,0.22143805147332168
3379,"3379 - Fabricación de colchones, persianas y cortineros",0.3523025337449802
3391,"3391 - Fabricación de equipo no electrónico y material desechable de uso médico, d...",0.7835186733412709
3399,3399 - Otras industrias manufactureras,0.618987370047973


**Resumen del CP2 (estructura de las fuentes):** 3 OK · 2 REVISAR · 0 ERROR

fuente,prueba,resultado,estado
MX_PROD_01,Se encuentran dos filas de encabezado consecutivas (año / mes),"Filas detectadas: [np.int64(6), np.int64(7)]",OK
MX_EXP_01,Hoja Datos: columnas esperadas,"Columnas leídas: Tipo de operación, Año, Mes, SCIAN, Clase, Valor en pesos, Valor en dólares",OK
MX_EXP_01,Hoja Autopartes,Serie derivada de origen no documentado (xparts no coincide con la rama 3363 de Datos; xparts_r_ae parece desestacionalizada). No se integra.,REVISAR
MX_ARAN_01,Columnas esperadas,"fecha, naics, descripcion, valor_importado, valor_gravable, aranceles, tasa_efectiva, tasa_gravable",OK
US_PROD_02,¿Existe una serie nacional total (industria o manufactura)?,No: solo series por industria. La producción de EE.UU. no puede entrar como variable nacional común.,REVISAR


## 3. Limpieza individual de cada fuente
### MX_PROD_01 · EMIM
(1) Encabezado doble → fecha; (2) solo filas `Dato`; (3) formato largo; (4) código SCIAN y nivel; (5) conversión a número distinguiendo `ND` (confidencial), `-` (no publicado) y `NC`; (6) nombres homologados con un diccionario explícito; (7) tabla *mes × rama*.

In [14]:
e = crudos["emim"]
pos_hdr = np.flatnonzero(((e.c1 == "Variable") & (e.c2 == "Tipo de dato")).to_numpy())
exigir(len(pos_hdr) == 2 and pos_hdr[1] - pos_hdr[0] == 1, "EMIM: no se encontraron las dos filas de encabezado consecutivas.")
f_anio, f_mes = pos_hdr
anios_hdr, meses_hdr = e.iloc[f_anio, 3:], e.iloc[f_mes, 3:]
cols_fecha = [i + 3 for i in range(len(anios_hdr)) if pd.notna(anios_hdr.iloc[i]) and pd.notna(meses_hdr.iloc[i])]
cols_sin_fecha = [c for c in range(3, e.shape[1]) if c not in cols_fecha]
exigir(not cols_sin_fecha or e.iloc[:, cols_sin_fecha].isna().all().all(), "EMIM: hay columnas con datos pero sin año/mes.")
meses_num = [MESES_ES.get(norm_txt(e.iat[f_mes, c])) for c in cols_fecha]
exigir(None not in meses_num, "EMIM: meses no reconocidos en el encabezado.")
fechas_emim = [pd.Timestamp(int(e.iat[f_anio, c]), m, 1) for c, m in zip(cols_fecha, meses_num)]
exigir(list(pd.date_range(fechas_emim[0], periods=len(fechas_emim), freq="MS")) == fechas_emim,
       "EMIM: las columnas de meses no forman una secuencia mensual consecutiva.")

pos_dato = np.flatnonzero((e.c2 == "Dato").to_numpy())
exigir(len(pos_dato) > 0 and pos_dato.min() > f_mes and (np.diff(pos_dato) == 1).all(),
       "EMIM: las filas 'Dato' no forman un bloque contiguo después del encabezado.")
otros = set(e.c2.iloc[f_mes + 1: pos_dato.max() + 1].dropna()) - {"Dato"}
exigir(not otros, "EMIM: tipos de dato no previstos: ", otros)
filas_no_dato = sorted(set(range(len(e))) - set(range(f_mes + 1)) - set(pos_dato))
notas_emim = [v for v in e.c1.iloc[filas_no_dato] if pd.notna(v) and str(v).strip()]
excluir("MX_PROD_01", f_mes + 1, "Filas de título, fecha de consulta y encabezado doble (no son datos)")
excluir("MX_PROD_01", len(filas_no_dato), "Filas de notas al pie y filas vacías después del bloque de datos")

dicc_emim = pd.DataFrame([
    ("Personal ocupado total (Número de personas)", "emim_personal_ocupado", "Número de personas"),
    ("Horas trabajadas por el personal ocupado total (Miles de horas)", "emim_horas_trabajadas", "Miles de horas"),
    ("Total de valor de producción de los productos elaborados (Miles de pesos corrientes)", "emim_valor_produccion", "Miles de pesos corrientes"),
    ("Total de valor de ventas de los productos elaborados (Miles de pesos corrientes)", "emim_valor_ventas", "Miles de pesos corrientes"),
], columns=["variable_original", "variable", "unidad"])
dicc_emim["clave"] = dicc_emim.variable_original.map(norm_txt)

col_fechas = [f.strftime("%Y-%m-%d") for f in fechas_emim]
bloque = e.iloc[pos_dato, [0, 1, 2] + cols_fecha].copy()
bloque.columns = ["variable_original", "tipo_dato", "unidad_scian"] + col_fechas
emim_largo = bloque.melt(id_vars=["variable_original", "tipo_dato", "unidad_scian"], var_name="mes", value_name="valor_txt")
emim_largo["mes"] = a_mes(emim_largo["mes"])
emim_largo["variable_original"] = emim_largo.variable_original.str.strip().str.replace(r"^-", "", regex=True).str.strip()
emim_largo["clave"] = emim_largo.variable_original.map(norm_txt)
emim_largo["codigo"] = emim_largo.unidad_scian.str.extract(r"^(\S+)")[0]
emim_largo["nombre"] = emim_largo.unidad_scian.str.replace(r"^\S+\s*", "", regex=True).str.strip()
cod = emim_largo.codigo
emim_largo["nivel"] = np.select([cod == "31-33", cod.str.match(r"^\d{3}$"), cod.str.match(r"^\d{4}$"), cod.str.match(r"^\d{6}$")],
                                ["sector", "subsector", "rama", "clase"], None)
vt = emim_largo.valor_txt.str.strip()
emim_largo["estatus_celda"] = np.select(
    [vt.isna(), vt == "-", vt == "ND", vt == "NC", pd.to_numeric(vt, errors="coerce").notna()],
    ["vacio", "no_publicado (-)", "confidencial (ND)", "no_calculable (NC)", "dato"], "no_convertible")
emim_largo = emim_largo.merge(dicc_emim[["clave", "variable", "unidad"]], on="clave", how="left", validate="many_to_one")
emim_largo["valor"] = convertir_num(emim_largo.valor_txt, "MX_PROD_01", "valores (todas las variables y niveles)", codigos=("-", "ND", "NC"))
desconocidas = emim_largo.loc[emim_largo.variable.isna(), "variable_original"].unique()
exigir(len(desconocidas) == 0, "EMIM: variables que no están en el diccionario: ", " | ".join(desconocidas))
exigir(emim_largo.nivel.notna().all(), "EMIM: hay códigos SCIAN no reconocidos.")
validar_llave(emim_largo, ["variable", "codigo", "mes"], "MX_PROD_01 (largo, todos los niveles)", "CP3")
registrar("CP3", "MX_PROD_01", "Celdas no convertibles a número", (emim_largo.estatus_celda == "no_convertible").sum() == 0,
          f"{(emim_largo.estatus_celda == 'no_convertible').sum()} celdas")

por_mes = emim_largo.groupby("mes").estatus_celda.agg(todo=lambda s: (s == "no_publicado (-)").all(),
                                                     alguno=lambda s: (s == "no_publicado (-)").any())
meses_no_pub = list(por_mes.index[por_mes.todo])
meses_parcial = list(por_mes.index[por_mes.alguno & ~por_mes.todo])
registrar("CP3", "MX_PROD_01", "Los '-' (no publicado) ocupan meses completos, no celdas sueltas", len(meses_parcial) == 0,
          ("Meses con '-' parcial: " + ", ".join(map(fmt_mes, meses_parcial))) if meses_parcial
          else "Meses completos sin publicar: " + ", ".join(map(fmt_mes, meses_no_pub)), si_falla="REVISAR")
if decisiones["emim_excluir_meses_no_publicados"] and meses_no_pub:
    excluir("MX_PROD_01", emim_largo.mes.isin(meses_no_pub).sum(),
            f"Celdas '-' de meses aún no publicados ({', '.join(map(fmt_mes, meses_no_pub))}), decisión D8")
    emim_largo = emim_largo[~emim_largo.mes.isin(meses_no_pub)].copy()

def mes_de_nota(patron):
    encontrados = set()
    for t in notas_emim:
        if re.search(patron, t, re.I):
            for m in re.finditer(r"a partir de\s+([A-Za-zÁÉÍÓÚáéíóú]{3})[A-Za-záéíóú]*\.?\s+(\d{4})", t, re.I):
                encontrados.add(pd.Timestamp(int(m.group(2)), MESES_ABR[norm_txt(m.group(1))], 1))
    return encontrados.pop() if len(encontrados) == 1 else None

f_revisadas = mes_de_nota(r"Cifras revisadas a partir de")
f_preliminares = mes_de_nota(r"Cifras preliminares a partir de")
registrar("CP3", "MX_PROD_01", "Se identificó en las notas desde cuándo hay cifras revisadas y preliminares",
          f_revisadas is not None and f_preliminares is not None,
          f"Revisadas desde {fmt_mes(f_revisadas)}; preliminares desde {fmt_mes(f_preliminares)}", si_falla="REVISAR")

emim_rama_largo = emim_largo[emim_largo.nivel == "rama"]
nombres_rama = emim_rama_largo[["codigo", "nombre"]].drop_duplicates().rename(columns={"codigo": "rama", "nombre": "rama_nombre"})
registrar("CP3", "MX_PROD_01", "Cada rama tiene un solo nombre en las cuatro variables", not nombres_rama.rama.duplicated().any(),
          f"{nombres_rama.rama.nunique()} ramas, {len(nombres_rama)} pares rama-nombre")
emim_rama = (emim_rama_largo.pivot(index=["mes", "codigo"], columns="variable", values="valor")
             .reset_index().rename(columns={"codigo": "rama"}).rename_axis(columns=None))
emim_rama = emim_rama[["mes", "rama", *dicc_emim.variable]]
emim_nd = (emim_rama_largo[emim_rama_largo.estatus_celda == "confidencial (ND)"]
           .groupby(["mes", "codigo"]).variable.apply(lambda s: "; ".join(sorted(s))).reset_index()
           .rename(columns={"codigo": "rama", "variable": "emim_celdas_nd"}))
emim_rama = emim_rama.merge(emim_nd, on=["mes", "rama"], how="left", validate="one_to_one")
emim_rama["emim_estatus_cifra"] = (None if f_revisadas is None or f_preliminares is None else
                                   np.select([emim_rama.mes < f_revisadas, emim_rama.mes < f_preliminares],
                                             ["definitiva", "revisada"], "preliminar"))
emim_rama = emim_rama.sort_values(["rama", "mes"]).reset_index(drop=True)

validar_fechas_mensuales(emim_rama.mes, "MX_PROD_01 (ramas)")
validar_llave(emim_rama, ["mes", "rama"], "MX_PROD_01 (ramas)", "CP3")
registrar("CP3", "MX_PROD_01", "Número de ramas (86 según el inventario)", emim_rama.rama.nunique() == 86,
          f"{emim_rama.rama.nunique()} ramas", si_falla="REVISAR")
registrar("CP3", "MX_PROD_01", "Panel de ramas completo (todas las ramas en todos los meses)",
          len(emim_rama) == emim_rama.rama.nunique() * emim_rama.mes.nunique(),
          f"{len(emim_rama)} filas = {emim_rama.rama.nunique()} ramas × {emim_rama.mes.nunique()} meses", si_falla="REVISAR")
resumir_limpieza("MX_PROD_01", "emim_rama", len(e), emim_rama, ["mes", "rama"], "Variable, Tipo de dato, Sector SCIAN + 1 columna por mes",
                 "Ancho (1 fila por variable × unidad SCIAN) a largo; se conservan las 86 ramas × 4 variables (sector, subsectores y clases quedan en emim_largo para validar).")
tabla(emim_largo.groupby(["nivel", "variable", "estatus_celda"]).size().reset_index(name="celdas"), "EMIM: estatus de las celdas por nivel y variable")
tabla(emim_nd.groupby(["rama", "emim_celdas_nd"]).agg(meses=("mes", "size"), desde=("mes", "min"), hasta=("mes", "max")).reset_index(),
      "EMIM: ramas con valores confidenciales (ND), que quedan como NA")
tabla(emim_rama.emim_estatus_cifra.value_counts().rename_axis("emim_estatus_cifra").reset_index(name="filas mes-rama"), "EMIM: estatus de las cifras")

**EMIM: estatus de las celdas por nivel y variable**

nivel,variable,estatus_celda,celdas
clase,emim_horas_trabajadas,dato,"21,218"
clase,emim_personal_ocupado,dato,"21,218"
clase,emim_valor_produccion,confidencial (ND),69
clase,emim_valor_produccion,dato,"21,149"
clase,emim_valor_ventas,confidencial (ND),67
clase,emim_valor_ventas,dato,"21,151"
rama,emim_horas_trabajadas,dato,8858
rama,emim_personal_ocupado,dato,8858
rama,emim_valor_produccion,confidencial (ND),24
rama,emim_valor_produccion,dato,8834


**EMIM: ramas con valores confidenciales (ND), que quedan como NA**

rama,emim_celdas_nd,meses,desde,hasta
3343,emim_valor_produccion; emim_valor_ventas,24,2021-08-01,2024-04-01


**EMIM: estatus de las cifras**

emim_estatus_cifra,filas mes-rama
definitiva,7224
preliminar,1032
revisada,602


### MX_EXP_01 · Exportaciones
Conversión de tipos, `SCIAN` → `clase_scian`, `Clase` → `rama`; verificación de que cada clase pertenece a su rama y **agregación de clase a rama por suma** (el total de un flujo es la suma de sus partes; la propia fuente asigna cada clase a su rama). Si alguna clase tuviera un faltante, la suma de la rama queda en NA.

In [15]:
x = con_encabezado(crudos["expo_datos"], 0)
exigir([norm_txt(c) for c in x.columns] == [norm_txt(c) for c in esperadas], "Exportaciones: columnas inesperadas: ", ", ".join(x.columns))
x.columns = ["tipo_operacion", "anio", "mes_num", "clase_scian", "rama", "valor_pesos", "valor_usd"]
expo = pd.DataFrame({
    "tipo_operacion": convertir_num(x.tipo_operacion, "MX_EXP_01", "Tipo de operación"),
    "anio": convertir_num(x.anio, "MX_EXP_01", "Año"),
    "mes_num": convertir_num(x.mes_num, "MX_EXP_01", "Mes"),
    "clase_scian": x.clase_scian.str.strip(), "rama": x.rama.str.strip(),
    "valor_pesos": convertir_num(x.valor_pesos, "MX_EXP_01", "Valor en pesos"),
    "valor_usd": convertir_num(x.valor_usd, "MX_EXP_01", "Valor en dólares")})
expo["mes"] = a_mes(pd.to_datetime(dict(year=expo.anio, month=expo.mes_num, day=1), errors="coerce"))
registrar("CP3", "MX_EXP_01", "Tipo de operación = 2 (exportaciones) en todas las filas", (expo.tipo_operacion == 2).all(),
          "Valores: " + ", ".join(map(str, expo.tipo_operacion.unique())))
registrar("CP3", "MX_EXP_01", "Clase SCIAN de 6 dígitos y rama de 4 dígitos",
          expo.clase_scian.str.match(r"^\d{6}$").all() and expo.rama.str.match(r"^\d{4}$").all(), "")
registrar("CP3", "MX_EXP_01", "La clase pertenece a la rama declarada (primeros 4 dígitos)",
          (expo.clase_scian.str[:4] == expo.rama).all(), f"{(expo.clase_scian.str[:4] != expo.rama).sum()} filas fuera de su rama")
registrar("CP3", "MX_EXP_01", "Valores sin faltantes", expo[["valor_pesos", "valor_usd"]].notna().all().all(),
          f"{expo.valor_pesos.isna().sum()} NA en pesos, {expo.valor_usd.isna().sum()} NA en dólares", si_falla="REVISAR")
registrar("CP3", "MX_EXP_01", "Valores no negativos", (expo[["valor_pesos", "valor_usd"]] >= 0).all().all(), "", si_falla="REVISAR")
validar_fechas_mensuales(expo.mes, "MX_EXP_01 (clases)")
validar_llave(expo, ["mes", "clase_scian"], "MX_EXP_01 (clases)", "CP3")
expo_rama = (expo.groupby(["mes", "rama"])
             .agg(expo_valor_pesos=("valor_pesos", lambda s: s.sum(skipna=False)),
                  expo_valor_usd=("valor_usd", lambda s: s.sum(skipna=False)),
                  expo_n_clases=("clase_scian", "size")).reset_index().sort_values(["rama", "mes"]).reset_index(drop=True))
validar_llave(expo_rama, ["mes", "rama"], "MX_EXP_01 (ramas)", "CP3")
registrar("CP3", "MX_EXP_01", "La agregación conserva el total (suma de clases = suma de ramas)",
          np.isclose(expo.valor_pesos.sum(), expo_rama.expo_valor_pesos.sum()) and np.isclose(expo.valor_usd.sum(), expo_rama.expo_valor_usd.sum()),
          f"Total pesos: {expo.valor_pesos.sum():,.0f}")
registrar("CP3", "MX_EXP_01", "Panel de ramas completo", len(expo_rama) == expo_rama.rama.nunique() * expo_rama.mes.nunique(),
          f"{len(expo_rama)} filas = {expo_rama.rama.nunique()} ramas × {expo_rama.mes.nunique()} meses", si_falla="REVISAR")
resumir_limpieza("MX_EXP_01", "expo (clases)", len(x), expo, ["mes", "clase_scian"], ", ".join(esperadas), "Sin eliminación de filas; conversión de tipos y fecha.")
resumir_limpieza("MX_EXP_01", "expo_rama", len(expo), expo_rama, ["mes", "rama"], "(derivado de expo)", "Suma de clases SCIAN de 6 dígitos a rama de 4 dígitos.")
cl = expo_rama.groupby("rama").expo_n_clases.agg(["min", "max"]).reset_index()
tabla(cl[cl["min"] != cl["max"]].rename(columns={"min": "clases_min", "max": "clases_max"}),
      "Ramas cuyo número de clases con registro cambia entre meses (composición variable)")

**Ramas cuyo número de clases con registro cambia entre meses (composición variable)**

rama,clases_min,clases_max
3119,9,10
3122,2,3
3152,5,6
3211,1,2
3221,3,4
3259,4,5
3315,1,2
3399,13,14


### MX_ARAN_01 · Importaciones de EE.UU. desde México y aranceles (Census)
Conversión de tipos, verificación de la llave *mes × NAICS*, **recálculo de las tasas** desde sus componentes y separación de los códigos no manufactureros (fuera del alcance: solo SCIAN 31-33). No se agrega ni promedia nada.

In [16]:
c = crudos["census"]
census = pd.DataFrame({"mes": a_fecha(c.fecha), "naics": c.naics.str.strip(), "descripcion": c.descripcion,
                       **{v: convertir_num(c[v], "MX_ARAN_01", v) for v in
                          ["valor_importado", "valor_gravable", "aranceles", "tasa_efectiva", "tasa_gravable"]}})
validar_fechas_mensuales(census.mes, "MX_ARAN_01", texto_original=c.fecha)
validar_llave(census, ["mes", "naics"], "MX_ARAN_01", "CP3")
registrar("CP3", "MX_ARAN_01", "Cada código NAICS tiene una sola descripción", (census.groupby("naics").descripcion.nunique() == 1).all(), "")
tasa_e = (census.aranceles / census.valor_importado).where(census.valor_importado > 0)
tasa_g = (census.aranceles / census.valor_gravable).where(census.valor_gravable > 0)
igual_t = lambda a, b: (a.isna() & b.isna()) | ((a - b).abs() <= 1e-9 * np.maximum(1, b.abs()))
registrar("CP3", "MX_ARAN_01", "tasa_efectiva = aranceles / valor_importado (NA si no hubo importaciones)",
          igual_t(tasa_e, census.tasa_efectiva).all(), f"{(~igual_t(tasa_e, census.tasa_efectiva)).sum()} filas no coinciden")
registrar("CP3", "MX_ARAN_01", "tasa_gravable = aranceles / valor_gravable (NA si valor gravable = 0)",
          igual_t(tasa_g, census.tasa_gravable).all(), f"{(~igual_t(tasa_g, census.tasa_gravable)).sum()} filas no coinciden")
supera = census[(census.valor_gravable > 0) & (census.aranceles > census.valor_gravable)]
revisar("CP3", "MX_ARAN_01", "Aranceles mayores que el valor gravable (compatible con aranceles específicos)",
        f"{len(supera)} filas en los códigos {', '.join(sorted(supera.naics.unique()))}. No se modifican (decisión 7 de las notas).")
es_manuf = census.naics.str.match(r"^3\d{3}$")
census_manuf = census[es_manuf].rename(columns={"naics": "rama"}).reset_index(drop=True)
census_no_manuf = census[~es_manuf].reset_index(drop=True)
excluir("MX_ARAN_01", len(census_no_manuf), f"Códigos no manufactureros ({census_no_manuf.naics.nunique()} códigos 11xx, 21xx, 9xxx): solo SCIAN 31-33 (decisión 3 de las notas).")
validar_fechas_mensuales(census_manuf.mes, "MX_ARAN_01 (manufactura)")
registrar("CP3", "MX_ARAN_01", "Códigos manufactureros con todos los meses",
          len(census_manuf) == census_manuf.rama.nunique() * census_manuf.mes.nunique(),
          f"{len(census_manuf)} filas; {census_manuf.rama.nunique()} códigos × {census_manuf.mes.nunique()} meses", si_falla="REVISAR")
resumir_limpieza("MX_ARAN_01", "census_manuf", len(c), census_manuf, ["mes", "rama"], ", ".join(c.columns),
                 "Se separan los códigos no manufactureros (se conservan en census_no_manuf; no se integran).")
tabla(census_no_manuf.groupby(["naics", "descripcion"]).mes.agg(meses="size", desde="min", hasta="max").reset_index(),
      "Códigos del Census que NO se integran (no manufactureros)")
tabla(census_manuf[census_manuf.tasa_efectiva.isna()].groupby(["rama", "descripcion"]).size().reset_index(name="meses_sin_importaciones"),
      "Códigos manufactureros con meses sin importaciones (tasa efectiva NA)")

**Códigos del Census que NO se integran (no manufactureros)**

naics,descripcion,meses,desde,hasta
1111,OILSEEDS & GRAINS,163,2013-01-01,2026-07-01
1112,VEGETABLES & MELONS,163,2013-01-01,2026-07-01
1113,FRUITS & TREE NUTS,163,2013-01-01,2026-07-01
1114,"MUSHROOMS, NURSERY & RELATED PRODUCTS",163,2013-01-01,2026-07-01
1119,OTHER AGRICULTURAL PRODUCTS,163,2013-01-01,2026-07-01
1121,CATTLE,155,2013-01-01,2025-12-01
1123,POULTRY & EGGS,39,2015-07-01,2025-12-01
1124,"SHEEP, GOATS & FINE ANIMAL HAIR",39,2013-11-01,2025-12-01
1125,FARMED FISH AND RELATED PRODUCTS,163,2013-01-01,2026-07-01
1129,OTHER ANIMALS,163,2013-01-01,2026-07-01


**Códigos manufactureros con meses sin importaciones (tasa efectiva NA)**

_(sin filas)_

### MX_ARAN_02 · Tasa efectiva en formato ancho (solo verificación)

In [17]:
a2 = crudos["aran2"]
exigir(a2.columns[0] == "fecha", "MX_ARAN_02: la primera columna no es 'fecha'.")
a2_largo = a2.melt(id_vars="fecha", var_name="naics", value_name="tasa_txt")
a2_largo["mes"] = a_fecha(a2_largo.fecha)
a2_largo["tasa_02"] = convertir_num(a2_largo.tasa_txt, "MX_ARAN_02", "tasa efectiva (todas las columnas)")
cmp_a2 = a2_largo[["mes", "naics", "tasa_02"]].merge(census[["mes", "naics", "tasa_efectiva"]], on=["mes", "naics"], how="outer", validate="one_to_one")
coincide = (cmp_a2.tasa_02.isna() & cmp_a2.tasa_efectiva.isna()) | ((cmp_a2.tasa_02 - cmp_a2.tasa_efectiva).abs() <= 6e-7)
registrar("CP3", "MX_ARAN_02", "Es la misma tasa efectiva que MX_ARAN_01 (6 decimales)", coincide.all(),
          f"{(~coincide).sum()} de {len(coincide)} celdas no coinciden. No se integra (duplicado).")
excluir("MX_ARAN_02", len(a2), "Derivado de MX_ARAN_01 (misma información en formato ancho): solo se usa para verificar.")

### US_PROD_01 y US_PROD_02 · Producción industrial de EE.UU. (G.17)

In [18]:
exigir(crudos["ip_sa"].columns[0] == "fecha", "G.17: la primera columna no es 'fecha'.")
exigir(list(crudos["ip_cat"].columns) == ["serie", "descripcion", "naics"], "G.17: columnas inesperadas en el catálogo.")
exigir(set(crudos["ip_sa"].columns[1:]) == set(crudos["ip_cat"].serie) and not crudos["ip_cat"].serie.duplicated().any(),
       "G.17: las series de datos y del catálogo no coinciden.")
ip_cat = crudos["ip_cat"].assign(tipo_codigo=lambda d: tipo_naics(d.naics), manufactura=lambda d: d.naics.str.match(r"^3[1-3]"))
validar_fechas_mensuales(a_fecha(crudos["ip_sa"].fecha), "US_PROD_01", texto_original=crudos["ip_sa"].fecha)
ip_largo = crudos["ip_sa"].melt(id_vars="fecha", var_name="serie", value_name="indice_txt")
ip_largo["mes"] = a_fecha(ip_largo.fecha)
ip_largo["indice"] = convertir_num(ip_largo.indice_txt, "US_PROD_01", "índices (todas las series)")
ip_largo = ip_largo[["mes", "serie", "indice"]].merge(ip_cat, on="serie", how="left", validate="many_to_one")
validar_llave(ip_largo, ["mes", "serie"], "US_PROD_01", "CP3")
cob_ip = ip_largo.dropna(subset=["indice"]).groupby("serie").mes.agg(primer_mes="min", ultimo_mes="max", meses_con_dato="size").reset_index()
cob_ip["huecos_internos"] = [len(pd.date_range(a, b, freq="MS")) - n for a, b, n in zip(cob_ip.primer_mes, cob_ip.ultimo_mes, cob_ip.meses_con_dato)]
registrar("CP3", "US_PROD_01", "Sin huecos internos en las series", (cob_ip.huecos_internos == 0).all(),
          f"{(cob_ip.huecos_internos > 0).sum()} series con huecos", si_falla="REVISAR")
ultimo_general = cob_ip.ultimo_mes.max()
cortas = cob_ip[cob_ip.ultimo_mes < ultimo_general]
revisar("CP3", "US_PROD_01", "Series que terminan antes del último mes del archivo",
        f"{len(cortas)} series terminan antes de {fmt_mes(ultimo_general)}: {', '.join(cortas.serie)}. Se dejan como NA, sin extrapolar.")
excluir("US_PROD_01", ip_largo.indice.isna().sum(), "Celdas vacías antes del inicio o después del fin de cada serie (no son datos; quedan como NA)")
resumir_limpieza("US_PROD_01", "ip_largo", len(crudos["ip_sa"]), ip_largo, ["mes", "serie"], "fecha + 1 columna por serie G.17", "Ancho (mes × serie) a largo.")
t = cob_ip.merge(ip_cat, on="serie")
tabla(t[t.manufactura].assign(inicio=lambda d: d.primer_mes.map(fmt_mes), fin=lambda d: d.ultimo_mes.map(fmt_mes))
      .groupby(["tipo_codigo", "inicio", "fin"]).size().reset_index(name="series"), "Cobertura de las series manufactureras del G.17")

**Cobertura de las series manufactureras del G.17**

tipo_codigo,inicio,fin,series
NAICS 3 dígitos,1972-01,2026-08,21
NAICS 4 dígitos,1972-01,2026-05,1
NAICS 4 dígitos,1972-01,2026-08,66
NAICS 5-6 dígitos,1972-01,2026-05,2
NAICS 5-6 dígitos,1972-01,2026-06,4
NAICS 5-6 dígitos,1972-01,2026-07,8
NAICS 5-6 dígitos,1972-01,2026-08,36
parte de industria (pt.),1972-01,2026-05,6
parte de industria (pt.),1972-01,2026-06,7
parte de industria (pt.),1972-01,2026-07,5


### PRECIOS_01 · INPP
Metadatos de cada serie (título, periodo, unidad, clave), subsector SCIAN desde el título y formato largo; `N/E` = faltante (la serie no existe en ese mes).

In [19]:
p = crudos["inpp"]
pos = lambda etiqueta: np.flatnonzero((p.c1.map(norm_txt) == etiqueta).to_numpy())
f_fecha, f_titulo, f_per, f_uni = pos("fecha"), pos("titulo"), pos("periodo disponible"), pos("unidad")
exigir(all(len(v) == 1 for v in (f_fecha, f_titulo, f_per, f_uni)), "INPP: no se encontraron las filas de metadatos.")
f_fecha, f_titulo, f_per, f_uni = f_fecha[0], f_titulo[0], f_per[0], f_uni[0]
inpp_cat = pd.DataFrame({"id_serie": [str(int(float(v))) for v in p.iloc[f_fecha, 1:]],
                         "titulo": p.iloc[f_titulo, 1:].values, "periodo_declarado": p.iloc[f_per, 1:].values,
                         "unidad_declarada": p.iloc[f_uni, 1:].values})
inpp_cat["subsector_titulo"] = inpp_cat.titulo.str.extract(r"31-33 Industrias manufactureras, (\d{3}) ")[0]
inpp_cat["descripcion_corta"] = inpp_cat.titulo.str.replace(r"^.*Por origen, ", "", regex=True)
exigir(not inpp_cat.id_serie.duplicated().any(), "INPP: claves de serie repetidas.")
datos_inpp = p.iloc[f_fecha + 1:].copy()
vacias = datos_inpp.c1.isna()
excluir("PRECIOS_01", f_fecha + 1, "Filas de metadatos del encabezado")
excluir("PRECIOS_01", vacias.sum(), "Filas vacías al final de la hoja")
datos_inpp = datos_inpp[~vacias]
datos_inpp.columns = ["fecha_txt", *inpp_cat.id_serie]
validar_fechas_mensuales(a_fecha(datos_inpp.fecha_txt), "PRECIOS_01", texto_original=datos_inpp.fecha_txt)
inpp_largo = datos_inpp.melt(id_vars="fecha_txt", var_name="id_serie", value_name="valor_txt")
inpp_largo["mes"] = a_fecha(inpp_largo.fecha_txt)
inpp_largo["valor"] = convertir_num(inpp_largo.valor_txt, "PRECIOS_01", "índices (todas las series)", codigos=("N/E",))
inpp_largo = inpp_largo[["mes", "id_serie", "valor"]]
validar_llave(inpp_largo, ["mes", "id_serie"], "PRECIOS_01", "CP3")
jul19 = inpp_largo[(inpp_largo.mes == "2019-07-01") & inpp_largo.valor.notna()]
registrar("CP3", "PRECIOS_01", "Base efectiva: todas las series valen 100 en julio de 2019",
          len(jul19) > 0 and ((jul19.valor - 100).abs() < 1e-6).all(),
          f"La unidad declarada es '{inpp_cat.unidad_declarada.iloc[0]}', pero los datos valen 100 en 2019-07: base efectiva jul-2019=100 (por confirmar con INEGI).",
          si_falla="REVISAR")
revisar("CP3", "PRECIOS_01", "Base declarada vs. base efectiva", "La unidad declarada (julio 2025=100) no coincide con los datos (100 en julio 2019).")
resumir_limpieza("PRECIOS_01", "inpp_largo", len(p), inpp_largo, ["mes", "id_serie"], "Fecha + 43 columnas por clave", "Se retiran metadatos y filas vacías; ancho a largo.")
cob_inpp = inpp_largo.dropna(subset=["valor"]).groupby("id_serie").mes.agg(lambda s: rango_txt(s)).rename("cobertura").reset_index()
tabla(inpp_cat[inpp_cat.subsector_titulo.notna()].merge(cob_inpp, on="id_serie")[["id_serie", "subsector_titulo", "periodo_declarado", "cobertura", "descripcion_corta"]],
      "Series INPP de los 21 subsectores manufactureros", ancho=140)

**Series INPP de los 21 subsectores manufactureros**

id_serie,subsector_titulo,periodo_declarado,cobertura,descripcion_corta
1370012,311,Jun 2010-Ago 2026,2010-06 a 2026-08,"Actividades secundarias con petróleo, 31-33 Industrias manufactureras, 311 Industria alimentaria"
1370013,312,Jun 2010-Ago 2026,2010-06 a 2026-08,"Actividades secundarias con petróleo, 31-33 Industrias manufactureras, 312 Industria de las bebidas y del tabaco"
1370014,313,Jun 2010-Ago 2026,2010-06 a 2026-08,"Actividades secundarias con petróleo, 31-33 Industrias manufactureras, 313 Fabricación de insumos textiles y acabado de textiles"
1370015,314,Jun 2010-Ago 2026,2010-06 a 2026-08,"Actividades secundarias con petróleo, 31-33 Industrias manufactureras, 314 Fabricación de productos textiles, excepto prendas de vestir"
1370016,315,Jun 2010-Ago 2026,2010-06 a 2026-08,"Actividades secundarias con petróleo, 31-33 Industrias manufactureras, 315 Fabricación de prendas de vestir"
1370017,316,Jun 2010-Ago 2026,2010-06 a 2026-08,"Actividades secundarias con petróleo, 31-33 Industrias manufactureras, 316 Curtido y acabado de cuero y piel, y fabricación de productos ..."
1770001,321,Ene 1981-Ago 2026,1981-01 a 2026-08,"Actividades secundarias con petróleo, 31-33 Industrias manufactureras, 321 Industria de la madera"
1370019,322,Jun 2010-Ago 2026,2010-06 a 2026-08,"Actividades secundarias con petróleo, 31-33 Industrias manufactureras, 322 Industria del papel"
1370020,323,Jun 2010-Ago 2026,2010-06 a 2026-08,"Actividades secundarias con petróleo, 31-33 Industrias manufactureras, 323 Impresión e industrias conexas"
1370021,324,Jun 2010-Ago 2026,2010-06 a 2026-08,"Actividades secundarias con petróleo, 31-33 Industrias manufactureras, 324 Fabricación de productos derivados del petróleo y del carbón"


### INC_01 · TPU mensual (las versiones trimestral y diaria solo se documentan)

In [20]:
tm = con_encabezado(crudos["tpu_m"])
exigir({"DATE", "N7", "TPU_RAW", "TPU_SHARE", "TPU"} <= set(tm.columns), "TPU_MONTHLY: faltan columnas esperadas.")
extra = [c for c in tm.columns if c not in ("DATE", "N7", "TPU_RAW", "TPU_SHARE", "TPU")]
exigir(not extra or tm[extra].isna().all().all(), "TPU_MONTHLY: columnas adicionales con contenido: ", extra)
tm = tm[tm.DATE.notna()]
tpu = pd.DataFrame({"mes": a_fecha(tm.DATE), "tpu": convertir_num(tm.TPU, "INC_01", "TPU"),
                    "tpu_share": convertir_num(tm.TPU_SHARE, "INC_01", "TPU_SHARE"),
                    "tpu_raw": convertir_num(tm.TPU_RAW, "INC_01", "TPU_RAW"), "tpu_n7": convertir_num(tm.N7, "INC_01", "N7")}).reset_index(drop=True)
validar_fechas_mensuales(tpu.mes, "INC_01 (mensual)", texto_original=tm.DATE)
validar_llave(tpu, ["mes"], "INC_01 (mensual)", "CP3")
registrar("CP3", "INC_01", "TPU = TPU_SHARE × 10,000 (TPU_SHARE es una proporción, aunque el ReadMe diga 'percent share')",
          np.allclose(tpu.tpu, tpu.tpu_share * 1e4, rtol=1e-8), "Confirma la unidad de TPU (100 = 1% de artículos).", si_falla="REVISAR")
registrar("CP3", "INC_01", "TPU_SHARE = TPU_RAW / N7", np.allclose(tpu.tpu_share, tpu.tpu_raw / tpu.tpu_n7, rtol=1e-8), "", si_falla="REVISAR")
revisar("CP3", "INC_01", "ReadMe desactualizado", f"El ReadMe dice 'monthly from 1985-2020'; los datos cubren {rango_txt(tpu.mes)}.")
resumir_limpieza("INC_01", "tpu", len(tm), tpu, ["mes"], ", ".join(con_encabezado(crudos["tpu_m"]).columns), "Solo conversión de tipos.")
tq, tdd = con_encabezado(crudos["tpu_q"]), con_encabezado(crudos["tpu_d"])
tpu_trim = pd.DataFrame({"trimestre": tq.DATEQ, "tpuq": convertir_num(tq.TPUQ, "INC_01", "TPUQ"),
                         "tariffvol": convertir_num(tq.TARIFFVOL, "INC_01", "TARIFFVOL")}).dropna(subset=["trimestre"])
tpu_diario = pd.DataFrame({"fecha": a_fecha(tdd.DATE), "tpud_index": convertir_num(tdd.TPUD_index, "INC_01", "TPUD_index")}).dropna(subset=["fecha"])
revisar("CP3", "INC_01", "Versiones trimestral y diaria no integradas",
        f"Trimestral: {tpu_trim.trimestre.iloc[0]} a {tpu_trim.trimestre.iloc[-1]} (TARIFFVOL hasta {tpu_trim.dropna(subset=['tariffvol']).trimestre.iloc[-1]}). "
        f"Diaria: {tpu_diario.fecha.min():%Y-%m-%d} a {tpu_diario.fecha.max():%Y-%m-%d}. Frecuencia distinta a la mensual; no se convierten.")
excluir("INC_01", len(tpu_trim) + len(tpu_diario), "Hojas TPU_QUARTERLY y TPU_DAILY: frecuencia no mensual (se documentan).")

### INC_02 · WUI, WTUI y WPUI

In [21]:
indices_wui = {"T1": "WUI", "T6": "WTUI", "T7": "WPUI"}
partes = []
for hoja, indice in indices_wui.items():
    xw = con_encabezado(crudos["wui"][hoja])
    exigir(xw.columns[0] == "date", f"WUI hoja {hoja}: la primera columna no es 'date'.")
    exigir(set(decisiones["wui_paises"]) <= set(xw.columns), f"WUI hoja {hoja}: faltan países seleccionados.")
    xw = xw[xw.date.notna()]
    validar_fechas_mensuales(a_fecha(xw.date), f"INC_02 ({hoja})", texto_original=xw.date)
    l = xw.melt(id_vars="date", var_name="pais", value_name="valor_txt")
    l["mes"] = a_fecha(l.date)
    l["valor"] = convertir_num(l.valor_txt, "INC_02", f"{hoja} ({indice}, todos los países)")
    partes.append(l.assign(indice=indice, hoja=hoja)[["indice", "hoja", "pais", "mes", "valor"]])
wui_largo = pd.concat(partes, ignore_index=True)
validar_llave(wui_largo, ["indice", "pais", "mes"], "INC_02 (T1, T6, T7)", "CP3")

def leer_wui_f(hoja, indice):
    xf = leer_texto(ruta("INC_02"), hoja, usecols="A:B")
    f = np.flatnonzero((xf.c1.map(norm_txt) == "date").to_numpy())
    exigir(len(f) == 1, f"WUI hoja {hoja}: no se encontró el encabezado 'date'.")
    d = xf.iloc[f[0] + 1:].dropna(subset=["c1"])
    out = pd.DataFrame({"mes": a_fecha(d.c1), "valor": convertir_num(d.c2, "INC_02", f"{hoja} ({indice})"),
                        "indice": indice, "hoja": hoja, "encabezado": xf.c2.iloc[f[0]]})
    validar_fechas_mensuales(out.mes, f"INC_02 ({hoja})")
    return out
wui_mundo = pd.concat([leer_wui_f("F1", "WUI"), leer_wui_f("F2", "WTUI"), leer_wui_f("F3", "WPUI")], ignore_index=True)
validar_llave(wui_mundo, ["indice", "mes"], "INC_02 (F1-F3)", "CP3")
ceros = (wui_largo[wui_largo.pais.isin(decisiones["wui_paises"])].groupby(["indice", "pais"]).valor
         .agg(meses="size", meses_en_cero=lambda s: int((s == 0).sum()), faltantes=lambda s: int(s.isna().sum())).reset_index())
ceros["pct_cero"] = (100 * ceros.meses_en_cero / ceros.meses).round(1)
for r in ceros.itertuples():
    if r.pct_cero > 25:
        revisar("CP3", "INC_02", f"{r.indice} {r.pais}: proporción de meses en cero",
                f"{r.meses_en_cero} de {r.meses} meses valen 0 ({r.pct_cero}%). Son ceros de la fuente (conteos bajos de palabras), no faltantes; no se modifican.")
revisar("CP3", "INC_02", "Arrastre de valores hecho por la fuente",
        "Nota 2 de T0: 'For countries with missing reports, the previous month's value is used'. No es una imputación de este código.")
revisar("CP3", "INC_02", "Escala de los promedios mundiales (F1-F3)", "Ponderados por PIB, con escala distinta a la de los países; no se reescalan.")
resumir_limpieza("INC_02", "wui_largo", sum(len(crudos["wui"][h]) for h in indices_wui), wui_largo, ["indice", "pais", "mes"],
                 "date + 71 columnas de país (por hoja)", "Tres hojas anchas apiladas en largo (mes × país × índice).")
resumir_limpieza("INC_02", "wui_mundo", sum(len(crudos["wui"][h]) for h in ("F1", "F2", "F3")), wui_mundo, ["indice", "mes"],
                 "date + promedio ponderado (+ celdas de gráfica)", "Solo columnas A:B; se descartan las celdas de texto de las gráficas.")
tabla(ceros, "Meses en cero por índice y país seleccionado")

**Meses en cero por índice y país seleccionado**

indice,pais,meses,meses_en_cero,faltantes,pct_cero
WPUI,MEX,224,63,0,28.1
WPUI,USA,224,82,0,36.6
WTUI,MEX,224,146,0,65.2
WTUI,USA,224,165,0,73.7
WUI,MEX,224,13,0,5.8
WUI,USA,224,8,0,3.6


### INC_03 · EPU México · MACRO_01 · Tipo de cambio FIX · MACRO_02 · VIX · DOC_CORR

In [22]:
ep = crudos["epu"]
exigir([norm_txt(v) for v in ep.iloc[0, :3]] == ["year", "month", "mexican policy uncertainty index"], "EPU: encabezado inesperado.")
ep = con_encabezado(ep)
ep.columns = ["anio", "mes_num", "epu", *ep.columns[3:]]
es_nota = ~ep.anio.fillna("").str.match(r"^\d{4}$")
excluir("INC_03", es_nota.sum(), "Filas que no son datos (nota de la fuente): " + " | ".join(ep.anio[es_nota].astype(str).str[:60]))
ep = ep[~es_nota]
epu = pd.DataFrame({"mes": a_mes(pd.to_datetime(dict(year=ep.anio.astype(int), month=convertir_num(ep.mes_num, "INC_03", "Month").astype(int), day=1))),
                    "epu_mex": convertir_num(ep.epu, "INC_03", "Mexican Policy Uncertainty Index")}).reset_index(drop=True)
validar_fechas_mensuales(epu.mes, "INC_03")
validar_llave(epu, ["mes"], "INC_03", "CP3")
resumir_limpieza("INC_03", "epu", len(crudos["epu"]), epu, ["mes"], "Year, Month, Mexican Policy Uncertainty Index", "Se retiran el encabezado y la nota de la fuente.")

tcr = crudos["tc"]
f_h = np.flatnonzero(((tcr.c1 == "Fecha") & (tcr.c2 == "SF17908")).to_numpy())
exigir(len(f_h) == 1, "Tipo de cambio: no se encontró la fila 'Fecha' / 'SF17908'.")
tc_meta = tcr.iloc[: f_h[0]].dropna(subset=["c1", "c2"]).rename(columns={"c1": "campo", "c2": "valor"})
tc_datos = tcr.iloc[f_h[0] + 1:].dropna(subset=["c1"])
excluir("MACRO_01", f_h[0] + 1, "Filas de metadatos del encabezado de Banxico")
tc = pd.DataFrame({"mes": a_fecha(tc_datos.c1), "tc_fix": convertir_num(tc_datos.c2, "MACRO_01", "SF17908")}).reset_index(drop=True)
validar_fechas_mensuales(tc.mes, "MACRO_01", texto_original=tc_datos.c1)
validar_llave(tc, ["mes"], "MACRO_01", "CP3")
ceros_tc = tc.mes[tc.tc_fix == 0]
if decisiones["tc_ceros_como_faltantes"] and len(ceros_tc):
    tc.loc[tc.tc_fix == 0, "tc_fix"] = np.nan
    excluir("MACRO_01", len(ceros_tc), f"Ceros convertidos a NA (sin dato, decisión D5): {rango_txt(ceros_tc)}")
registrar("CP3", "MACRO_01", "Sin ceros ni valores no positivos tras la limpieza", (tc.tc_fix.dropna() > 0).all(), "", si_falla="REVISAR")
resumir_limpieza("MACRO_01", "tc", len(tcr), tc, ["mes"], "Fecha, SF17908", "Se retiran metadatos; ceros de 1991 a NA (D5).")
tabla(tc_meta[["campo", "valor"]], "Metadatos de la serie (Banxico)", ancho=200)

exigir(list(crudos["vix"].columns) == ["observation_date", "VIXCLS"], "VIX: columnas inesperadas.")
vix = pd.DataFrame({"fecha": a_fecha(crudos["vix"].observation_date),
                    "vix": convertir_num(crudos["vix"].VIXCLS, "MACRO_02", "VIXCLS", codigos=(".",))})
registrar("CP3", "MACRO_02", "Fechas diarias válidas y únicas", vix.fecha.notna().all() and not vix.fecha.duplicated().any(),
          f"{vix.fecha.min():%Y-%m-%d} a {vix.fecha.max():%Y-%m-%d}")
registrar("CP3", "MACRO_02", "Todo valor no vacío se convirtió a número (los vacíos son feriados)",
          (vix.vix.notna() | crudos["vix"].VIXCLS.isna() | (crudos["vix"].VIXCLS == ".")).all(),
          f"{vix.vix.isna().sum()} días hábiles sin dato; se conservan como NA")
vix_mensual = None
if decisiones["vix_regla_mensual"] is None:
    revisar("CP3", "MACRO_02", "VIX no integrado", "Sin regla de conversión (D6).")
else:
    v = vix.dropna(subset=["vix"]).sort_values("fecha").assign(mes=lambda d: a_mes(d.fecha.dt.to_period("M").dt.to_timestamp()))
    agg = "mean" if decisiones["vix_regla_mensual"] == "promedio_mensual" else "last"
    vix_mensual = v.groupby("mes").vix.agg([agg, "size"]).reset_index()
    vix_mensual.columns = ["mes", f"vix_{decisiones['vix_regla_mensual']}", "vix_dias_con_dato"]
    registrar("CP3", "MACRO_02", "VIX mensual: un valor por mes, calculado solo con días con dato",
              not vix_mensual.mes.duplicated().any() and (vix_mensual.vix_dias_con_dato > 0).all(),
              f"Regla: {decisiones['vix_regla_mensual']} (D6); {len(vix_mensual)} meses, {rango_txt(vix_mensual.mes)}; "
              f"días con dato por mes: {vix_mensual.vix_dias_con_dato.min()} a {vix_mensual.vix_dias_con_dato.max()}.")
    ultimo = vix.fecha.max()
    if ultimo < ultimo + pd.offsets.MonthEnd(0):
        revisar("CP3", "MACRO_02", "Último mes del VIX incompleto", f"El último dato es {ultimo:%Y-%m-%d}; el mes {fmt_mes(ultimo)} no está completo (queda fuera del panel).")
resumir_limpieza("MACRO_02", "vix (diario)", len(crudos["vix"]), vix, ["fecha"], "observation_date, VIXCLS", "Solo conversión de tipos.")

corr = crudos["corr"].copy()
exigir({"rama_scian", "nombre_rama", "subsector_scian", "id_serie_inpp"} <= set(corr.columns), "correspondencia_rama_inpp.csv: faltan columnas.")
for col in ("rama_scian", "subsector_scian", "id_serie_inpp"):
    corr[col] = corr[col].str.strip()
validar_llave(corr, ["rama_scian"], "DOC_CORR", "CP3")
registrar("CP3", "DOC_CORR", "El subsector de la tabla = primeros 3 dígitos de la rama", (corr.subsector_scian == corr.rama_scian.str[:3]).all(), "")
registrar("CP3", "DOC_CORR", "Todas las series asignadas existen en el INPP", corr.id_serie_inpp.isin(inpp_cat.id_serie).all(),
          "No encontradas: " + ", ".join(sorted(set(corr.id_serie_inpp) - set(inpp_cat.id_serie))))
chk = corr.merge(inpp_cat[["id_serie", "subsector_titulo"]], left_on="id_serie_inpp", right_on="id_serie", how="left")
registrar("CP3", "DOC_CORR", "La serie INPP asignada corresponde al subsector de la rama (según su título)",
          (chk.subsector_titulo == chk.subsector_scian).all(), f"{(chk.subsector_titulo != chk.subsector_scian).sum()} ramas con serie de otro subsector")

**Metadatos de la serie (Banxico)**

campo,valor
Título,"Tipo de cambio Pesos por dólar E.U.A., Para solventar obligaciones denominadas en moneda extranjera, Fecha de determinación (FIX) Cotizaciones promedio"
Periodo disponible,Ene 1991 - Ago 2026
Periodicidad,Mensual
Cifra,Tipo de Cambio
Unidad,Pesos por Dólar
Tipo de información,Niveles


'OK'

### MIP_01 · Coeficiente de exportación de la MIP 2018 (medida de exposición, decisión D9)
Un valor por rama, fijo en el tiempo (año 2018, anterior al panel de estimación). Se verifica que cubra exactamente las 86 ramas de la EMIM, con los mismos nombres, y que sea una proporción entre 0 y 1.

In [23]:
mx = crudos["mip"]
f_mip = np.flatnonzero((mx.c2.map(norm_txt) == "rama").to_numpy())
exigir(len(f_mip) == 1 and norm_txt(mx.c3.iloc[f_mip[0]]) == "export", "MIP: no se encontró el encabezado 'Rama' / 'Export'.")
dm = mx.iloc[f_mip[0] + 1:].dropna(subset=["c1"])
mip = pd.DataFrame({"rama": dm.c1.str.strip().str.replace(r"\.0+$", "", regex=True),
                    "rama_mip": dm.c2.str.strip(),
                    "mip_coef_exportacion_2018": convertir_num(dm.c3, "MIP_01", "Export")}).reset_index(drop=True)
mip["nombre_mip"] = mip.rama_mip.str.split("-", n=1).str[1].str.strip()
validar_llave(mip, ["rama"], "MIP_01", "CP3")
registrar("CP3", "MIP_01", "Cubre exactamente las ramas de la EMIM", set(mip.rama) == set(emim_rama.rama),
          f"{mip.rama.nunique()} ramas; solo en MIP: {sorted(set(mip.rama) - set(emim_rama.rama))}; solo en EMIM: {sorted(set(emim_rama.rama) - set(mip.rama))}")
nm = mip.merge(nombres_rama, on="rama", how="left")
registrar("CP3", "MIP_01", "Nombres de rama iguales a los de la EMIM", (nm.nombre_mip.map(norm_txt) == nm.rama_nombre.map(norm_txt)).all(),
          f"{(nm.nombre_mip.map(norm_txt) != nm.rama_nombre.map(norm_txt)).sum()} ramas con nombre distinto")
registrar("CP3", "MIP_01", "Coeficiente entre 0 y 1, sin faltantes", mip.mip_coef_exportacion_2018.between(0, 1).all(),
          f"Mínimo {mip.mip_coef_exportacion_2018.min():.4f}, mediana {mip.mip_coef_exportacion_2018.median():.4f}, máximo {mip.mip_coef_exportacion_2018.max():.4f}")
resumir_limpieza("MIP_01", "mip", len(mx), mip[["rama", "mip_coef_exportacion_2018"]], ["rama"], "(sin nombre), Rama, Export",
                 "Se retira la fila de encabezado; un valor por rama.")
tabla(mip.sort_values("mip_coef_exportacion_2018", ascending=False)[["rama", "nombre_mip", "mip_coef_exportacion_2018"]],
      "Coeficiente de exportación MIP 2018 por rama (de mayor a menor)", max_filas=100)

**Coeficiente de exportación MIP 2018 por rama (de mayor a menor)**

rama,nombre_mip,mip_coef_exportacion_2018
3364,Fabricación de equipo aeroespacial,0.98967
3343,Fabricación de equipo de audio y de video,0.947836
3345,"Fabricación de instrumentos de medición, control, navegación, y equipo médico electrónico",0.920317
3353,Fabricación de equipo de generación y distribución de energía eléctrica,0.898921
3342,Fabricación de equipo de comunicación,0.895055
3333,Fabricación de maquinaria y equipo para el comercio y los servicios,0.866485
3334,"Fabricación de equipo de aire acondicionado, calefacción, y de refrigeración industrial...",0.848202
3336,"Fabricación de motores de combustión interna, turbinas y transmisiones",0.838525
3351,Fabricación de accesorios de iluminación,0.831395
3325,Fabricación de herrajes y cerraduras,0.813433


### Checkpoint 3: resumen de la limpieza individual

In [24]:
titulo_checkpoint("CHECKPOINT 3 · Limpieza individual")
tabla(pd.DataFrame(REG["limpieza"]), "Filas y columnas antes y después de limpiar, por fuente", ancho=220)
tabla(pd.DataFrame(REG["exclusiones"]), "Filas o celdas descartadas y motivo (nada se descarta sin quedar aquí)", ancho=250)
conv = pd.DataFrame(REG["conversiones"])
tabla(conv, "Conversión de texto a número por fuente y variable", max_filas=200)
registrar("CP3", "Todas", "Celdas con contenido que no pudieron convertirse a número", conv.no_convertibles.sum() == 0,
          f"{conv.no_convertibles.sum()} celdas (ver tabla de conversión)")
tabla(pd.concat([
    faltantes_por_variable(emim_rama, list(dicc_emim.variable)).assign(fuente="MX_PROD_01"),
    faltantes_por_variable(expo_rama, ["expo_valor_pesos", "expo_valor_usd"]).assign(fuente="MX_EXP_01"),
    faltantes_por_variable(census_manuf, ["valor_importado", "valor_gravable", "aranceles", "tasa_efectiva", "tasa_gravable"]).assign(fuente="MX_ARAN_01"),
    faltantes_por_variable(tpu, ["tpu", "tpu_share", "tpu_raw", "tpu_n7"]).assign(fuente="INC_01"),
    faltantes_por_variable(epu, ["epu_mex"]).assign(fuente="INC_03"),
    faltantes_por_variable(tc, ["tc_fix"]).assign(fuente="MACRO_01"),
    faltantes_por_variable(vix, ["vix"]).assign(fuente="MACRO_02 (diario)")]).iloc[:, [4, 0, 1, 2, 3]],
    "Valores faltantes por variable en las fuentes limpias")
cerrar_checkpoint("CP3", "limpieza individual")

**Filas y columnas antes y después de limpiar, por fuente**

fuente,objeto_limpio,filas_iniciales,filas_finales,explicacion_cambio_filas,columnas_originales,columnas_homologadas,rango_fechas,n_ramas_o_series,llave,duplicados_en_llave
MX_PROD_01,emim_rama,1286,8858,"Ancho (1 fila por variable × unidad SCIAN) a largo; se conservan las 86 ramas × 4 variables (sector, subsectores y clases quedan en emim_largo para validar).","Variable, Tipo de dato, Sector SCIAN + 1 columna por mes","mes, rama, emim_personal_ocupado, emim_horas_trabajadas, emim_valor_produccion, emim_valor_ventas, emim_celdas_nd, emim_estatus_cifra",2018-01 a 2026-07,86,mes + rama,0
MX_EXP_01,expo (clases),"25,210","25,210",Sin eliminación de filas; conversión de tipos y fecha.,"Tipo de operación, Año, Mes, SCIAN, Clase, Valor en pesos, Valor en dólares","tipo_operacion, anio, mes_num, clase_scian, rama, valor_pesos, valor_usd, mes",2019-01 a 2026-07,86,mes + clase_scian,0
MX_EXP_01,expo_rama,"25,210",7826,Suma de clases SCIAN de 6 dígitos a rama de 4 dígitos.,(derivado de expo),"mes, rama, expo_valor_pesos, expo_valor_usd, expo_n_clases",2019-01 a 2026-07,86,mes + rama,0
MX_ARAN_01,census_manuf,"17,024","13,855",Se separan los códigos no manufactureros (se conservan en census_no_manuf; no se integran).,"fecha, naics, descripcion, valor_importado, valor_gravable, aranceles, tasa_efectiva, tasa_gravable","mes, rama, descripcion, valor_importado, valor_gravable, aranceles, tasa_efectiva, tasa_gravable",2013-01 a 2026-07,85,mes + rama,0
US_PROD_01,ip_largo,1292,"298,452",Ancho (mes × serie) a largo.,fecha + 1 columna por serie G.17,"mes, serie, indice, descripcion, naics, tipo_codigo, manufactura",1919-01 a 2026-08,231,mes + serie,0
PRECIOS_01,inpp_largo,560,"23,564",Se retiran metadatos y filas vacías; ancho a largo.,Fecha + 43 columnas por clave,"mes, id_serie, valor",1981-01 a 2026-08,43,mes + id_serie,0
INC_01,tpu,800,800,Solo conversión de tipos.,"DATE, N7, TPU_RAW, TPU_SHARE, TPU","mes, tpu, tpu_share, tpu_raw, tpu_n7",1960-01 a 2026-08,NA,mes,0
INC_02,wui_largo,675,"47,712",Tres hojas anchas apiladas en largo (mes × país × índice).,date + 71 columnas de país (por hoja),"indice, hoja, pais, mes, valor",2008-01 a 2026-08,71,indice + pais + mes,0
INC_02,wui_mundo,681,672,Solo columnas A:B; se descartan las celdas de texto de las gráficas.,date + promedio ponderado (+ celdas de gráfica),"mes, valor, indice, hoja, encabezado",2008-01 a 2026-08,NA,indice + mes,0
INC_03,epu,370,368,Se retiran el encabezado y la nota de la fuente.,"Year, Month, Mexican Policy Uncertainty Index","mes, epu_mex",1996-01 a 2026-08,NA,mes,0


**Filas o celdas descartadas y motivo (nada se descarta sin quedar aquí)**

fuente,filas_o_celdas,motivo
MX_PROD_01,8,"Filas de título, fecha de consulta y encabezado doble (no son datos)"
MX_PROD_01,22,Filas de notas al pie y filas vacías después del bloque de datos
MX_PROD_01,6280,"Celdas '-' de meses aún no publicados (2026-08, 2026-09, 2026-10, 2026-11, 2026-12), decisión D8"
MX_ARAN_01,3169,"Códigos no manufactureros (23 códigos 11xx, 21xx, 9xxx): solo SCIAN 31-33 (decisión 3 de las notas)."
MX_ARAN_02,163,Derivado de MX_ARAN_01 (misma información en formato ancho): solo se usa para verificar.
US_PROD_01,"146,969",Celdas vacías antes del inicio o después del fin de cada serie (no son datos; quedan como NA)
PRECIOS_01,12,Filas de metadatos del encabezado
PRECIOS_01,0,Filas vacías al final de la hoja
INC_01,4548,Hojas TPU_QUARTERLY y TPU_DAILY: frecuencia no mensual (se documentan).
INC_03,1,"Filas que no son datos (nota de la fuente): Source: 'Measuring Economic Policy Uncertainty' QJE, Nov. 20"


**Conversión de texto a número por fuente y variable**

fuente,variable,celdas_con_contenido,convertidas,codigos_de_faltante,codigos,no_convertibles,ejemplos_no_convertibles
MX_PROD_01,valores (todas las variables y niveles),"135,648","129,184",6464,"-, ND",0,
MX_EXP_01,Tipo de operación,"25,210","25,210",0,,0,
MX_EXP_01,Año,"25,210","25,210",0,,0,
MX_EXP_01,Mes,"25,210","25,210",0,,0,
MX_EXP_01,Valor en pesos,"25,210","25,210",0,,0,
MX_EXP_01,Valor en dólares,"25,210","25,210",0,,0,
MX_ARAN_01,valor_importado,"17,024","17,024",0,,0,
MX_ARAN_01,valor_gravable,"17,024","17,024",0,,0,
MX_ARAN_01,aranceles,"17,024","17,024",0,,0,
MX_ARAN_01,tasa_efectiva,"16,871","16,871",0,,0,


**Valores faltantes por variable en las fuentes limpias**

fuente,variable,n,n_faltantes,pct_faltantes
MX_PROD_01,emim_personal_ocupado,8858,0,0
MX_PROD_01,emim_horas_trabajadas,8858,0,0
MX_PROD_01,emim_valor_produccion,8858,24,0.27
MX_PROD_01,emim_valor_ventas,8858,24,0.27
MX_EXP_01,expo_valor_pesos,7826,0,0
MX_EXP_01,expo_valor_usd,7826,0,0
MX_ARAN_01,valor_importado,"13,855",0,0
MX_ARAN_01,valor_gravable,"13,855",0,0
MX_ARAN_01,aranceles,"13,855",0,0
MX_ARAN_01,tasa_efectiva,"13,855",0,0


**Resumen del CP3 (limpieza individual):** 90 OK · 14 REVISAR · 0 ERROR

fuente,prueba,resultado,estado
"MX_PROD_01 (largo, todos los niveles)",Llave única (variable + codigo + mes),0 combinaciones duplicadas de 135648 filas,OK
MX_PROD_01,Celdas no convertibles a número,0 celdas,OK
MX_PROD_01,"Los '-' (no publicado) ocupan meses completos, no celdas sueltas","Meses completos sin publicar: 2026-08, 2026-09, 2026-10, 2026-11, 2026-12",OK
MX_PROD_01,Se identificó en las notas desde cuándo hay cifras revisadas y preliminares,Revisadas desde 2025-01; preliminares desde 2025-08,OK
MX_PROD_01,Cada rama tiene un solo nombre en las cuatro variables,"86 ramas, 86 pares rama-nombre",OK
MX_PROD_01 (ramas),Fechas convertidas sin NA,0 fechas no convertidas,OK
MX_PROD_01 (ramas),Fechas en día 1 (frecuencia mensual),0 fechas con día distinto de 1,OK
MX_PROD_01 (ramas),"Meses consecutivos, sin huecos",2018-01 a 2026-07 (103 meses),OK
MX_PROD_01 (ramas),Llave única (mes + rama),0 combinaciones duplicadas de 8858 filas,OK
MX_PROD_01,Número de ramas (86 según el inventario),86 ramas,OK


## 4. Validaciones de cada fuente
Que los agregados de la EMIM sumen sus componentes, que la agregación de exportaciones coincida con la tabla dinámica del usuario y que el tipo de cambio implícito de las exportaciones sea razonable frente al FIX.

In [25]:
titulo_checkpoint("CHECKPOINT 3B · Validaciones de contenido por fuente")
emim_num = emim_largo[["variable", "nivel", "codigo", "mes", "valor"]]

def comparar_niveles(hijo, padre):
    h = emim_num[emim_num.nivel == hijo].copy()
    h["padre"] = "31-33" if padre == "sector" else h.codigo.str[: 3 if padre == "subsector" else 4]
    s = h.groupby(["variable", "padre", "mes"]).valor.agg(n_hijos="size", hijos_na=lambda v: int(v.isna().sum()),
                                                         suma_hijos=lambda v: v.sum(skipna=False)).reset_index()
    pa = emim_num[emim_num.nivel == padre].rename(columns={"codigo": "padre", "valor": "valor_padre"})[["variable", "padre", "mes", "valor_padre"]]
    s = s.merge(pa, on=["variable", "padre", "mes"], how="left", validate="one_to_one")
    s["comparable"] = (s.hijos_na == 0) & s.valor_padre.notna()
    s["diferencia"] = s.suma_hijos - s.valor_padre
    s["nivel"] = f"{hijo} → {padre}"
    return s
agg = pd.concat([comparar_niveles("subsector", "sector"), comparar_niveles("rama", "subsector"), comparar_niveles("clase", "rama")], ignore_index=True)
res_agg = agg.groupby("nivel").apply(lambda d: pd.Series(dict(
    comparaciones=len(d), comparables=int(d.comparable.sum()), no_comparables_por_ND=int((~d.comparable).sum()),
    con_diferencia=int((d.comparable & (d.diferencia.abs() > 0.5)).sum()),
    max_dif_abs=d.loc[d.comparable, "diferencia"].abs().max())), include_groups=False).reset_index()
for r in res_agg.itertuples():
    registrar("CP3B", "MX_PROD_01", f"La suma de componentes reproduce el agregado: {r.nivel}", r.con_diferencia == 0,
              f"{r.con_diferencia} de {r.comparables} comparaciones con diferencia > 0.5; {r.no_comparables_por_ND} no comparables por ND",
              si_falla="REVISAR")
tabla(res_agg, "EMIM: consistencia de agregación (sector ← subsectores ← ramas ← clases)")
dif = agg[agg.comparable & (agg.diferencia.abs() > 0.5)]
tabla(dif.groupby(["nivel", "variable", "padre"]).agg(meses=("mes", "size"), max_dif=("diferencia", lambda v: v.abs().max())).reset_index(),
      "EMIM: agregados cuya suma de componentes no coincide (si hay)")

td = crudos["expo_td"]
f_td = np.flatnonzero(((td.c1.map(norm_txt) == "ano") & (td.c2.map(norm_txt) == "mes")).to_numpy())
if len(f_td) == 1:
    cod_td = [str(v) if pd.notna(v) else f"sin_codigo_{i}" for i, v in enumerate(td.iloc[f_td[0]])]
    td_d = td.iloc[f_td[0] + 1:].copy()
    td_d.columns = ["anio", "mes_num", *cod_td[2:]]
    td_d["anio"] = td_d["anio"].ffill()  # completa el año de la tabla dinámica (solo verificación; nunca sobre la base)
    td_d = td_d[td_d.mes_num.fillna("").str.match(r"^\d+$")]
    td_l = td_d.melt(id_vars=["anio", "mes_num"], value_vars=[c for c in td_d.columns if re.fullmatch(r"\d{4}", str(c))],
                     var_name="rama", value_name="td_txt")
    td_l["mes"] = a_mes(pd.to_datetime(dict(year=td_l.anio.astype(float).astype(int), month=td_l.mes_num.astype(int), day=1)))
    td_l["td_pesos"] = pd.to_numeric(td_l.td_txt, errors="coerce")
    cmp_td = td_l[["mes", "rama", "td_pesos"]].merge(expo_rama[["mes", "rama", "expo_valor_pesos"]], on=["mes", "rama"], how="outer", validate="one_to_one")
    ok_td = cmp_td.td_pesos.notna() & cmp_td.expo_valor_pesos.notna() & ((cmp_td.td_pesos - cmp_td.expo_valor_pesos).abs() <= 1e-6 * np.maximum(1, cmp_td.expo_valor_pesos.abs()))
    registrar("CP3B", "MX_EXP_01", "La suma por rama coincide con la tabla dinámica TD del usuario", ok_td.all(),
              f"{(~ok_td).sum()} de {len(ok_td)} celdas mes-rama no coinciden o faltan en una de las dos", si_falla="REVISAR")
else:
    revisar("CP3B", "MX_EXP_01", "Comparación con la tabla dinámica TD", "No se encontró el encabezado Año/Mes en la hoja TD.")

tc_impl = expo.groupby("mes").apply(lambda d: d.valor_pesos.sum() / d.valor_usd.sum(), include_groups=False).rename("tc_implicito").reset_index()
tc_impl = tc_impl.merge(tc, on="mes", how="left", validate="one_to_one")
tc_impl["dif_pct"] = 100 * (tc_impl.tc_implicito / tc_impl.tc_fix - 1)
registrar("CP3B", "MX_EXP_01", "Tipo de cambio implícito (pesos/dólares) a menos de 5% del FIX en todos los meses",
          (tc_impl.dif_pct.abs() < 5).all(),
          f"Diferencia: mediana {tc_impl.dif_pct.median():.2f}%, mínima {tc_impl.dif_pct.min():.2f}%, máxima {tc_impl.dif_pct.max():.2f}%", si_falla="REVISAR")
tabla(tc_impl.reindex(tc_impl.dif_pct.abs().sort_values(ascending=False).index).head(10), "Meses con mayor diferencia entre tipo de cambio implícito y FIX")
clases_emim = set(emim_largo.codigo[emim_largo.nivel == "clase"])
clases_expo = set(expo.clase_scian)
revisar("CP3B", "MX_EXP_01", "Clases de exportación que no están en la EMIM",
        f"{len(clases_expo - clases_emim)} de {len(clases_expo)} clases de exportación no están entre las {len(clases_emim)} clases de la EMIM. "
        "Al agregar a rama se incluyen todas.")
cerrar_checkpoint("CP3B", "validaciones de contenido por fuente")

**EMIM: consistencia de agregación (sector ← subsectores ← ramas ← clases)**

nivel,comparaciones,comparables,no_comparables_por_ND,con_diferencia,max_dif_abs
clase → rama,"35,432.00","35,296.00",136,0,0
rama → subsector,8652,8604,48,0,0
subsector → sector,412,412,0,0,0


**EMIM: agregados cuya suma de componentes no coincide (si hay)**

_(sin filas)_

**Meses con mayor diferencia entre tipo de cambio implícito y FIX**

mes,tc_implicito,tc_fix,dif_pct
2020-03-01,22.1403,22.3784,-1.06416
2024-06-01,18.0879,18.2174,-0.711105
2023-11-01,17.4693,17.3767,0.532634
2020-10-01,21.3838,21.2705,0.532512
2020-09-01,21.7945,21.681,0.52349
2020-11-01,20.4824,20.3819,0.492929
2026-04-01,17.5044,17.4187,0.491888
2020-05-01,23.5263,23.423,0.441016
2024-07-01,18.1871,18.1117,0.41648
2022-06-01,19.9435,20.0237,-0.400402


**Resumen del CP3B (validaciones de contenido por fuente):** 5 OK · 1 REVISAR · 0 ERROR

fuente,prueba,resultado,estado
MX_PROD_01,La suma de componentes reproduce el agregado: clase → rama,0.0 de 35296.0 comparaciones con diferencia > 0.5; 136.0 no comparables por ND,OK
MX_PROD_01,La suma de componentes reproduce el agregado: rama → subsector,0.0 de 8604.0 comparaciones con diferencia > 0.5; 48.0 no comparables por ND,OK
MX_PROD_01,La suma de componentes reproduce el agregado: subsector → sector,0.0 de 412.0 comparaciones con diferencia > 0.5; 0.0 no comparables por ND,OK
MX_EXP_01,La suma por rama coincide con la tabla dinámica TD del usuario,0 de 7826 celdas mes-rama no coinciden o faltan en una de las dos,OK
MX_EXP_01,Tipo de cambio implícito (pesos/dólares) a menos de 5% del FIX en todos los meses,"Diferencia: mediana 0.05%, mínima -1.06%, máxima 0.53%",OK
MX_EXP_01,Clases de exportación que no están en la EMIM,81 de 279 clases de exportación no están entre las 206 clases de la EMIM. Al agregar a rama se incluyen todas.,REVISAR


## 5. Homologación de fechas, ramas y unidades
### Checkpoint 4: correspondencias
Todas las fuentes quedan con `mes` (primer día del mes) y las ramas con el **código SCIAN de 4 dígitos como texto**. La correspondencia NAICS = SCIAN se **verifica con los clasificadores oficiales** (INEGI y Census) antes de usarla: (a) la rama/subsector SCIAN 2018 es categoría trilateral (título con «T»); (b) el código existe en NAICS 2017 y 2022; (c) ninguna clase cambió de rama/subsector entre NAICS 2012, 2017 y 2022.

In [26]:
titulo_checkpoint("CHECKPOINT 4 · Homologación y verificación de correspondencias")
hom_meses = pd.DataFrame([
    ("MX_PROD_01", "Encabezado doble: año + nombre del mes en español", emim_rama.mes.min(), emim_rama.mes.max()),
    ("MX_EXP_01", "Columnas Año y Mes (números)", expo_rama.mes.min(), expo_rama.mes.max()),
    ("MX_ARAN_01", "Texto AAAA-MM-DD (día 1)", census_manuf.mes.min(), census_manuf.mes.max()),
    ("US_PROD_01", "Texto AAAA-MM-DD (día 1)", ip_largo.mes.min(), ip_largo.mes.max()),
    ("PRECIOS_01", "Fecha de Excel", inpp_largo.mes.min(), inpp_largo.mes.max()),
    ("INC_01", "Fecha de Excel (día 1)", tpu.mes.min(), tpu.mes.max()),
    ("INC_02", "Fecha de Excel (día 1)", wui_largo.mes.min(), wui_largo.mes.max()),
    ("INC_03", "Columnas Year (texto) y Month (número)", epu.mes.min(), epu.mes.max()),
    ("MACRO_01", "Fecha de Excel (día 1)", tc.mes.min(), tc.mes.max()),
    ("MACRO_02", "Texto AAAA-MM-DD, diario", vix.fecha.min(), vix.fecha.max()),
], columns=["fuente", "formato_original", "primera_fecha", "ultima_fecha"])
hom_meses["formato_homologado"] = np.where(hom_meses.fuente == "MACRO_02", "diario → mensual con la regla D6", "mes = fecha, día 1")
tabla(hom_meses, "Meses: formato original y homologado", ancho=120)

def leer_scian(hoja):
    xs = leer_texto(ruta("CLAS_01"), hoja)
    f = np.flatnonzero((xs.c1.map(norm_txt) == "codigo").to_numpy())
    exigir(len(f) >= 1, f"Catálogo SCIAN 2018, hoja {hoja}: no se encontró el encabezado 'Código'.")
    d = xs.iloc[f[0] + 1:].dropna(subset=["c1"])
    out = pd.DataFrame({"codigo": d.c1.str.strip().str.replace(r"\.0+$", "", regex=True), "titulo_original": d.c2.str.strip()})
    out["trilateral"] = out.titulo_original.str.endswith("T")
    out["titulo"] = out.titulo_original.str.replace(r"T$", "", regex=True).str.strip()
    return out

def leer_naics(id_):
    xs = leer_texto(ruta(id_), 0)
    f = np.flatnonzero(xs.c2.map(norm_txt).fillna("").str.contains("naics.*code").to_numpy())
    exigir(len(f) >= 1, f"{id_}: no se encontró el encabezado de códigos NAICS en la columna B.")
    d = xs.iloc[f[0] + 1:].dropna(subset=["c2"])
    return pd.DataFrame({"codigo": d.c2.str.strip().str.replace(r"\.0+$", "", regex=True),
                         "titulo": d.c3.str.strip().str.replace(r"T$", "", regex=True).str.strip()})

def leer_concordancia(id_):
    xs = leer_texto(ruta(id_), 0)
    f = np.flatnonzero(xs.c1.map(norm_txt).fillna("").str.contains("naics code").to_numpy())
    exigir(len(f) >= 1, f"{id_}: no se encontró el encabezado de la concordancia.")
    d = xs.iloc[f[0] + 1:].dropna(subset=["c1", "c3"])
    return pd.DataFrame({"codigo_nuevo": d.c1.str.strip().str.replace(r"\.0+$", "", regex=True),
                         "codigo_anterior": d.c3.str.strip().str.replace(r"\.0+$", "", regex=True)})

scian_rama = leer_scian("RAMA").query("codigo.str.match('^3[1-3]\\\\d{2}$')", engine="python")
scian_sub = leer_scian("SUBSECTOR").query("codigo.str.match('^3[1-3]\\\\d$')", engine="python")
naics17, naics22 = leer_naics("CLAS_02"), leer_naics("CLAS_03")
exigir(len(scian_rama) and len(naics17) and len(naics22), "No se pudieron leer los clasificadores oficiales.")
conc = pd.concat([leer_concordancia("CLAS_04").assign(transicion="NAICS 2017 -> 2022"),
                  leer_concordancia("CLAS_05").assign(transicion="NAICS 2012 -> 2017")], ignore_index=True)
conc = conc[conc.codigo_nuevo.str.match(r"^\d{6}$") & conc.codigo_anterior.str.match(r"^\d{6}$") &
            (conc.codigo_nuevo.str.match(r"^3[1-3]") | conc.codigo_anterior.str.match(r"^3[1-3]"))]
cambios_rama = conc[conc.codigo_nuevo.str[:4] != conc.codigo_anterior.str[:4]]
cambios_sub = conc[conc.codigo_nuevo.str[:3] != conc.codigo_anterior.str[:3]]
registrar("CP4", "Clasificadores", "Se leyeron las concordancias NAICS 2012-2017-2022 (clases manufactureras)", len(conc) > 0, f"{len(conc)} pares de clases")
ramas_con_cambio = set(cambios_rama.codigo_nuevo.str[:4]) | set(cambios_rama.codigo_anterior.str[:4])
subs_con_cambio = set(cambios_sub.codigo_nuevo.str[:3]) | set(cambios_sub.codigo_anterior.str[:3])

verif_corr = pd.DataFrame({"rama": sorted(emim_rama.rama.unique())})
verif_corr["subsector"] = verif_corr.rama.str[:3]
verif_corr = (verif_corr
    .merge(scian_rama.rename(columns={"codigo": "rama", "titulo": "scian_titulo", "trilateral": "scian_trilateral"})[["rama", "scian_titulo", "scian_trilateral"]], on="rama", how="left", validate="one_to_one")
    .merge(scian_sub.rename(columns={"codigo": "subsector", "trilateral": "subsector_trilateral"})[["subsector", "subsector_trilateral"]], on="subsector", how="left", validate="many_to_one")
    .merge(naics17.rename(columns={"codigo": "rama", "titulo": "naics2017_titulo"}), on="rama", how="left", validate="one_to_one")
    .merge(naics22.rename(columns={"codigo": "rama", "titulo": "naics2022_titulo"}), on="rama", how="left", validate="one_to_one"))
verif_corr["scian_trilateral"] = verif_corr.scian_trilateral.eq(True)
verif_corr["subsector_trilateral"] = verif_corr.subsector_trilateral.eq(True)
verif_corr["en_naics_2017"] = verif_corr.naics2017_titulo.notna()
verif_corr["en_naics_2022"] = verif_corr.naics2022_titulo.notna()
verif_corr["rama_sin_cambios_2012_2022"] = ~verif_corr.rama.isin(ramas_con_cambio)
verif_corr["subsector_en_naics_2017_2022"] = verif_corr.subsector.isin(naics17.codigo) & verif_corr.subsector.isin(naics22.codigo)
verif_corr["subsector_sin_cambios_2012_2022"] = ~verif_corr.subsector.isin(subs_con_cambio)
verif_corr["correspondencia_rama_verificada"] = (verif_corr.scian_trilateral & verif_corr.en_naics_2017 & verif_corr.en_naics_2022
                                                 & verif_corr.rama_sin_cambios_2012_2022)
verif_corr["correspondencia_subsector_verificada"] = (verif_corr.subsector_trilateral & verif_corr.subsector_en_naics_2017_2022
                                                      & verif_corr.subsector_sin_cambios_2012_2022)
ramas_verificadas = set(verif_corr.rama[verif_corr.correspondencia_rama_verificada])
subsectores_verificados = set(verif_corr.subsector[verif_corr.correspondencia_subsector_verificada])
registrar("CP4", "Correspondencia NAICS-SCIAN", "Todas las ramas EMIM son categorías trilaterales (marca T) en el SCIAN 2018",
          verif_corr.scian_trilateral.all(), "No trilaterales: " + ", ".join(verif_corr.rama[~verif_corr.scian_trilateral]), si_falla="REVISAR")
registrar("CP4", "Correspondencia NAICS-SCIAN", "Todas las ramas EMIM existen con el mismo código en NAICS 2017 y 2022",
          (verif_corr.en_naics_2017 & verif_corr.en_naics_2022).all(),
          "Faltan: " + ", ".join(verif_corr.rama[~(verif_corr.en_naics_2017 & verif_corr.en_naics_2022)]), si_falla="REVISAR")
registrar("CP4", "Correspondencia NAICS-SCIAN", "Ninguna clase manufacturera cambió de rama entre NAICS 2012, 2017 y 2022",
          len(cambios_rama) == 0, "Ramas afectadas: " + ", ".join(sorted(ramas_con_cambio)), si_falla="REVISAR")
registrar("CP4", "Correspondencia NAICS-SCIAN", "Los 21 subsectores son trilaterales, existen en NAICS 2017/2022 y no cambiaron",
          verif_corr.correspondencia_subsector_verificada.all(),
          "No verificados: " + ", ".join(sorted(set(verif_corr.subsector) - subsectores_verificados)), si_falla="REVISAR")
if len(ramas_verificadas) < len(verif_corr):
    revisar("CP4", "Correspondencia NAICS-SCIAN", "Ramas excluidas de la asignación por código",
            "No se integran datos del Census ni del G.17 (rama) para: " + ", ".join(sorted(set(verif_corr.rama) - ramas_verificadas)))
tabla(verif_corr[["rama", "scian_titulo", "naics2017_titulo", "scian_trilateral", "en_naics_2022", "rama_sin_cambios_2012_2022",
                  "correspondencia_rama_verificada", "correspondencia_subsector_verificada"]],
      "Verificación oficial de la correspondencia rama SCIAN 2018 = NAICS (INEGI y Census)", max_filas=100, ancho=70)

ramas_emim = nombres_rama.sort_values("rama").reset_index(drop=True)
subsectores_emim = (emim_largo[emim_largo.nivel == "subsector"][["codigo", "nombre"]].drop_duplicates()
                    .rename(columns={"codigo": "subsector", "nombre": "subsector_nombre"}))
g17_4 = ip_cat[(ip_cat.tipo_codigo == "NAICS 4 dígitos") & ip_cat.manufactura].rename(columns={"naics": "rama", "serie": "g17_serie_rama", "descripcion": "g17_desc_rama"})[["rama", "g17_serie_rama", "g17_desc_rama"]]
g17_3 = ip_cat[(ip_cat.tipo_codigo == "NAICS 3 dígitos") & ip_cat.manufactura].rename(columns={"naics": "subsector", "serie": "g17_serie_subsector", "descripcion": "g17_desc_subsector"})[["subsector", "g17_serie_subsector", "g17_desc_subsector"]]
exigir(not g17_4.rama.duplicated().any() and not g17_3.subsector.duplicated().any(), "G.17: códigos NAICS repetidos; correspondencia ambigua.")
census_desc = census_manuf[["rama", "descripcion"]].drop_duplicates().rename(columns={"descripcion": "census_desc"})
correspondencias = pd.DataFrame({"rama": sorted(set(emim_rama.rama) | set(expo_rama.rama) | set(census_manuf.rama))})
correspondencias = (correspondencias.merge(ramas_emim, on="rama", how="left")
                    .assign(subsector=lambda d: d.rama.str[:3], en_emim=lambda d: d.rama.isin(emim_rama.rama),
                            en_exportaciones=lambda d: d.rama.isin(expo_rama.rama), en_census=lambda d: d.rama.isin(census_manuf.rama))
                    .merge(census_desc, on="rama", how="left")
                    .merge(corr[["rama_scian", "id_serie_inpp", "nombre_rama"]].rename(columns={"rama_scian": "rama", "id_serie_inpp": "inpp_serie", "nombre_rama": "nombre_rama_corr"}), on="rama", how="left")
                    .merge(g17_3, on="subsector", how="left").merge(g17_4, on="rama", how="left")
                    .merge(verif_corr[["rama", "correspondencia_rama_verificada", "correspondencia_subsector_verificada"]], on="rama", how="left"))
tabla(correspondencias[["rama", "rama_nombre", "en_emim", "en_exportaciones", "en_census", "census_desc", "inpp_serie",
                        "g17_serie_subsector", "g17_serie_rama", "g17_desc_rama"]],
      "Correspondencia de ramas entre fuentes (solo por código idéntico)", max_filas=200, ancho=60)
registrar("CP4", "Ramas", "Las ramas de exportaciones son las mismas que las de la EMIM", set(expo_rama.rama) == set(emim_rama.rama), "", si_falla="REVISAR")
solo_census = sorted(set(census_manuf.rama) - set(emim_rama.rama))
registrar("CP4", "Ramas", "Todo código manufacturero del Census existe como rama EMIM", not solo_census,
          "Sin rama EMIM (no se integran): " + ", ".join(solo_census) if solo_census else "Sí", si_falla="REVISAR")
revisar("CP4", "Ramas", "Ramas EMIM sin datos del Census",
        ", ".join(sorted(set(emim_rama.rama) - set(census_manuf.rama))) + ": quedan con NA en eeuu_imp_mx_* (no se completan).")
registrar("CP4", "Ramas", "La correspondencia rama-INPP cubre todas las ramas EMIM", set(corr.rama_scian) == set(emim_rama.rama), "")
dif_n = correspondencias[correspondencias.en_emim & correspondencias.nombre_rama_corr.notna() &
                         (correspondencias.rama_nombre.map(norm_txt) != correspondencias.nombre_rama_corr.map(norm_txt))]
registrar("CP4", "Ramas", "Nombres de rama iguales en la EMIM y en la tabla de correspondencia", len(dif_n) == 0,
          ("Distintos: " + ", ".join(dif_n.rama)) if len(dif_n) else "Sí", si_falla="REVISAR")
revisar("CP4", "Conceptos", "Census: clasificación por producto, no por establecimiento",
        "La correspondencia de códigos está verificada, pero el Census asigna cada producto importado a una industria NAICS (concordancia HS-NAICS), "
        "mientras la EMIM clasifica establecimientos. Miden conceptos relacionados, no idénticos.")
sin_g17 = correspondencias[correspondencias.en_emim & correspondencias.g17_serie_rama.isna()]
revisar("CP4", "US_PROD_01", "Ramas sin serie G.17 de 4 dígitos idéntica",
        f"{len(sin_g17)} de {correspondencias.en_emim.sum()} ramas ({', '.join(sin_g17.rama)}). Quedan en NA en eeuu_ip_rama; no se usan agregados ni rangos para completarlas.")
g17_no_usadas = ip_cat[ip_cat.manufactura & ~ip_cat.tipo_codigo.isin(["NAICS 4 dígitos", "NAICS 3 dígitos"])]
revisar("CP4", "US_PROD_01", "Series G.17 manufactureras no asignadas (rangos, listas, partes de industria o 5-6 dígitos)",
        f"{len(g17_no_usadas)} series. Asignarlas requeriría una regla de correspondencia explícita.")
registrar("CP4", "US_PROD_01", "Todos los subsectores EMIM tienen serie G.17 de 3 dígitos idéntica",
          set(emim_rama.rama.str[:3]) <= set(g17_3.subsector), "", si_falla="REVISAR")
hom_unid = pd.DataFrame([
    ("MX_PROD_01", "Valor de producción y ventas", "Miles de pesos corrientes", "Se conservan y se agrega versión en miles de pesos reales (D4)."),
    ("MX_PROD_01", "Personal ocupado / horas", "Personas / miles de horas", "Se conservan."),
    ("MX_EXP_01", "Valor de exportaciones", "Pesos y dólares corrientes", "Se conservan; se agrega expo_valor_real = pesos / 1,000, deflactado (D4)."),
    ("MX_ARAN_01", "Importaciones, valor gravable y aranceles", "Dólares corrientes", "Se conservan; se agregan versiones × FIX / 1,000, deflactadas (D4)."),
    ("MX_ARAN_01", "Tasas", "Proporción (0.01 = 1%)", "Se conservan."),
    ("PRECIOS_01", "INPP", "Índice; base declarada jul-2025=100, base efectiva jul-2019=100", "Se conserva; es el deflactor."),
    ("US_PROD_01", "Producción industrial EE.UU.", "Índice 2017=100 (según el script; por confirmar), SA", "Se conserva."),
    ("INC_01", "TPU", "Índice (100 = 1% de artículos)", "Se conserva."),
    ("INC_02", "WUI/WTUI/WPUI", "Palabras / total de palabras × 1,000 (países); promedio ponderado por PIB con otra escala (mundo)", "Se conservan."),
    ("INC_03", "EPU México", "Índice normalizado", "Se conserva."),
    ("MACRO_01", "Tipo de cambio FIX", "Pesos por dólar, promedio mensual", "Se conserva; convierte dólares a pesos (D4)."),
    ("MACRO_02", "VIX", "Puntos de índice", "Promedio mensual (D6)."),
], columns=["fuente", "variable", "unidad_original", "tratamiento"])
tabla(hom_unid, "Unidades: originales y tratamiento", ancho=150)
revisar("CP4", "Unidades", "Un solo deflactor para todas las variables monetarias de la rama (D4)",
        "Producción, ventas, exportaciones e importaciones de EE.UU. desde México se deflactan con el INPP de mercancías finales del subsector "
        "(base efectiva jul-2019=100); los dólares se convierten antes con el FIX promedio del mes.")
revisar("CP4", "Conceptos", "Exportaciones mexicanas vs. importaciones de EE.UU. desde México",
        "Flujos distintos (destino, registro, valoración); se conservan como variables separadas.")
revisar("CP4", "Conceptos", "INPP de mercancías finales como deflactor", "Excluye bienes intermedios y es de subsector: las ramas de un subsector comparten deflactor.")
cerrar_checkpoint("CP4", "homologación")

**Meses: formato original y homologado**

fuente,formato_original,primera_fecha,ultima_fecha,formato_homologado
MX_PROD_01,Encabezado doble: año + nombre del mes en español,2018-01-01,2026-07-01,"mes = fecha, día 1"
MX_EXP_01,Columnas Año y Mes (números),2019-01-01,2026-07-01,"mes = fecha, día 1"
MX_ARAN_01,Texto AAAA-MM-DD (día 1),2013-01-01,2026-07-01,"mes = fecha, día 1"
US_PROD_01,Texto AAAA-MM-DD (día 1),1919-01-01,2026-08-01,"mes = fecha, día 1"
PRECIOS_01,Fecha de Excel,1981-01-01,2026-08-01,"mes = fecha, día 1"
INC_01,Fecha de Excel (día 1),1960-01-01,2026-08-01,"mes = fecha, día 1"
INC_02,Fecha de Excel (día 1),2008-01-01,2026-08-01,"mes = fecha, día 1"
INC_03,Columnas Year (texto) y Month (número),1996-01-01,2026-08-01,"mes = fecha, día 1"
MACRO_01,Fecha de Excel (día 1),1991-01-01,2026-08-01,"mes = fecha, día 1"
MACRO_02,"Texto AAAA-MM-DD, diario",1990-01-02,2026-09-22,diario → mensual con la regla D6


**Verificación oficial de la correspondencia rama SCIAN 2018 = NAICS (INEGI y Census)**

rama,scian_titulo,naics2017_titulo,scian_trilateral,en_naics_2022,rama_sin_cambios_2012_2022,correspondencia_rama_verificada,correspondencia_subsector_verificada
3111,Elaboración de alimentos para animales,Animal Food Manufacturing,TRUE,TRUE,TRUE,TRUE,TRUE
3112,Molienda de granos y de semillas y obtención de aceites y grasas,Grain and Oilseed Milling,TRUE,TRUE,TRUE,TRUE,TRUE
3113,"Elaboración de azúcares, chocolates, dulces y similares",Sugar and Confectionery Product Manufacturing,TRUE,TRUE,TRUE,TRUE,TRUE
3114,"Conservación de frutas, verduras, guisos y otros alimentos preparados",Fruit and Vegetable Preserving and Specialty Food Manufacturing,TRUE,TRUE,TRUE,TRUE,TRUE
3115,Elaboración de productos lácteos,Dairy Product Manufacturing,TRUE,TRUE,TRUE,TRUE,TRUE
3116,"Matanza, empacado y procesamiento de carne de ganado, aves y otros ...",Animal Slaughtering and Processing,TRUE,TRUE,TRUE,TRUE,TRUE
3117,Preparación y envasado de pescados y mariscos,Seafood Product Preparation and Packaging,TRUE,TRUE,TRUE,TRUE,TRUE
3118,Elaboración de productos de panadería y tortillas,Bakeries and Tortilla Manufacturing,TRUE,TRUE,TRUE,TRUE,TRUE
3119,Otras industrias alimentarias,Other Food Manufacturing,TRUE,TRUE,TRUE,TRUE,TRUE
3121,Industria de las bebidas,Beverage Manufacturing,TRUE,TRUE,TRUE,TRUE,TRUE


**Correspondencia de ramas entre fuentes (solo por código idéntico)**

rama,rama_nombre,en_emim,en_exportaciones,en_census,census_desc,inpp_serie,g17_serie_subsector,g17_serie_rama,g17_desc_rama
3111,Elaboración de alimentos para animales,TRUE,TRUE,TRUE,ANIMAL FOODS,1370012,G311,G3111,Animal food
3112,Molienda de granos y de semillas y obtención de aceites y...,TRUE,TRUE,TRUE,GRAIN & OILSEED MILLING PRODUCTS,1370012,G311,G3112,Grain and oilseed milling
3113,"Elaboración de azúcares, chocolates, dulces y similares",TRUE,TRUE,TRUE,SUGAR & CONFECTIONERY PRODUCTS,1370012,G311,G3113,Sugar and confectionery product
3114,"Conservación de frutas, verduras, guisos y otros alimento...",TRUE,TRUE,TRUE,FRUITS & VEG PRESERVES & SPECIALTY FOODS,1370012,G311,G3114,Fruit and vegetable preserving and specialty food
3115,Elaboración de productos lácteos,TRUE,TRUE,TRUE,DAIRY PRODUCTS,1370012,G311,G3115,Dairy product
3116,"Matanza, empacado y procesamiento de carne de ganado, ave...",TRUE,TRUE,TRUE,MEAT PRODUCTS & MEAT PACKAGING PRODUCTS,1370012,G311,G3116,Animal slaughtering and processing
3117,Preparación y envasado de pescados y mariscos,TRUE,TRUE,TRUE,"SEAFOOD PRODS, PREPARED, CANNED & PACKAGED",1370012,G311,NA,NA
3118,Elaboración de productos de panadería y tortillas,TRUE,TRUE,TRUE,BAKERY & TORTILLA PRODUCTS,1370012,G311,N3118,Bakeries and tortilla
3119,Otras industrias alimentarias,TRUE,TRUE,TRUE,"FOODS, NESOI",1370012,G311,G3119,Other food
3121,Industria de las bebidas,TRUE,TRUE,TRUE,BEVERAGES,1370013,G312,G3121,Beverage


**Unidades: originales y tratamiento**

fuente,variable,unidad_original,tratamiento
MX_PROD_01,Valor de producción y ventas,Miles de pesos corrientes,Se conservan y se agrega versión en miles de pesos reales (D4).
MX_PROD_01,Personal ocupado / horas,Personas / miles de horas,Se conservan.
MX_EXP_01,Valor de exportaciones,Pesos y dólares corrientes,"Se conservan; se agrega expo_valor_real = pesos / 1,000, deflactado (D4)."
MX_ARAN_01,"Importaciones, valor gravable y aranceles",Dólares corrientes,"Se conservan; se agregan versiones × FIX / 1,000, deflactadas (D4)."
MX_ARAN_01,Tasas,Proporción (0.01 = 1%),Se conservan.
PRECIOS_01,INPP,"Índice; base declarada jul-2025=100, base efectiva jul-2019=100",Se conserva; es el deflactor.
US_PROD_01,Producción industrial EE.UU.,"Índice 2017=100 (según el script; por confirmar), SA",Se conserva.
INC_01,TPU,Índice (100 = 1% de artículos),Se conserva.
INC_02,WUI/WTUI/WPUI,"Palabras / total de palabras × 1,000 (países); promedio ponderado por PIB con otra escala (mundo)",Se conservan.
INC_03,EPU México,Índice normalizado,Se conserva.


**Resumen del CP4 (homologación):** 10 OK · 7 REVISAR · 0 ERROR

fuente,prueba,resultado,estado
Clasificadores,Se leyeron las concordancias NAICS 2012-2017-2022 (clases manufactureras),725 pares de clases,OK
Correspondencia NAICS-SCIAN,Todas las ramas EMIM son categorías trilaterales (marca T) en el SCIAN 2018,No trilaterales:,OK
Correspondencia NAICS-SCIAN,Todas las ramas EMIM existen con el mismo código en NAICS 2017 y 2022,Faltan:,OK
Correspondencia NAICS-SCIAN,"Ninguna clase manufacturera cambió de rama entre NAICS 2012, 2017 y 2022",Ramas afectadas:,OK
Correspondencia NAICS-SCIAN,"Los 21 subsectores son trilaterales, existen en NAICS 2017/2022 y no cambiaron",No verificados:,OK
Ramas,Las ramas de exportaciones son las mismas que las de la EMIM,,OK
Ramas,Todo código manufacturero del Census existe como rama EMIM,Sí,OK
Ramas,Ramas EMIM sin datos del Census,3328: quedan con NA en eeuu_imp_mx_* (no se completan).,REVISAR
Ramas,La correspondencia rama-INPP cubre todas las ramas EMIM,,OK
Ramas,Nombres de rama iguales en la EMIM y en la tabla de correspondencia,Sí,OK


## 6. Integración de las variables por rama
Cada fuente se convierte en una *pieza* con una llave explícita: `mes + rama` (por rama), `rama` (atributos fijos) o `mes` (comunes). Cada pieza conserva toda su cobertura; la unión documenta qué parte cae fuera del panel.

In [27]:
piezas = {}
def pieza(nombre, fuente_id, llave, datos, cobertura):
    tipo = "por rama (mes-rama)" if llave == ["mes", "rama"] else ("atributo de rama" if llave == ["rama"] else "común por mes")
    return dict(nombre=nombre, fuente_id=fuente_id, llave=llave, datos=datos.reset_index(drop=True), cobertura_geografica=cobertura, tipo=tipo)

piezas["atributos"] = pieza("Atributos de la rama (EMIM)", "MX_PROD_01", ["rama"],
                            ramas_emim.assign(subsector=lambda d: d.rama.str[:3]).merge(subsectores_emim, on="subsector", how="left", validate="many_to_one"), "México")
piezas["exposicion"] = pieza("Exposición: coeficiente de exportación MIP 2018 (D9)", "MIP_01", ["rama"],
                             mip[["rama", "mip_coef_exportacion_2018"]], "México (todos los destinos)")
piezas["emim"] = pieza("EMIM", "MX_PROD_01", ["mes", "rama"], emim_rama, "México")
piezas["expo"] = pieza("Exportaciones", "MX_EXP_01", ["mes", "rama"], expo_rama, "México")
if decisiones["aranceles_por_codigo_identico"]:
    piezas["census"] = pieza("Census: importaciones de EE.UU. desde México y aranceles", "MX_ARAN_01", ["mes", "rama"],
        census_manuf[census_manuf.rama.isin(ramas_verificadas)].rename(columns={
            "valor_importado": "eeuu_imp_mx_valor_usd", "valor_gravable": "eeuu_imp_mx_valor_gravable_usd", "aranceles": "eeuu_imp_mx_aranceles_usd",
            "tasa_efectiva": "eeuu_imp_mx_tasa_efectiva", "tasa_gravable": "eeuu_imp_mx_tasa_gravable"}).drop(columns="descripcion"),
        "EE.UU. (importador) / México (origen)")
piezas["inpp"] = pieza("INPP del subsector de la rama", "PRECIOS_01", ["mes", "rama"],
    corr[["rama_scian", "id_serie_inpp"]].rename(columns={"rama_scian": "rama", "id_serie_inpp": "id_serie"})
    .merge(inpp_largo, on="id_serie", how="inner", validate="many_to_many")
    .rename(columns={"valor": "inpp_subsector_indice", "id_serie": "inpp_subsector_serie"})[["mes", "rama", "inpp_subsector_indice", "inpp_subsector_serie"]], "México")
ip_m = ip_largo[["mes", "serie", "indice"]]
if decisiones["ip_eeuu_subsector_codigo_identico"]:
    piezas["g17_sub"] = pieza("G.17: producción industrial de EE.UU. del subsector (NAICS 3 idéntico)", "US_PROD_01", ["mes", "rama"],
        ramas_emim[["rama"]].assign(subsector=lambda d: d.rama.str[:3]).query("subsector in @subsectores_verificados")
        .merge(g17_3, on="subsector", how="inner", validate="many_to_one")
        .merge(ip_m, left_on="g17_serie_subsector", right_on="serie", how="inner", validate="many_to_many")
        .rename(columns={"indice": "eeuu_ip_subsector", "g17_serie_subsector": "eeuu_ip_subsector_serie"})[["mes", "rama", "eeuu_ip_subsector", "eeuu_ip_subsector_serie"]], "EE.UU.")
if decisiones["ip_eeuu_rama_codigo_identico"]:
    piezas["g17_rama"] = pieza("G.17: producción industrial de EE.UU. de la rama (NAICS 4 idéntico)", "US_PROD_01", ["mes", "rama"],
        ramas_emim[["rama"]].query("rama in @ramas_verificadas")
        .merge(g17_4, on="rama", how="inner", validate="one_to_one")
        .merge(ip_m, left_on="g17_serie_rama", right_on="serie", how="inner", validate="one_to_many")
        .rename(columns={"indice": "eeuu_ip_rama", "g17_serie_rama": "eeuu_ip_rama_serie"})[["mes", "rama", "eeuu_ip_rama", "eeuu_ip_rama_serie"]], "EE.UU.")
piezas["tpu"] = pieza("TPU (Caldara, Iacoviello et al.)", "INC_01", ["mes"], tpu, "EE.UU. (prensa estadounidense)")
wui_sel = wui_largo[wui_largo.pais.isin(decisiones["wui_paises"])].assign(variable=lambda d: d.indice.str.lower() + "_" + d.pais.str.lower())[["mes", "variable", "valor"]]
if decisiones["wui_incluir_promedio_mundial"]:
    wui_sel = pd.concat([wui_sel, wui_mundo.assign(variable=lambda d: d.indice.str.lower() + "_mundo_pib")[["mes", "variable", "valor"]]])
wui_ancho = wui_sel.pivot(index="mes", columns="variable", values="valor").reset_index().rename_axis(columns=None)
orden_wui = [f"{i}_{s}" for i in ("wui", "wtui", "wpui") for s in [p.lower() for p in decisiones["wui_paises"]] + ["mundo_pib"]]
piezas["wui"] = pieza("World Uncertainty Index (WUI, WTUI, WPUI)", "INC_02", ["mes"], wui_ancho[["mes", *[c for c in orden_wui if c in wui_ancho]]], "País (MEX, USA) y mundo")
piezas["epu"] = pieza("EPU México (Baker, Bloom y Davis)", "INC_03", ["mes"], epu, "México")
piezas["tc"] = pieza("Tipo de cambio FIX", "MACRO_01", ["mes"], tc, "México")
if vix_mensual is not None:
    piezas["vix"] = pieza("VIX mensual", "MACRO_02", ["mes"], vix_mensual, "EE.UU. (mercado financiero)")
for p_ in piezas.values():
    if "mes" in p_["datos"]:
        p_["datos"]["mes"] = a_mes(p_["datos"]["mes"])
tabla(pd.DataFrame([dict(pieza=k, nombre=v["nombre"], fuente=v["fuente_id"], tipo=v["tipo"], llave=" + ".join(v["llave"]),
                         variables=", ".join(c for c in v["datos"].columns if c not in v["llave"])) for k, v in piezas.items()]),
      "Piezas que se integrarán, en este orden", ancho=250)

**Piezas que se integrarán, en este orden**

pieza,nombre,fuente,tipo,llave,variables
atributos,Atributos de la rama (EMIM),MX_PROD_01,atributo de rama,rama,"rama_nombre, subsector, subsector_nombre"
exposicion,Exposición: coeficiente de exportación MIP 2018 (D9),MIP_01,atributo de rama,rama,mip_coef_exportacion_2018
emim,EMIM,MX_PROD_01,por rama (mes-rama),mes + rama,"emim_personal_ocupado, emim_horas_trabajadas, emim_valor_produccion, emim_valor_ventas, emim_celdas_nd, emim_estatus_cifra"
expo,Exportaciones,MX_EXP_01,por rama (mes-rama),mes + rama,"expo_valor_pesos, expo_valor_usd, expo_n_clases"
census,Census: importaciones de EE.UU. desde México y aranceles,MX_ARAN_01,por rama (mes-rama),mes + rama,"eeuu_imp_mx_valor_usd, eeuu_imp_mx_valor_gravable_usd, eeuu_imp_mx_aranceles_usd, eeuu_imp_mx_tasa_efectiva, eeuu_imp_mx_tasa_gravable"
inpp,INPP del subsector de la rama,PRECIOS_01,por rama (mes-rama),mes + rama,"inpp_subsector_indice, inpp_subsector_serie"
g17_sub,G.17: producción industrial de EE.UU. del subsector (NAICS 3 idéntico),US_PROD_01,por rama (mes-rama),mes + rama,"eeuu_ip_subsector, eeuu_ip_subsector_serie"
g17_rama,G.17: producción industrial de EE.UU. de la rama (NAICS 4 idéntico),US_PROD_01,por rama (mes-rama),mes + rama,"eeuu_ip_rama, eeuu_ip_rama_serie"
tpu,"TPU (Caldara, Iacoviello et al.)",INC_01,común por mes,mes,"tpu, tpu_share, tpu_raw, tpu_n7"
wui,"World Uncertainty Index (WUI, WTUI, WPUI)",INC_02,común por mes,mes,"wui_mex, wui_usa, wui_mundo_pib, wtui_mex, wtui_usa, wtui_mundo_pib, wpui_mex, wpui_usa, wpui_mundo_pib"


### Checkpoint 5: llaves y cobertura antes de integrar

In [28]:
titulo_checkpoint("CHECKPOINT 5 · Llaves y cobertura antes de integrar")
fuentes_llave = {"MX_PROD_01": emim_rama, "MX_EXP_01": expo_rama, "MX_ARAN_01": piezas.get("census", {}).get("datos")}
fuentes_llave = {k: fuentes_llave[k] for k in decisiones["universo_llaves"]}
llaves = (pd.concat([d[["mes", "rama"]] for d in fuentes_llave.values()]).drop_duplicates()
          .assign(mes=lambda d: a_mes(d.mes)).sort_values(["rama", "mes"]).reset_index(drop=True))
etiquetas = {"MX_PROD_01": "obs_emim", "MX_EXP_01": "obs_exportaciones", "MX_ARAN_01": "obs_census"}
for k, d in fuentes_llave.items():
    llaves[etiquetas[k]] = pd.MultiIndex.from_frame(llaves[["mes", "rama"]]).isin(pd.MultiIndex.from_frame(d[["mes", "rama"]].assign(mes=lambda z: a_mes(z.mes))))
meses_panel = set(llaves.mes)
filas = []
for k, p_ in piezas.items():
    d = p_["datos"]
    dup = n_duplicados(d, p_["llave"])
    registrar("CP5", p_["nombre"], f"Llave única ({' + '.join(p_['llave'])})", dup == 0, f"{dup} combinaciones duplicadas")
    filas.append(dict(pieza=k, fuente=p_["fuente_id"], tipo=p_["tipo"], observaciones=len(d),
                      meses=d.mes.nunique() if "mes" in d else None, ramas=d.rama.nunique() if "rama" in d else None,
                      periodo=rango_txt(d.mes) if "mes" in d else "no aplica", duplicados_llave=dup,
                      ramas_fuera_del_panel=", ".join(sorted(set(d.rama) - set(llaves.rama))) if "rama" in d else "",
                      ramas_del_panel_sin_dato=", ".join(sorted(set(llaves.rama) - set(d.rama))) if "rama" in d else "",
                      meses_del_panel_sin_dato=len(meses_panel - set(d.mes)) if "mes" in d else None,
                      meses_fuera_del_panel=len(set(d.mes) - meses_panel) if "mes" in d else None))
registrar("CP5", "Panel", "Universo de llaves mes-rama sin duplicados", not llaves.duplicated(["mes", "rama"]).any(),
          f"{len(llaves)} combinaciones; {llaves.rama.nunique()} ramas; {rango_txt(llaves.mes)}")
tabla(pd.DataFrame(filas), "Llaves y cobertura de cada pieza (frente al universo del panel)", ancho=120)
tabla(llaves.groupby("rama").agg(primer_mes=("mes", "min"), ultimo_mes=("mes", "max"), meses=("mes", "size"))
      .groupby(["primer_mes", "ultimo_mes", "meses"]).size().reset_index(name="ramas"), "Universo del panel: periodo por rama (el panel NO se balancea)")
aviso(f"<b>Universo del panel (D1):</b> unión de las combinaciones mes-rama de {', '.join(decisiones['universo_llaves'])}: "
      f"{len(llaves):,} filas, {llaves.rama.nunique()} ramas, {rango_txt(llaves.mes)}. Las variables comunes y las asignadas por "
      f"correspondencia solo se pegan sobre estas filas; sus meses fuera del rango quedan documentados en el checkpoint 6 y en la hoja 'cobertura'.")
cerrar_checkpoint("CP5", "llaves y cobertura")

**Llaves y cobertura de cada pieza (frente al universo del panel)**

pieza,fuente,tipo,observaciones,meses,ramas,periodo,duplicados_llave,ramas_fuera_del_panel,ramas_del_panel_sin_dato,meses_del_panel_sin_dato,meses_fuera_del_panel
atributos,MX_PROD_01,atributo de rama,86,NA,86,no aplica,0,,,NA,NA
exposicion,MIP_01,atributo de rama,86,NA,86,no aplica,0,,,NA,NA
emim,MX_PROD_01,por rama (mes-rama),8858,103,86,2018-01 a 2026-07,0,,,0,0
expo,MX_EXP_01,por rama (mes-rama),7826,91,86,2019-01 a 2026-07,0,,,12,0
census,MX_ARAN_01,por rama (mes-rama),"13,855",163,85,2013-01 a 2026-07,0,,3328,0,60
inpp,PRECIOS_01,por rama (mes-rama),"47,128",548,86,1981-01 a 2026-08,0,,,0,445
g17_sub,US_PROD_01,por rama (mes-rama),"111,112",1292,86,1919-01 a 2026-08,0,,,0,1189
g17_rama,US_PROD_01,por rama (mes-rama),"86,564",1292,67,1919-01 a 2026-08,0,,"3117, 3151, 3152, 3159, 3161, 3162, 3169, 3231, 3241, 3259, 3311, 3312, 3324, 3333, 3339, 3346, 3372, 3379, 3399",0,1189
tpu,INC_01,común por mes,800,800,NA,1960-01 a 2026-08,0,,,0,697
wui,INC_02,común por mes,224,224,NA,2008-01 a 2026-08,0,,,0,121


**Universo del panel: periodo por rama (el panel NO se balancea)**

primer_mes,ultimo_mes,meses,ramas
2018-01-01,2026-07-01,103,86


**Resumen del CP5 (llaves y cobertura):** 14 OK · 0 REVISAR · 0 ERROR

fuente,prueba,resultado,estado
Atributos de la rama (EMIM),Llave única (rama),0 combinaciones duplicadas,OK
Exposición: coeficiente de exportación MIP 2018 (D9),Llave única (rama),0 combinaciones duplicadas,OK
EMIM,Llave única (mes + rama),0 combinaciones duplicadas,OK
Exportaciones,Llave única (mes + rama),0 combinaciones duplicadas,OK
Census: importaciones de EE.UU. desde México y aranceles,Llave única (mes + rama),0 combinaciones duplicadas,OK
INPP del subsector de la rama,Llave única (mes + rama),0 combinaciones duplicadas,OK
G.17: producción industrial de EE.UU. del subsector (NAICS 3 idéntico),Llave única (mes + rama),0 combinaciones duplicadas,OK
G.17: producción industrial de EE.UU. de la rama (NAICS 4 idéntico),Llave única (mes + rama),0 combinaciones duplicadas,OK
"TPU (Caldara, Iacoviello et al.)",Llave única (mes),0 combinaciones duplicadas,OK
"World Uncertainty Index (WUI, WTUI, WPUI)",Llave única (mes),0 combinaciones duplicadas,OK


### Checkpoint 6: integración, unión por unión
Siempre uniones *left* sobre el universo de llaves con validación de cardinalidad (`validate`), que **aborta** si una unión fuera a multiplicar filas. Nunca se usa una unión interna sobre la base.

In [29]:
titulo_checkpoint("CHECKPOINT 6 · Integración de las fuentes, unión por unión")
def integrar(base, p_, orden):
    vars_ = [c for c in p_["datos"].columns if c not in p_["llave"]]
    exigir(not set(vars_) & set(base.columns), f"Integración de '{p_['nombre']}': variables ya existentes: {set(vars_) & set(base.columns)}")
    rel = "one_to_one" if p_["llave"] == ["mes", "rama"] else "many_to_one"
    n0, k0 = base.shape
    nueva = base.merge(p_["datos"], on=p_["llave"], how="left", validate=rel)
    claves_base = base[p_["llave"]].drop_duplicates()
    m = p_["datos"][p_["llave"]].merge(claves_base, on=p_["llave"], how="left", indicator=True)
    sin_corr = p_["datos"][(m["_merge"] == "left_only").to_numpy()]
    m2 = claves_base.merge(p_["datos"][p_["llave"]], on=p_["llave"], how="left", indicator=True)
    registrar("CP6", p_["nombre"], "La unión no multiplica ni elimina filas", len(nueva) == n0, f"{n0} filas antes, {len(nueva)} después")
    REG["uniones"].append(dict(
        orden=orden, fuente_incorporada=p_["nombre"], fuente_id=p_["fuente_id"], llave=" + ".join(p_["llave"]), relacion=rel,
        filas_antes=n0, filas_despues=len(nueva), columnas_antes=k0, columnas_despues=nueva.shape[1],
        meses_resultantes=nueva.mes.nunique(), ramas_resultantes=nueva.rama.nunique(), variables_nuevas=", ".join(vars_),
        obs_fuente_sin_correspondencia=len(sin_corr),
        periodo_sin_correspondencia=rango_txt(sin_corr.mes) if "mes" in sin_corr else ", ".join(sin_corr.rama.unique()),
        llaves_del_panel_sin_dato_en_fuente=int((m2["_merge"] == "left_only").sum()),
        na_en_variables_nuevas="; ".join(f"{v}={int(nueva[v].isna().sum())}" for v in vars_)))
    return nueva

base = llaves.copy()
for i, (k, p_) in enumerate(piezas.items(), start=1):
    base = integrar(base, p_, i)

conversiones = pd.DataFrame([
    ("emim_valor_produccion_real", "emim_valor_produccion", "miles_de_pesos"),
    ("emim_valor_ventas_real", "emim_valor_ventas", "miles_de_pesos"),
    ("expo_valor_real", "expo_valor_pesos", "pesos"),
    ("eeuu_imp_mx_valor_real", "eeuu_imp_mx_valor_usd", "dolares"),
    ("eeuu_imp_mx_valor_gravable_real", "eeuu_imp_mx_valor_gravable_usd", "dolares"),
    ("eeuu_imp_mx_aranceles_real", "eeuu_imp_mx_aranceles_usd", "dolares"),
], columns=["variable", "origen", "moneda_origen"])
conversiones = conversiones[conversiones.origen.isin(base.columns)].reset_index(drop=True)
if not decisiones["valores_en_miles_de_pesos_reales"]:
    conversiones = conversiones.iloc[0:0]
FACTOR = {"miles_de_pesos": lambda b: 1.0, "pesos": lambda b: 1 / 1000, "dolares": lambda b: b.tc_fix / 1000}
for cv in conversiones.itertuples():
    base[cv.variable] = base[cv.origen] * FACTOR[cv.moneda_origen](base) / (base.inpp_subsector_indice / 100)
    insumos = base[cv.origen].notna() & base.inpp_subsector_indice.notna() & ((cv.moneda_origen != "dolares") | base.tc_fix.notna())
    registrar("CP6", "Conversión D4", f"{cv.variable}: existe exactamente donde hay valor original, INPP" + (" y tipo de cambio" if cv.moneda_origen == "dolares" else ""),
              (base[cv.variable].notna() == insumos).all(), f"{base[cv.variable].notna().sum()} valores calculados; {(~insumos).sum()} filas sin algún insumo")
formulas = {"miles_de_pesos": "{o} / (inpp_subsector_indice / 100)", "pesos": "({o} / 1000) / (inpp_subsector_indice / 100)",
            "dolares": "({o} × tc_fix / 1000) / (inpp_subsector_indice / 100)"}
tabla(conversiones.assign(formula=[formulas[m].format(o=o) for o, m in zip(conversiones.origen, conversiones.moneda_origen)]),
      "Variables en miles de pesos reales de julio de 2019 (D4)", ancho=150)
tabla(pd.DataFrame(REG["uniones"]), "Resumen de cada unión", ancho=200)
cerrar_checkpoint("CP6", "integración")

**Variables en miles de pesos reales de julio de 2019 (D4)**

variable,origen,moneda_origen,formula
emim_valor_produccion_real,emim_valor_produccion,miles_de_pesos,emim_valor_produccion / (inpp_subsector_indice / 100)
emim_valor_ventas_real,emim_valor_ventas,miles_de_pesos,emim_valor_ventas / (inpp_subsector_indice / 100)
expo_valor_real,expo_valor_pesos,pesos,(expo_valor_pesos / 1000) / (inpp_subsector_indice / 100)
eeuu_imp_mx_valor_real,eeuu_imp_mx_valor_usd,dolares,(eeuu_imp_mx_valor_usd × tc_fix / 1000) / (inpp_subsector_indice / 100)
eeuu_imp_mx_valor_gravable_real,eeuu_imp_mx_valor_gravable_usd,dolares,(eeuu_imp_mx_valor_gravable_usd × tc_fix / 1000) / (inpp_subsector_indice / 100)
eeuu_imp_mx_aranceles_real,eeuu_imp_mx_aranceles_usd,dolares,(eeuu_imp_mx_aranceles_usd × tc_fix / 1000) / (inpp_subsector_indice / 100)


**Resumen de cada unión**

orden,fuente_incorporada,fuente_id,llave,relacion,filas_antes,filas_despues,columnas_antes,columnas_despues,meses_resultantes,ramas_resultantes,variables_nuevas,obs_fuente_sin_correspondencia,periodo_sin_correspondencia,llaves_del_panel_sin_dato_en_fuente,na_en_variables_nuevas
1,Atributos de la rama (EMIM),MX_PROD_01,rama,many_to_one,8858,8858,4,7,103,86,"rama_nombre, subsector, subsector_nombre",0,,0,rama_nombre=0; subsector=0; subsector_nombre=0
2,Exposición: coeficiente de exportación MIP 2018 (D9),MIP_01,rama,many_to_one,8858,8858,7,8,103,86,mip_coef_exportacion_2018,0,,0,mip_coef_exportacion_2018=0
3,EMIM,MX_PROD_01,mes + rama,one_to_one,8858,8858,8,14,103,86,"emim_personal_ocupado, emim_horas_trabajadas, emim_valor_produccion, emim_valor_ventas, emim_celdas_nd, emim_estatus_cifra",0,sin datos,0,emim_personal_ocupado=0; emim_horas_trabajadas=0; emim_valor_produccion=24; emim_valor_ventas=24; emim_celdas_nd=8834; emim_estatus_cifra=0
4,Exportaciones,MX_EXP_01,mes + rama,one_to_one,8858,8858,14,17,103,86,"expo_valor_pesos, expo_valor_usd, expo_n_clases",0,sin datos,1032,expo_valor_pesos=1032; expo_valor_usd=1032; expo_n_clases=1032
5,Census: importaciones de EE.UU. desde México y aranceles,MX_ARAN_01,mes + rama,one_to_one,8858,8858,17,22,103,86,"eeuu_imp_mx_valor_usd, eeuu_imp_mx_valor_gravable_usd, eeuu_imp_mx_aranceles_usd, eeuu_imp_mx_tasa_efectiva, eeuu_imp_mx_tasa_gravable",5100,2013-01 a 2017-12,103,eeuu_imp_mx_valor_usd=103; eeuu_imp_mx_valor_gravable_usd=103; eeuu_imp_mx_aranceles_usd=103; eeuu_imp_mx_tasa_efectiva=103; eeuu_imp_mx_tasa_gravable=1124
6,INPP del subsector de la rama,PRECIOS_01,mes + rama,one_to_one,8858,8858,22,24,103,86,"inpp_subsector_indice, inpp_subsector_serie","38,270",1981-01 a 2026-08,0,inpp_subsector_indice=0; inpp_subsector_serie=0
7,G.17: producción industrial de EE.UU. del subsector (NAICS 3 idéntico),US_PROD_01,mes + rama,one_to_one,8858,8858,24,26,103,86,"eeuu_ip_subsector, eeuu_ip_subsector_serie","102,254",1919-01 a 2026-08,0,eeuu_ip_subsector=0; eeuu_ip_subsector_serie=0
8,G.17: producción industrial de EE.UU. de la rama (NAICS 4 idéntico),US_PROD_01,mes + rama,one_to_one,8858,8858,26,28,103,86,"eeuu_ip_rama, eeuu_ip_rama_serie","79,663",1919-01 a 2026-08,1957,eeuu_ip_rama=1959; eeuu_ip_rama_serie=1957
9,"TPU (Caldara, Iacoviello et al.)",INC_01,mes,many_to_one,8858,8858,28,32,103,86,"tpu, tpu_share, tpu_raw, tpu_n7",697,1960-01 a 2026-08,0,tpu=0; tpu_share=0; tpu_raw=0; tpu_n7=0
10,"World Uncertainty Index (WUI, WTUI, WPUI)",INC_02,mes,many_to_one,8858,8858,32,41,103,86,"wui_mex, wui_usa, wui_mundo_pib, wtui_mex, wtui_usa, wtui_mundo_pib, wpui_mex, wpui_usa, wpui_mundo_pib",121,2008-01 a 2026-08,0,wui_mex=0; wui_usa=0; wui_mundo_pib=0; wtui_mex=0; wtui_usa=0; wtui_mundo_pib=0; wpui_mex=0; wpui_usa=0; wpui_mundo_pib=0


**Resumen del CP6 (integración):** 19 OK · 0 REVISAR · 0 ERROR

fuente,prueba,resultado,estado
Atributos de la rama (EMIM),La unión no multiplica ni elimina filas,"8858 filas antes, 8858 después",OK
Exposición: coeficiente de exportación MIP 2018 (D9),La unión no multiplica ni elimina filas,"8858 filas antes, 8858 después",OK
EMIM,La unión no multiplica ni elimina filas,"8858 filas antes, 8858 después",OK
Exportaciones,La unión no multiplica ni elimina filas,"8858 filas antes, 8858 después",OK
Census: importaciones de EE.UU. desde México y aranceles,La unión no multiplica ni elimina filas,"8858 filas antes, 8858 después",OK
INPP del subsector de la rama,La unión no multiplica ni elimina filas,"8858 filas antes, 8858 después",OK
G.17: producción industrial de EE.UU. del subsector (NAICS 3 idéntico),La unión no multiplica ni elimina filas,"8858 filas antes, 8858 después",OK
G.17: producción industrial de EE.UU. de la rama (NAICS 4 idéntico),La unión no multiplica ni elimina filas,"8858 filas antes, 8858 después",OK
"TPU (Caldara, Iacoviello et al.)",La unión no multiplica ni elimina filas,"8858 filas antes, 8858 después",OK
"World Uncertainty Index (WUI, WTUI, WPUI)",La unión no multiplica ni elimina filas,"8858 filas antes, 8858 después",OK


## 7. Incorporación de variables comunes por mes
Las variables comunes ya entraron como piezas con llave `mes`. Aquí se verifica que su valor sea **idéntico entre todas las ramas de un mismo mes**, y que los atributos de rama sean constantes dentro de cada rama.

In [30]:
titulo_checkpoint("CHECKPOINT 6B · Variables comunes por mes y atributos de rama")
filas = []
for k, p_ in piezas.items():
    if p_["llave"] == ["mes", "rama"]:
        continue
    for v in [c for c in p_["datos"].columns if c not in p_["llave"]]:
        n = base.groupby(p_["llave"])[v].nunique(dropna=False)
        registrar("CP6B", p_["nombre"], f"{v}: un solo valor por {p_['llave'][0]}", (n == 1).all(), f"{(n > 1).sum()} grupos con más de un valor")
        filas.append(dict(variable=v, fuente=p_["fuente_id"], constante_por=p_["llave"][0], grupos=len(n), grupos_con_varios_valores=int((n > 1).sum())))
tabla(pd.DataFrame(filas), "Constancia de variables comunes (por mes) y atributos (por rama)", max_filas=100)
cerrar_checkpoint("CP6B", "variables comunes")

**Constancia de variables comunes (por mes) y atributos (por rama)**

variable,fuente,constante_por,grupos,grupos_con_varios_valores
rama_nombre,MX_PROD_01,rama,86,0
subsector,MX_PROD_01,rama,86,0
subsector_nombre,MX_PROD_01,rama,86,0
mip_coef_exportacion_2018,MIP_01,rama,86,0
tpu,INC_01,mes,103,0
tpu_share,INC_01,mes,103,0
tpu_raw,INC_01,mes,103,0
tpu_n7,INC_01,mes,103,0
wui_mex,INC_02,mes,103,0
wui_usa,INC_02,mes,103,0


**Resumen del CP6B (variables comunes):** 21 OK · 0 REVISAR · 0 ERROR

fuente,prueba,resultado,estado
Atributos de la rama (EMIM),rama_nombre: un solo valor por rama,0 grupos con más de un valor,OK
Atributos de la rama (EMIM),subsector: un solo valor por rama,0 grupos con más de un valor,OK
Atributos de la rama (EMIM),subsector_nombre: un solo valor por rama,0 grupos con más de un valor,OK
Exposición: coeficiente de exportación MIP 2018 (D9),mip_coef_exportacion_2018: un solo valor por rama,0 grupos con más de un valor,OK
"TPU (Caldara, Iacoviello et al.)",tpu: un solo valor por mes,0 grupos con más de un valor,OK
"TPU (Caldara, Iacoviello et al.)",tpu_share: un solo valor por mes,0 grupos con más de un valor,OK
"TPU (Caldara, Iacoviello et al.)",tpu_raw: un solo valor por mes,0 grupos con más de un valor,OK
"TPU (Caldara, Iacoviello et al.)",tpu_n7: un solo valor por mes,0 grupos con más de un valor,OK
"World Uncertainty Index (WUI, WTUI, WPUI)",wui_mex: un solo valor por mes,0 grupos con más de un valor,OK
"World Uncertainty Index (WUI, WTUI, WPUI)",wui_usa: un solo valor por mes,0 grupos con más de un valor,OK


## 8. Construcción de la base final
### Checkpoint 7: base integrada preliminar

In [31]:
titulo_checkpoint("CHECKPOINT 7 · Base integrada preliminar")
base["periodo"] = base.mes.dt.strftime("%Y-%m")
pref = lambda r: [c for c in base.columns if re.match(r, c)]
orden_cols = (["mes", "periodo", "rama", "rama_nombre", "subsector", "subsector_nombre"] + pref(r"^mip_") + pref(r"^obs_") + pref(r"^emim_") + pref(r"^inpp_")
              + pref(r"^expo_") + pref(r"^eeuu_imp_mx_") + pref(r"^eeuu_ip_") + pref(r"^tpu") + pref(r"^(wui|wtui|wpui)_")
              + pref(r"^epu_") + pref(r"^tc_") + pref(r"^vix_"))
exigir(sorted(orden_cols) == sorted(base.columns) and len(set(orden_cols)) == len(orden_cols),
       "Columnas sin lugar en el orden final: ", set(base.columns) - set(orden_cols))
base = base[orden_cols].sort_values(["rama", "mes"]).reset_index(drop=True)

R = lambda ids: " + ".join(ids)
dicc = [
    ("mes", "Mes de referencia (primer día del mes)", "fecha", "mensual", "identificador", "—", "construida", "—", "Fechas homologadas de todas las fuentes"),
    ("periodo", "Mes en texto AAAA-MM", "texto", "mensual", "identificador", "—", "construida", "—", "mes en formato AAAA-MM"),
    ("rama", "Rama SCIAN 2018 de 4 dígitos (texto)", "código", "—", "identificador", "México", "MX_PROD_01", "Tabulado: Sector SCIAN (código)", "Código separado del nombre"),
    ("rama_nombre", "Nombre de la rama según la EMIM", "texto", "—", "atributo de rama", "México", "MX_PROD_01", "Tabulado: Sector SCIAN (nombre)", "—"),
    ("subsector", "Subsector SCIAN (3 primeros dígitos de la rama)", "código", "—", "atributo de rama", "México", "MX_PROD_01", "—", "Primeros 3 dígitos de rama"),
    ("subsector_nombre", "Nombre del subsector según la EMIM", "texto", "—", "atributo de rama", "México", "MX_PROD_01", "Tabulado: filas de subsector", "—"),
    ("mip_coef_exportacion_2018", "Coeficiente de exportación de la rama en 2018 (exportaciones / producción bruta), medida de exposición (D9)", "proporción (0-1)", "anual (2018), fijo en el tiempo", "atributo de rama", "México (todos los destinos)", "MIP_01", "TABLA: Export", "Ninguna"),
    ("obs_emim", "TRUE si la combinación mes-rama existe en la EMIM", "lógico", "—", "mes-rama", "—", "MX_PROD_01", "—", "Indicador de cobertura"),
    ("obs_exportaciones", "TRUE si la combinación mes-rama existe en las exportaciones", "lógico", "—", "mes-rama", "—", "MX_EXP_01", "—", "Indicador de cobertura"),
    ("obs_census", "TRUE si la combinación mes-rama existe en el Census", "lógico", "—", "mes-rama", "—", "MX_ARAN_01", "—", "Indicador de cobertura"),
    ("emim_personal_ocupado", "Personal ocupado total", "número de personas", "mensual", "mes-rama", "México", "MX_PROD_01", dicc_emim.variable_original[0], "Ninguna"),
    ("emim_horas_trabajadas", "Horas trabajadas por el personal ocupado total", "miles de horas", "mensual", "mes-rama", "México", "MX_PROD_01", dicc_emim.variable_original[1], "Ninguna"),
    ("emim_valor_produccion", "Valor de producción de los productos elaborados (nominal, sin ajuste estacional)", "miles de pesos corrientes", "mensual", "mes-rama", "México", "MX_PROD_01", dicc_emim.variable_original[2], "ND -> NA"),
    ("emim_valor_ventas", "Valor de ventas de los productos elaborados (nominal)", "miles de pesos corrientes", "mensual", "mes-rama", "México", "MX_PROD_01", dicc_emim.variable_original[3], "ND -> NA"),
    ("emim_celdas_nd", "Variables de la EMIM confidenciales (ND) en esa rama y mes", "texto", "mensual", "mes-rama", "México", "MX_PROD_01", "celdas 'ND'", "Motivo del NA"),
    ("emim_estatus_cifra", "Estatus de la cifra según las notas de INEGI", "texto (definitiva / revisada / preliminar)", "mensual", "mes-rama", "México", "MX_PROD_01", "notas al pie", "Clasificado por fecha"),
    ("emim_valor_produccion_real", "Valor de producción real", "miles de pesos de julio de 2019", "mensual", "mes-rama", "México", R(["MX_PROD_01", "PRECIOS_01"]), "—", "emim_valor_produccion / (inpp_subsector_indice / 100) (D4)"),
    ("emim_valor_ventas_real", "Valor de ventas real", "miles de pesos de julio de 2019", "mensual", "mes-rama", "México", R(["MX_PROD_01", "PRECIOS_01"]), "—", "emim_valor_ventas / (inpp_subsector_indice / 100) (D4)"),
    ("inpp_subsector_indice", "INPP de mercancías y servicios finales por origen, subsector de la rama", "índice, base efectiva jul-2019=100", "mensual", "mes-rama (asignado por subsector)", "México", "PRECIOS_01", "serie según docs/correspondencia_rama_inpp.csv", "Asignación rama -> subsector"),
    ("inpp_subsector_serie", "Clave INEGI de la serie INPP asignada", "código", "—", "mes-rama", "México", R(["PRECIOS_01", "DOC_CORR"]), "fila 'Fecha'", "—"),
    ("expo_valor_pesos", "Exportaciones de la rama (suma de clases SCIAN de 6 dígitos)", "pesos corrientes", "mensual", "mes-rama", "México (todos los destinos)", "MX_EXP_01", "Datos: Valor en pesos", "Suma por rama"),
    ("expo_valor_usd", "Exportaciones de la rama (suma de clases SCIAN de 6 dígitos)", "dólares corrientes", "mensual", "mes-rama", "México (todos los destinos)", "MX_EXP_01", "Datos: Valor en dólares", "Suma por rama"),
    ("expo_n_clases", "Número de clases SCIAN con registro de exportación en el mes", "número", "mensual", "mes-rama", "México", "MX_EXP_01", "Datos: SCIAN", "Conteo por rama"),
    ("expo_valor_real", "Exportaciones de la rama, reales", "miles de pesos de julio de 2019", "mensual", "mes-rama", "México (todos los destinos)", R(["MX_EXP_01", "PRECIOS_01"]), "—", "(expo_valor_pesos / 1000) / (inpp_subsector_indice / 100) (D4)"),
    ("eeuu_imp_mx_valor_usd", "Importaciones de EE.UU. desde México para consumo (CON_VAL_MO)", "dólares corrientes", "mensual", "mes-rama (NAICS 4 = SCIAN 4)", "EE.UU. <- México", "MX_ARAN_01", "valor_importado", "Ninguna (D2)"),
    ("eeuu_imp_mx_valor_gravable_usd", "Valor gravable de esas importaciones (DUT_VAL_MO)", "dólares corrientes", "mensual", "mes-rama (NAICS 4 = SCIAN 4)", "EE.UU. <- México", "MX_ARAN_01", "valor_gravable", "Ninguna (D2)"),
    ("eeuu_imp_mx_aranceles_usd", "Aranceles calculados (CAL_DUT_MO)", "dólares corrientes", "mensual", "mes-rama (NAICS 4 = SCIAN 4)", "EE.UU. <- México", "MX_ARAN_01", "aranceles", "Ninguna (D2)"),
    ("eeuu_imp_mx_tasa_efectiva", "Tasa arancelaria efectiva: aranceles / valor importado", "proporción", "mensual", "mes-rama (NAICS 4 = SCIAN 4)", "EE.UU. <- México", "MX_ARAN_01", "tasa_efectiva", "Ninguna (D2); NA si no hubo importaciones"),
    ("eeuu_imp_mx_tasa_gravable", "Tasa sobre valor gravable: aranceles / valor gravable", "proporción", "mensual", "mes-rama (NAICS 4 = SCIAN 4)", "EE.UU. <- México", "MX_ARAN_01", "tasa_gravable", "Ninguna (D2); NA si valor gravable = 0"),
    ("eeuu_imp_mx_valor_real", "Importaciones de EE.UU. desde México, reales", "miles de pesos de julio de 2019", "mensual", "mes-rama (NAICS 4 = SCIAN 4)", "EE.UU. <- México", R(["MX_ARAN_01", "MACRO_01", "PRECIOS_01"]), "—", "(eeuu_imp_mx_valor_usd × tc_fix / 1000) / (inpp_subsector_indice / 100) (D4)"),
    ("eeuu_imp_mx_valor_gravable_real", "Valor gravable de esas importaciones, real", "miles de pesos de julio de 2019", "mensual", "mes-rama (NAICS 4 = SCIAN 4)", "EE.UU. <- México", R(["MX_ARAN_01", "MACRO_01", "PRECIOS_01"]), "—", "(eeuu_imp_mx_valor_gravable_usd × tc_fix / 1000) / (inpp_subsector_indice / 100) (D4)"),
    ("eeuu_imp_mx_aranceles_real", "Aranceles calculados, reales", "miles de pesos de julio de 2019", "mensual", "mes-rama (NAICS 4 = SCIAN 4)", "EE.UU. <- México", R(["MX_ARAN_01", "MACRO_01", "PRECIOS_01"]), "—", "(eeuu_imp_mx_aranceles_usd × tc_fix / 1000) / (inpp_subsector_indice / 100) (D4)"),
    ("eeuu_ip_subsector", "Producción industrial de EE.UU. del subsector NAICS de 3 dígitos idéntico al de la rama", "índice (2017=100, por confirmar), SA", "mensual", "mes-rama (asignado por subsector)", "EE.UU.", "US_PROD_01", "columna de la serie G.17", "Asignación por código verificado (D3a)"),
    ("eeuu_ip_subsector_serie", "Serie G.17 asignada (subsector)", "código", "—", "mes-rama", "EE.UU.", "US_PROD_02", "serie", "—"),
    ("eeuu_ip_rama", "Producción industrial de EE.UU. de la industria NAICS de 4 dígitos idéntica a la rama", "índice (2017=100, por confirmar), SA", "mensual", "mes-rama (asignado por código)", "EE.UU.", "US_PROD_01", "columna de la serie G.17", "Asignación por código verificado (D3b)"),
    ("eeuu_ip_rama_serie", "Serie G.17 asignada (rama)", "código", "—", "mes-rama", "EE.UU.", "US_PROD_02", "serie", "—"),
    ("tpu", "Índice de incertidumbre de política comercial (TPU)", "índice (100 = 1% de artículos)", "mensual", "común por mes", "EE.UU. (prensa)", "INC_01", "TPU_MONTHLY: TPU", "Ninguna"),
    ("tpu_share", "Proporción de artículos sobre TPU", "proporción", "mensual", "común por mes", "EE.UU. (prensa)", "INC_01", "TPU_MONTHLY: TPU_SHARE", "Ninguna"),
    ("tpu_raw", "Número de artículos sobre TPU", "artículos", "mensual", "común por mes", "EE.UU. (prensa)", "INC_01", "TPU_MONTHLY: TPU_RAW", "Ninguna"),
    ("tpu_n7", "Total de artículos en los 7 periódicos", "artículos", "mensual", "común por mes", "EE.UU. (prensa)", "INC_01", "TPU_MONTHLY: N7", "Ninguna"),
    ("epu_mex", "Índice de incertidumbre de política económica para México (EPU)", "índice", "mensual", "común por mes", "México", "INC_03", "Mexican Policy Uncertainty Index", "Ninguna"),
    ("tc_fix", "Tipo de cambio FIX, promedio mensual (SF17908)", "pesos por dólar", "mensual", "común por mes", "México", "MACRO_01", "SF17908", "Ceros de 1991 -> NA (D5)"),
    ("vix_promedio_mensual", "VIX: promedio de los días con dato en el mes", "puntos de índice", "diaria", "común por mes", "EE.UU.", "MACRO_02", "VIXCLS", "Promedio mensual (D6)"),
    ("vix_ultimo_dato_del_mes", "VIX: último día con dato en el mes", "puntos de índice", "diaria", "común por mes", "EE.UU.", "MACRO_02", "VIXCLS", "Último dato del mes (D6)"),
    ("vix_dias_con_dato", "Días con dato del VIX en el mes", "días", "diaria", "común por mes", "EE.UU.", "MACRO_02", "VIXCLS", "Conteo (D6)"),
]
nombres_ind = {"wui": "World Uncertainty Index (WUI, general)", "wtui": "World Trade Uncertainty Index (WTUI, comercial)",
               "wpui": "World Policy Uncertainty Index (WPUI, política)"}
hojas_ind, hojas_f = {"wui": "T1", "wtui": "T6", "wpui": "T7"}, {"wui": "F1", "wtui": "F2", "wpui": "F3"}
for v in pref(r"^(wui|wtui|wpui)_"):
    ind, suf = v.split("_", 1)
    mundo = suf == "mundo_pib"
    dicc.append((v, nombres_ind[ind] + (", promedio mundial ponderado por PIB" if mundo else f", {suf.upper()}"),
                 "índice (escala del promedio ponderado)" if mundo else "palabras / total de palabras × 1,000",
                 "mensual", "común por mes", "Mundo (71 países)" if mundo else suf.upper(), "INC_02",
                 f"Hoja {hojas_f[ind]}, columna B" if mundo else f"Hoja {hojas_ind[ind]}, columna {suf.upper()}", "Ninguna (D7)"))
diccionario = pd.DataFrame(dicc, columns=["variable", "descripcion", "unidad", "frecuencia_original", "nivel_en_base",
                                          "cobertura_geografica", "fuente_id", "hoja_o_columna_original", "transformacion"])
diccionario = diccionario[diccionario.variable.isin(base.columns)].copy()
diccionario["archivo_original"] = diccionario.fuente_id.map(lambda f: " + ".join(fuentes.ruta[fuentes.id.isin(f.split(" + "))]))
diccionario = diccionario.set_index("variable").loc[list(base.columns)].reset_index()

id_cols = ["mes", "periodo", "rama", "rama_nombre", "subsector", "subsector_nombre", "mip_coef_exportacion_2018"]
vars_datos = [c for c in base.columns if c not in id_cols and not c.startswith("obs_")]
display(Markdown(f"- **Dimensiones:** {base.shape[0]:,} filas × {base.shape[1]} columnas\n- **Meses:** {base.mes.nunique()} ({rango_txt(base.mes)})\n"
                 f"- **Ramas:** {base.rama.nunique()}\n- **Variables de datos:** {len(vars_datos)}"))
tabla(pd.DataFrame({"columna": base.columns, "tipo": [str(t) for t in base.dtypes]}).merge(
      diccionario[["variable", "unidad", "fuente_id"]], left_on="columna", right_on="variable", how="left").drop(columns="variable"),
      "Variables de la base", max_filas=200)
tabla(base.groupby("rama").size().value_counts().rename_axis("observaciones_por_rama").reset_index(name="ramas"), "Número de observaciones por rama (distribución)")
cobertura_var = pd.DataFrame([dict(variable=v, primer_mes=base.mes[base[v].notna()].min(), ultimo_mes=base.mes[base[v].notna()].max(),
                                   ramas_con_dato=base.rama[base[v].notna()].nunique(), no_faltantes=int(base[v].notna().sum()),
                                   faltantes=int(base[v].isna().sum()), pct_faltantes=round(100 * base[v].isna().mean(), 2)) for v in vars_datos])
tabla(cobertura_var, "Cobertura temporal y faltantes por variable en la base", max_filas=200)
inc = base.assign(n_falt=base[vars_datos].isna().sum(axis=1)).groupby("periodo").agg(filas=("rama", "size"),
      filas_incompletas=("n_falt", lambda s: int((s > 0).sum())), faltantes_promedio=("n_falt", "mean")).reset_index()
tabla(inc[inc.filas_incompletas > 0], "Meses con filas incompletas (resumen)", max_filas=30)
registrar("CP7", "Base", "La base no se restringió al periodo común ni se balanceó (filas = universo de llaves; rango = el de las fuentes del universo)",
          len(base) == len(llaves) and base.mes.min() == min(d.mes.min() for d in fuentes_llave.values())
          and base.mes.max() == max(d.mes.max() for d in fuentes_llave.values()),
          f"{len(base):,} filas; {base.rama.nunique()} ramas; {rango_txt(base.mes)}; ninguna fila se eliminó por faltantes")
registrar("CP7", "Base", "Todas las columnas tienen entrada en el diccionario", set(diccionario.variable) == set(base.columns), "")
cerrar_checkpoint("CP7", "base integrada preliminar")

- **Dimensiones:** 8,858 filas × 52 columnas
- **Meses:** 103 (2018-01 a 2026-07)
- **Ramas:** 86
- **Variables de datos:** 43

**Variables de la base**

columna,tipo,unidad,fuente_id
mes,datetime64[ns],fecha,construida
periodo,str,texto,construida
rama,str,código,MX_PROD_01
rama_nombre,str,texto,MX_PROD_01
subsector,str,código,MX_PROD_01
subsector_nombre,str,texto,MX_PROD_01
mip_coef_exportacion_2018,float64,proporción (0-1),MIP_01
obs_emim,bool,lógico,MX_PROD_01
obs_exportaciones,bool,lógico,MX_EXP_01
emim_personal_ocupado,float64,número de personas,MX_PROD_01


**Número de observaciones por rama (distribución)**

observaciones_por_rama,ramas
103,86


...

**Meses con filas incompletas (resumen)**

periodo,filas,filas_incompletas,faltantes_promedio
2018-01,86,86,5.68605
2018-02,86,86,5.68605
2018-03,86,86,5.68605
2018-04,86,86,5.67442
2018-05,86,86,5.69767
2018-06,86,86,5.67442
2018-07,86,86,5.67442
2018-08,86,86,5.67442
2018-09,86,86,5.68605
2018-10,86,86,5.66279


_Se muestran 30 de 103 filas._

**Resumen del CP7 (base integrada preliminar):** 2 OK · 0 REVISAR · 0 ERROR

fuente,prueba,resultado,estado
Base,La base no se restringió al periodo común ni se balanceó (filas = universo de llaves; rango = el de las fuentes del universo),"8,858 filas; 86 ramas; 2018-01 a 2026-07; ninguna fila se eliminó por faltantes",OK
Base,Todas las columnas tienen entrada en el diccionario,,OK


## 9. Validaciones de la base integrada
### Checkpoint 8: pruebas finales antes de exportar
Doce pruebas explícitas (la 12 se ejecuta al escribir el Excel). Los errores críticos detienen la exportación.

In [32]:
titulo_checkpoint("CHECKPOINT 8 · Validaciones finales")
registrar("CP8", "1. Llave", "La combinación mes-rama es única", not base.duplicated(["mes", "rama"]).any(), f"{base.duplicated(['mes', 'rama']).sum()} duplicados")
registrar("CP8", "2. Fechas", "mes es fecha, sin NA y siempre día 1",
          pd.api.types.is_datetime64_any_dtype(base.mes) and base.mes.notna().all() and (base.mes.dt.day == 1).all(), str(base.mes.dtype))
registrar("CP8", "2. Fechas", "periodo = mes en formato AAAA-MM", (base.periodo == base.mes.dt.strftime("%Y-%m")).all(), "")
registrar("CP8", "3. Ramas", "rama es texto de 4 dígitos", base.rama.map(lambda r: isinstance(r, str) and bool(re.fullmatch(r"\d{4}", r))).all(), "")
registrar("CP8", "3. Ramas", "Toda rama tiene nombre EMIM", base.rama_nombre.notna().all(), "", si_falla="REVISAR")
registrar("CP8", "3. Ramas", "subsector = primeros 3 dígitos de la rama", (base.subsector == base.rama.str[:3]).all(), "")

filas = []
for k, p_ in piezas.items():
    vars_ = [c for c in p_["datos"].columns if c not in p_["llave"]]
    esencial = base[p_["llave"] + vars_].drop_duplicates()
    cmp = esencial.merge(p_["datos"], on=p_["llave"], how="inner", suffixes=(".base", ".fuente"), validate="many_to_one")
    for v in vars_:
        fuente_nn = p_["datos"][p_["datos"][v].notna()]
        if p_["llave"] == ["mes", "rama"]:
            dentro = fuente_nn.merge(base[["mes", "rama"]], on=["mes", "rama"], how="inner")
            esperado = len(dentro)
            m_fuente = dentro.mes
        else:
            cuenta = base.groupby(p_["llave"]).size().rename("n_filas").reset_index()
            esperado = int(cuenta.merge(fuente_nn[p_["llave"]], on=p_["llave"]).n_filas.sum())
            m_fuente = fuente_nn.mes[fuente_nn.mes.isin(base.mes)] if "mes" in p_["llave"] else None
        filas.append(dict(pieza=k, fuente=p_["fuente_id"], cobertura_geografica=p_["cobertura_geografica"], variable=v,
                          valores_identicos_a_fuente=iguales(cmp[f"{v}.base"], cmp[f"{v}.fuente"]),
                          no_faltantes_base=int(base[v].notna().sum()), no_faltantes_esperados=esperado,
                          rango_en_base=rango_txt(base.mes[base[v].notna()]) if m_fuente is not None else "no aplica",
                          rango_fuente_dentro_del_panel=rango_txt(m_fuente) if m_fuente is not None else "no aplica",
                          rango_fuente_completo=rango_txt(fuente_nn.mes) if "mes" in fuente_nn else "no aplica"))
verif = pd.DataFrame(filas)
registrar("CP8", "4. México vs. EE.UU.", "Cada variable conserva exactamente los valores de su fuente (sin mezclar fuentes ni países)",
          verif.valores_identicos_a_fuente.all(), "Con diferencias: " + ", ".join(verif.variable[~verif.valores_identicos_a_fuente]))
pref_mx, pref_us = pref(r"^(emim_|expo_|inpp_)"), pref(r"^eeuu_ip_")
fid = diccionario.set_index("variable").fuente_id
registrar("CP8", "4. México vs. EE.UU.", "emim_/expo_/inpp_ provienen solo de fuentes mexicanas y eeuu_ip_ solo del G.17",
          not fid[pref_mx].str.contains("US_PROD").any() and fid[pref_us].isin(["US_PROD_01", "US_PROD_02"]).all(), "Según el diccionario")
registrar("CP8", "4. México vs. EE.UU.", "La producción mexicana (emim_valor_produccion) y la de EE.UU. (eeuu_ip_*) son columnas distintas",
          not set(pref_us) & set(pref_mx) and "emim_valor_produccion" in base, "")
for cv in conversiones.itertuples():
    esperado = base[cv.origen] * FACTOR[cv.moneda_origen](base) * 100 / base.inpp_subsector_indice
    registrar("CP8", "4b. Conversión D4", f"{cv.variable} se reproduce desde su variable original, el FIX y el INPP",
              np.allclose(base[cv.variable], esperado, rtol=1e-10, equal_nan=True), "")
comunes = [c for p_ in piezas.values() if p_["llave"] == ["mes"] for c in p_["datos"].columns if c != "mes"]
no_const = [v for v in comunes if (base.groupby("mes")[v].nunique(dropna=False) > 1).any()]
registrar("CP8", "5. Comunes", "Las variables comunes tienen el mismo valor en todas las ramas de cada mes", not no_const,
          ("No constantes: " + ", ".join(no_const)) if no_const else f"{len(comunes)} variables comunes")
un = pd.DataFrame(REG["uniones"])
registrar("CP8", "6. Filas", "Ninguna unión multiplicó filas y la base tiene tantas filas como el universo de llaves",
          (un.filas_antes == un.filas_despues).all() and len(base) == len(llaves), f"Universo: {len(llaves):,}; base: {len(base):,}")
faltantes_var = faltantes_por_variable(base, vars_datos)
registrar("CP8", "7. Faltantes", "Los faltantes están cuantificados para todas las variables",
          faltantes_var.n_faltantes.sum() == base[vars_datos].isna().sum().sum(), f"{faltantes_var.n_faltantes.sum():,} celdas faltantes en {len(vars_datos)} variables")
difr = verif[verif.rango_en_base != verif.rango_fuente_dentro_del_panel]
registrar("CP8", "8. Cobertura", "El rango de cada variable en la base coincide con el de su fuente (dentro del panel)", difr.empty,
          "Con diferencias: " + ", ".join(difr.variable))
difn = verif[verif.no_faltantes_base != verif.no_faltantes_esperados]
registrar("CP8", "9. Sin imputación", "Los valores no faltantes son exactamente los que aporta la fuente (ni más: imputación; ni menos: pérdida)",
          difn.empty, "Con diferencias: " + ", ".join(difn.variable))
nb_path = RAIZ / "code" / "Python" / "01_preparacion_base_integrada.ipynb"
codigo = "\n".join("".join(c["source"]) for c in json.loads(nb_path.read_text(encoding="utf-8"))["cells"] if c["cell_type"] == "code")
vetadas = ["interpolate(", "bfill(", "ffill(", "method='pad'", 'method="pad"', "method='ffill'", 'method="ffill"']
usos = [(f, i + 1) for i, l in enumerate(codigo.splitlines()) for f in vetadas if f in l
        and "vetadas" not in l and 'td_d["anio"] = td_d["anio"].ffill()' not in l]
registrar("CP8", "9. Sin imputación", "El código no usa funciones de interpolación, arrastre o imputación sobre los datos", not usos,
          ("Revise: " + ", ".join(f"{f} (línea {n})" for f, n in usos)) if usos
          else "Se revisó el propio notebook: el único arrastre es el del año en la tabla dinámica TD (solo verificación), nunca sobre la base.")
registrar("CP8", "10. Diccionario", "Toda variable tiene descripción, unidad y fuente en el diccionario",
          set(diccionario.variable) == set(base.columns) and diccionario.unidad.notna().all() and diccionario.fuente_id.notna().all(), "")
revisar("CP8", "10. Diccionario", "Unidades por confirmar con la fuente",
        ", ".join(diccionario.variable[diccionario.unidad.str.contains("confirmar|efectiva")]))
md5_fin = {f.id: hashlib.md5((RAIZ / f.ruta).read_bytes()).hexdigest() for f in fuentes.itertuples()}
registrar("CP8", "11. Reproducibilidad", "Los archivos originales no se modificaron durante la ejecución", md5_inicio == md5_fin, "Huellas MD5 al inicio y al final")
tabla(verif, "Verificación de cada variable contra su fuente limpia", max_filas=200, ancho=60)
cerrar_checkpoint("CP8", "validaciones finales")

**Verificación de cada variable contra su fuente limpia**

pieza,fuente,cobertura_geografica,variable,valores_identicos_a_fuente,no_faltantes_base,no_faltantes_esperados,rango_en_base,rango_fuente_dentro_del_panel,rango_fuente_completo
atributos,MX_PROD_01,México,rama_nombre,TRUE,8858,8858,no aplica,no aplica,no aplica
atributos,MX_PROD_01,México,subsector,TRUE,8858,8858,no aplica,no aplica,no aplica
atributos,MX_PROD_01,México,subsector_nombre,TRUE,8858,8858,no aplica,no aplica,no aplica
exposicion,MIP_01,México (todos los destinos),mip_coef_exportacion_2018,TRUE,8858,8858,no aplica,no aplica,no aplica
emim,MX_PROD_01,México,emim_personal_ocupado,TRUE,8858,8858,2018-01 a 2026-07,2018-01 a 2026-07,2018-01 a 2026-07
emim,MX_PROD_01,México,emim_horas_trabajadas,TRUE,8858,8858,2018-01 a 2026-07,2018-01 a 2026-07,2018-01 a 2026-07
emim,MX_PROD_01,México,emim_valor_produccion,TRUE,8834,8834,2018-01 a 2026-07,2018-01 a 2026-07,2018-01 a 2026-07
emim,MX_PROD_01,México,emim_valor_ventas,TRUE,8834,8834,2018-01 a 2026-07,2018-01 a 2026-07,2018-01 a 2026-07
emim,MX_PROD_01,México,emim_celdas_nd,TRUE,24,24,2021-08 a 2024-04,2021-08 a 2024-04,2021-08 a 2024-04
emim,MX_PROD_01,México,emim_estatus_cifra,TRUE,8858,8858,2018-01 a 2026-07,2018-01 a 2026-07,2018-01 a 2026-07


**Resumen del CP8 (validaciones finales):** 23 OK · 1 REVISAR · 0 ERROR

fuente,prueba,resultado,estado
1. Llave,La combinación mes-rama es única,0 duplicados,OK
2. Fechas,"mes es fecha, sin NA y siempre día 1",datetime64[ns],OK
2. Fechas,periodo = mes en formato AAAA-MM,,OK
3. Ramas,rama es texto de 4 dígitos,,OK
3. Ramas,Toda rama tiene nombre EMIM,,OK
3. Ramas,subsector = primeros 3 dígitos de la rama,,OK
4. México vs. EE.UU.,Cada variable conserva exactamente los valores de su fuente (sin mezclar fuentes ni países),Con diferencias:,OK
4. México vs. EE.UU.,emim_/expo_/inpp_ provienen solo de fuentes mexicanas y eeuu_ip_ solo del G.17,Según el diccionario,OK
4. México vs. EE.UU.,La producción mexicana (emim_valor_produccion) y la de EE.UU. (eeuu_ip_*) son columnas distintas,,OK
4b. Conversión D4,"emim_valor_produccion_real se reproduce desde su variable original, el FIX y el INPP",,OK


## 10. Exportación a Excel
### Checkpoint 9: escritura y verificación del archivo
Hoja principal `base_integrada` y hojas auxiliares de documentación (`diccionario`, `cobertura`, `validaciones`, `faltantes`, `correspondencia_ramas`), que **no** son bases alternativas. Primero se escribe un archivo temporal, se vuelve a leer y se compara con la base (prueba 12); solo si coincide se guarda con el nombre definitivo.

In [33]:
titulo_checkpoint("CHECKPOINT 9 · Exportación y verificación del Excel")
filas_cob = [dict(tipo="variable integrada", fuente=r.fuente, variable=r.variable, cobertura_geografica=r.cobertura_geografica,
                  periodo_en_fuente=r.rango_fuente_completo, periodo_en_base=r.rango_en_base, no_faltantes_en_base=r.no_faltantes_base,
                  nota="La fuente tiene meses fuera del panel; no se pierden en la fuente, solo no hay filas mes-rama para ellos."
                  if r.rango_fuente_completo != r.rango_en_base else "") for r in verif.itertuples()]
for v in conversiones.variable:
    d = diccionario.set_index("variable").loc[v]
    filas_cob.append(dict(tipo="variable derivada (D4)", fuente=d.fuente_id, variable=v, cobertura_geografica=d.cobertura_geografica,
                          periodo_en_fuente=None, periodo_en_base=rango_txt(base.mes[base[v].notna()]),
                          no_faltantes_en_base=int(base[v].notna().sum()), nota="Miles de pesos de julio de 2019"))
no_integradas = [
    ("MX_ARAN_01", "códigos no manufactureros (11xx, 21xx, 9xxx)", "EE.UU. <- México", rango_txt(census_no_manuf.mes), "Fuera del alcance: solo manufacturas."),
    ("MX_ARAN_02", "tasa efectiva (formato ancho)", "EE.UU. <- México", rango_txt(a2_largo.mes), "Duplicado de MX_ARAN_01; solo verificación."),
    ("MX_EXP_01", "hoja Autopartes", "México", "—", "Hoja de trabajo del usuario; origen no documentado."),
    ("INC_01", "TPU_QUARTERLY (TPUQ, TARIFFVOL)", "EE.UU.", f"{tpu_trim.trimestre.iloc[0]} a {tpu_trim.trimestre.iloc[-1]}", "Frecuencia trimestral."),
    ("INC_01", "TPU_DAILY", "EE.UU.", f"{tpu_diario.fecha.min():%Y-%m-%d} a {tpu_diario.fecha.max():%Y-%m-%d}", "Frecuencia diaria."),
    ("US_PROD_01", "series G.17 sin código NAICS de 3 o 4 dígitos idéntico", "EE.UU.", rango_txt(ip_largo.mes), "Requieren una regla de correspondencia."),
    ("INC_02", "WUI/WTUI/WPUI de otros países", "69 países", rango_txt(wui_largo.mes), "Solo se integran los países de la decisión D7."),
    ("PRECIOS_01", "INPP de sectores, total y servicios", "México", rango_txt(inpp_largo.mes), "Solo se usa el INPP de los 21 subsectores manufactureros."),
]
filas_cob += [dict(tipo="fuente no integrada", fuente=a, variable=b, cobertura_geografica=c_, periodo_en_fuente=d_, nota=e_) for a, b, c_, d_, e_ in no_integradas]
cobertura = pd.DataFrame(filas_cob)

na_l = base[["rama", "periodo", *vars_datos]].copy()
na_l[vars_datos] = na_l[vars_datos].isna()
na_l = na_l.melt(id_vars=["rama", "periodo"], var_name="variable", value_name="es_na")
fx_r = na_l.groupby(["variable", "rama"]).es_na.agg(observaciones="size", faltantes="sum").reset_index().query("faltantes > 0")
fx_m = na_l.groupby(["variable", "periodo"]).es_na.agg(observaciones="size", faltantes="sum").reset_index().query("faltantes > 0")
faltantes = pd.concat([
    faltantes_var.rename(columns={"n": "observaciones", "n_faltantes": "faltantes"}).assign(dimension="variable (total)"),
    fx_r.assign(dimension="variable x rama"), fx_m.assign(dimension="variable x mes")], ignore_index=True)
faltantes["pct_faltantes"] = (100 * faltantes.faltantes / faltantes.observaciones).round(2)
faltantes = faltantes[["dimension", "variable", "rama", "periodo", "observaciones", "faltantes", "pct_faltantes"]]

hoja_corr = (correspondencias.merge(verif_corr[["rama", "scian_titulo", "scian_trilateral", "subsector_trilateral", "naics2017_titulo", "naics2022_titulo",
                                                "rama_sin_cambios_2012_2022", "subsector_sin_cambios_2012_2022"]], on="rama", how="left")
             [["rama", "rama_nombre", "scian_titulo", "scian_trilateral", "naics2017_titulo", "naics2022_titulo", "rama_sin_cambios_2012_2022",
               "correspondencia_rama_verificada", "subsector", "subsector_trilateral", "subsector_sin_cambios_2012_2022",
               "correspondencia_subsector_verificada", "en_emim", "en_exportaciones", "en_census", "census_desc", "inpp_serie",
               "g17_serie_subsector", "g17_desc_subsector", "g17_serie_rama", "g17_desc_rama"]])
HOJAS = ["base_integrada", "diccionario", "cobertura", "validaciones", "faltantes", "correspondencia_ramas"]

def escribir_libro(ruta_salida):
    validaciones = pd.DataFrame(REG["checks"])
    hojas = dict(zip(HOJAS, [base, diccionario, cobertura, validaciones, faltantes, hoja_corr]))
    with pd.ExcelWriter(ruta_salida, engine="openpyxl", datetime_format="yyyy-mm", date_format="yyyy-mm") as w:
        for nombre, df in hojas.items():
            df.to_excel(w, sheet_name=nombre, index=False)
            ws = w.sheets[nombre]
            for celda in ws[1]:
                celda.font = openpyxl.styles.Font(bold=True)
                celda.fill = openpyxl.styles.PatternFill("solid", fgColor="DCE6F1")
            ws.freeze_panes = "D2" if nombre == "base_integrada" else "A2"
            ws.auto_filter.ref = ws.dimensions
            for j in range(1, df.shape[1] + 1):
                ws.column_dimensions[openpyxl.utils.get_column_letter(j)].width = 16 if nombre == "base_integrada" else 30
        ws = w.sheets["base_integrada"]
        for fila in ws.iter_rows(min_row=2, min_col=1, max_col=1):
            fila[0].number_format = "yyyy-mm"

DIR_PROCESADOS.mkdir(parents=True, exist_ok=True)
ruta_tmp = DIR_PROCESADOS / f".tmp_{datetime.now():%Y%m%d%H%M%S}.xlsx"
escribir_libro(ruta_tmp)
cols_texto = [c for c in base.columns if pd.api.types.is_string_dtype(base[c]) or base[c].dtype == object]
leida = pd.read_excel(ruta_tmp, sheet_name="base_integrada", dtype={c: str for c in cols_texto})
leida["mes"] = a_mes(leida.mes)
def comparar_col(v):
    a, b = base[v], leida.get(v)
    if b is None:
        return False
    if a.isna().all():
        return bool(b.isna().all())
    if pd.api.types.is_bool_dtype(a):
        return bool((a.astype(bool).to_numpy() == b.astype(bool).to_numpy()).all())
    return iguales(a, b)
cols_ok = {v: comparar_col(v) for v in base.columns}
registrar("CP9", "12. Excel", "La hoja base_integrada tiene las mismas filas y columnas que la base", leida.shape == base.shape and list(leida.columns) == list(base.columns),
          f"Base: {base.shape[0]:,} × {base.shape[1]}; Excel: {leida.shape[0]:,} × {leida.shape[1]}")
registrar("CP9", "12. Excel", "Todos los valores coinciden (números, fechas, texto y lógicos)", all(cols_ok.values()),
          "Columnas con diferencias: " + ", ".join(k for k, v in cols_ok.items() if not v))
ws_chk = openpyxl.load_workbook(ruta_tmp, read_only=True)["base_integrada"]
col_rama = list(base.columns).index("rama") + 1
tipos_rama = {fila[0].data_type for fila in ws_chk.iter_rows(min_row=2, min_col=col_rama, max_col=col_rama)}
registrar("CP9", "12. Excel", "Los códigos de rama se guardaron como celdas de texto en el Excel (Excel no los convierte en números)",
          tipos_rama == {"s"}, f"Tipos de celda en la columna rama: {sorted(tipos_rama)} (s = texto)")
registrar("CP9", "12. Excel", "El libro contiene exactamente las hojas previstas", pd.ExcelFile(ruta_tmp).sheet_names == HOJAS,
          ", ".join(pd.ExcelFile(ruta_tmp).sheet_names))
if not any(c["checkpoint"] == "CP9" and c["estado"] == "ERROR" for c in REG["checks"]):
    escribir_libro(RUTA_EXCEL_FINAL)   # se reescribe con la hoja 'validaciones' completa (incluye la prueba 12)
    ruta_tmp.unlink()
    final = pd.read_excel(RUTA_EXCEL_FINAL, sheet_name="base_integrada")
    exigir(final.shape == base.shape, "El archivo final no coincide con la base validada.")
    aviso(f"<b>Excel exportado:</b> <code>{RUTA_EXCEL_FINAL.relative_to(RAIZ)}</code> ({base.shape[0]:,} filas × {base.shape[1]} columnas en "
          f"<code>base_integrada</code>; {RUTA_EXCEL_FINAL.stat().st_size / 1024 ** 2:.1f} MB).", "ok")
else:
    ruta_tmp.unlink()
cerrar_checkpoint("CP9", "exportación")

**Resumen del CP9 (exportación):** 4 OK · 0 REVISAR · 0 ERROR

fuente,prueba,resultado,estado
12. Excel,La hoja base_integrada tiene las mismas filas y columnas que la base,"Base: 8,858 × 52; Excel: 8,858 × 52",OK
12. Excel,"Todos los valores coinciden (números, fechas, texto y lógicos)",Columnas con diferencias:,OK
12. Excel,Los códigos de rama se guardaron como celdas de texto en el Excel (Excel no los convierte en números),Tipos de celda en la columna rama: ['s'] (s = texto),OK
12. Excel,El libro contiene exactamente las hojas previstas,"base_integrada, diccionario, cobertura, validaciones, faltantes, correspondencia_ramas",OK


## Resumen general de validaciones

In [34]:
chk = pd.DataFrame(REG["checks"])
tabla(chk.groupby(["checkpoint", "estado"]).size().unstack(fill_value=0).reset_index(), "Comprobaciones por checkpoint y estado")
tabla(chk[chk.estado == "REVISAR"][["checkpoint", "fuente", "prueba", "resultado"]], "Todo lo que requiere revisión (estado REVISAR)", max_filas=300, ancho=400)
tabla(base.head(12)[["periodo", "rama", "rama_nombre", "emim_valor_produccion", "emim_valor_produccion_real", "expo_valor_real",
                     "eeuu_imp_mx_tasa_efectiva", "tpu", "wtui_mex", "epu_mex", "tc_fix", "vix_promedio_mensual"]],
      "Vista previa de la base integrada (primeras filas)")
display(Markdown(f"**Fin de la ejecución:** {datetime.now():%Y-%m-%d %H:%M:%S} · duración {(datetime.now() - INICIO).total_seconds():.0f} s  \n"
                 f"**Versiones:** Python {platform.python_version()}, pandas {pd.__version__}, numpy {np.__version__}, openpyxl {openpyxl.__version__}"))

**Comprobaciones por checkpoint y estado**

checkpoint,OK,REVISAR
CP1,58,0
CP2,3,2
CP3,90,14
CP3B,5,1
CP4,10,7
CP5,14,0
CP6,19,0
CP6B,21,0
CP7,2,0
CP8,23,1


**Todo lo que requiere revisión (estado REVISAR)**

checkpoint,fuente,prueba,resultado
CP2,MX_EXP_01,Hoja Autopartes,Serie derivada de origen no documentado (xparts no coincide con la rama 3363 de Datos; xparts_r_ae parece desestacionalizada). No se integra.
CP2,US_PROD_02,¿Existe una serie nacional total (industria o manufactura)?,No: solo series por industria. La producción de EE.UU. no puede entrar como variable nacional común.
CP3,MX_ARAN_01,Aranceles mayores que el valor gravable (compatible con aranceles específicos),"60 filas en los códigos 1119, 3112, 3119. No se modifican (decisión 7 de las notas)."
CP3,US_PROD_01,Series que terminan antes del último mes del archivo,"46 series terminan antes de 2026-08: N21113G, N21221, N221111, N221112, N221113, N221114T8, N22112C, N22112M, N22112R, N2212C, N2212M, N2212R, N2212T, N311511, N311512, N311513, N311514, N31152, N311611T3B, N311611T3P, N311611T3Z, N311615, N31212, N3211, N321991, N32211, N32212P, N32213, N32221, N32411A, N32411D, N32411G, N32411R, N32518C, N32731, N3311A2B, N3311A2C, N3311A2D, N3311A2E, N3311A2..."
CP3,PRECIOS_01,Base declarada vs. base efectiva,La unidad declarada (julio 2025=100) no coincide con los datos (100 en julio 2019).
CP3,INC_01,"TPU = TPU_SHARE × 10,000 (TPU_SHARE es una proporción, aunque el ReadMe diga 'percent share')",Confirma la unidad de TPU (100 = 1% de artículos).
CP3,INC_01,TPU_SHARE = TPU_RAW / N7,
CP3,INC_01,ReadMe desactualizado,El ReadMe dice 'monthly from 1985-2020'; los datos cubren 1960-01 a 2026-08.
CP3,INC_01,Versiones trimestral y diaria no integradas,Trimestral: 1960Q1 a 2026Q2 (TARIFFVOL hasta 2018Q4). Diaria: 2015-01-01 a 2026-09-21. Frecuencia distinta a la mensual; no se convierten.
CP3,INC_02,WPUI MEX: proporción de meses en cero,"63 de 224 meses valen 0 (28.1%). Son ceros de la fuente (conteos bajos de palabras), no faltantes; no se modifican."


**Vista previa de la base integrada (primeras filas)**

periodo,rama,rama_nombre,emim_valor_produccion,emim_valor_produccion_real,expo_valor_real,eeuu_imp_mx_tasa_efectiva,tpu,wtui_mex,epu_mex,tc_fix,vix_promedio_mensual
2018-01,3111,Elaboración de alimentos para animales,"10,821,469.00","11,710,063.69",NA,0,73.2147,0.394225,66.2948,18.9074,11.0624
2018-02,3111,Elaboración de alimentos para animales,"10,650,891.00","11,473,564.90",NA,0,57.0297,0.241736,54.9692,18.6449,22.4647
2018-03,3111,Elaboración de alimentos para animales,"10,922,444.00","11,707,660.81",NA,0,260.556,0.205402,45.4321,18.6308,19.0238
2018-04,3111,Elaboración de alimentos para animales,"10,749,906.00","11,435,013.76",NA,0,174.208,0.3314,56.6906,18.3872,18.2676
2018-05,3111,Elaboración de alimentos para animales,"11,553,945.00","12,202,049.59",NA,0,152.25,0.306373,84.1799,19.591,14.1245
2018-06,3111,Elaboración de alimentos para animales,"11,450,831.00","12,040,358.67",NA,0,228.623,0.307673,75.6292,20.3032,13.6781
2018-07,3111,Elaboración de alimentos para animales,"11,121,759.00","11,707,105.51",NA,0.000623057,246.382,0.657419,44.1576,19.0095,13.1476
2018-08,3111,Elaboración de alimentos para animales,"11,352,257.00","11,891,899.10",NA,0,225.829,0.319081,73.3644,18.8575,12.547
2018-09,3111,Elaboración de alimentos para animales,"10,367,020.00","10,807,412.46",NA,0,150.109,0.222556,48.7473,19.0154,12.9105
2018-10,3111,Elaboración de alimentos para animales,"10,764,523.00","11,227,457.96",NA,0,155.207,0.186312,86.6823,19.1859,19.3522


**Fin de la ejecución:** 2026-10-01 21:01:50 · duración 54 s  
**Versiones:** Python 3.11.15, pandas 3.0.6, numpy 2.4.6, openpyxl 3.1.5